#Uploading the dataset

In [3]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import re

# Upload the existing CADEC ZIP file
from google.colab import files

uploaded = files.upload()

# Get the uploaded file's name
ZIP_PATH = Path(next(iter(uploaded)))

print("Uploaded file:", ZIP_PATH.name)

Saving CADEC.v2.zip to CADEC.v2 (1).zip
Uploaded file: CADEC.v2 (1).zip


#Check the dataset

In [4]:
# Check that the uploaded file exists
if not ZIP_PATH.exists():
    raise FileNotFoundError(f"File not found: {ZIP_PATH}")

# Inspect the existing dataset without modifying it
with ZipFile(ZIP_PATH, "r") as dataset:
    files = dataset.namelist()

    print("Dataset accessed successfully!")
    print("Total entries:", len(files))
    print("\nExample text files:")

    text_files = [
        file for file in files
        if file.endswith(".txt")
    ]

    for file in text_files[:5]:
        print(file)

    print("\nTotal patient text files:", len(text_files))

Dataset accessed successfully!
Total entries: 5005

Example text files:
cadec/text/ARTHROTEC.1.txt
cadec/text/ARTHROTEC.10.txt
cadec/text/ARTHROTEC.100.txt
cadec/text/ARTHROTEC.101.txt
cadec/text/ARTHROTEC.102.txt

Total patient text files: 1250


In [5]:
with ZipFile(ZIP_PATH, "r") as dataset:

    all_files = dataset.namelist()

    folders = {
        "Patient text files": "cadec/text/",
        "Original annotations": "cadec/original/",
        "MedDRA mappings": "cadec/meddra/",
        "SNOMED CT mappings": "cadec/sct/"
    }

    for name, folder in folders.items():

        if folder.endswith("text/"):
            extension = ".txt"
        else:
            extension = ".ann"

        matching_files = [
            f for f in all_files
            if f.startswith(folder) and f.endswith(extension)
        ]

        print(f"{name}: {len(matching_files)}")

Patient text files: 1250
Original annotations: 1250
MedDRA mappings: 1250
SNOMED CT mappings: 1250


#Inspecting one example

In [6]:
# Choose one patient post to inspect
sample_id = "ARTHROTEC.8"

# Corresponding files in the existing dataset
sample_files = {
    "Patient's original text": f"cadec/text/{sample_id}.txt",
    "Original medical annotations": f"cadec/original/{sample_id}.ann",
    "MedDRA mappings": f"cadec/meddra/{sample_id}.ann",
    "SNOMED CT mappings": f"cadec/sct/{sample_id}.ann"
}

with ZipFile(ZIP_PATH, "r") as dataset:

    for title, file_path in sample_files.items():
        print("\n" + "=" * 60)
        print(title)
        print("=" * 60)

        content = dataset.read(file_path).decode(
            "utf-8", errors="replace"
        )

        print(content if content.strip() else "[No annotations]")


Patient's original text
No side effect to begin, but now starting to upset my stomach even while taking with food and plenty of water.
Didn't work for me.
Still can't sleep, can't work without pain.
Didn't see any positive change.


Original medical annotations
T1	ADR 33 61	starting to upset my stomach
T2	Symptom 169 173	pain
T3	Symptom 137 148	can't sleep


MedDRA mappings
TT1	10042076 33 61	starting to upset my stomach


SNOMED CT mappings
TT1	271681002 | Stomach ache | 33 61	starting to upset my stomach
TT3	193462001 | Insomnia | 137 148	can't sleep
TT2	22253000 | Pain | 169 173	pain



#Load the existing data into tables

In [7]:
# Step 4: Load CADEC data with robust annotation parsing

from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import re


def split_ann_line(line):
    """Read annotation rows with tabs or multiple-space separators."""
    raw = line.rstrip("\r\n")

    if "\t" in raw:
        parts = raw.split("\t", 2)

        if len(parts) == 3:
            return (
                parts[0].strip(),
                parts[1].strip(),
                parts[2].strip()
            )

        if len(parts) == 2:
            left, mention = parts
            fields = left.strip().split(maxsplit=1)

            if len(fields) == 2:
                return fields[0], fields[1].strip(), mention.strip()

    # Some annotation rows use multiple spaces instead of tabs
    parts = re.split(r"\s{2,}", raw.strip(), maxsplit=2)

    if len(parts) == 3:
        return tuple(part.strip() for part in parts)

    return None


def parse_offsets(offset_text):
    """Preserve continuous and discontinuous character spans."""
    offset_text = offset_text.strip()

    pattern = r"\d+\s+\d+(?:;\s*\d+\s+\d+)*"

    if not re.fullmatch(pattern, offset_text):
        return None

    spans = [
        (int(start), int(end))
        for start, end in re.findall(
            r"(\d+)\s+(\d+)", offset_text
        )
    ]

    # Sort segments by their position in the original text
    return sorted(spans, key=lambda span: span[0])


def parse_span_suffix(details):
    """Separate concept information from the trailing character offsets."""
    match = re.fullmatch(
        r"(.*?)\s+(\d+\s+\d+(?:;\s*\d+\s+\d+)*)",
        details.strip()
    )

    if not match:
        return None

    prefix = match.group(1).strip()
    spans = parse_offsets(match.group(2))

    if spans is None:
        return None

    return prefix, spans


def span_metadata(spans):
    """Create a consistent key for matching annotation mappings."""
    return {
        "spans": tuple(spans),
        "span_key": ";".join(
            f"{start}:{end}" for start, end in spans
        ),
        "start": spans[0][0],
        "end": spans[-1][1],
        "is_discontinuous": len(spans) > 1
    }


documents = []
original_annotations = []
meddra_mappings = []
snomed_mappings = []

with ZipFile(ZIP_PATH, "r") as dataset:

    all_files = set(dataset.namelist())

    text_files = sorted(
        f for f in all_files
        if f.startswith("cadec/text/")
        and f.endswith(".txt")
    )

    for text_path in text_files:

        doc_id = Path(text_path).stem

        # Load the original patient post
        text = dataset.read(text_path).decode(
            "utf-8", errors="replace"
        )

        documents.append({
            "doc_id": doc_id,
            "text": text
        })

        # Load each existing annotation file
        for category in ["original", "meddra", "sct"]:

            ann_path = f"cadec/{category}/{doc_id}.ann"

            if ann_path not in all_files:
                continue

            content = dataset.read(ann_path).decode(
                "utf-8", errors="replace"
            )

            for line in content.splitlines():

                if not line.strip() or line.lstrip().startswith("#"):
                    continue

                parsed = split_ann_line(line)

                if parsed is None:
                    continue

                annotation_id, details, mention = parsed

                # Original entity labels
                if category == "original":

                    match = re.fullmatch(
                        r"(\S+)\s+(\d+\s+\d+(?:;\s*\d+\s+\d+)*)",
                        details.strip()
                    )

                    if not match:
                        continue

                    entity_type, offset_text = match.groups()
                    spans = parse_offsets(offset_text)

                    if spans is None:
                        continue

                    record = {
                        "doc_id": doc_id,
                        "entity_id": annotation_id,
                        "entity_type": entity_type,
                        "mention": mention
                    }

                    record.update(span_metadata(spans))
                    original_annotations.append(record)

                # MedDRA mappings
                elif category == "meddra":

                    parsed_mapping = parse_span_suffix(details)

                    if parsed_mapping is None:
                        continue

                    concept_id, spans = parsed_mapping

                    record = {
                        "doc_id": doc_id,
                        "meddra_id": concept_id,
                        "mention": mention
                    }

                    record.update(span_metadata(spans))
                    meddra_mappings.append(record)

                # SNOMED CT mappings
                elif category == "sct":

                    parsed_mapping = parse_span_suffix(details)

                    if parsed_mapping is None:
                        continue

                    concept_info, spans = parsed_mapping

                    if "|" in concept_info:
                        concept_id, concept_name = (
                            concept_info.split("|", 1)
                        )

                        concept_id = concept_id.strip()
                        concept_name = (
                            concept_name.strip().strip("|").strip()
                            or None
                        )
                    else:
                        # Some entries have no standardized concept
                        concept_id = concept_info.strip()
                        concept_name = None

                    record = {
                        "doc_id": doc_id,
                        "snomed_id": concept_id,
                        "snomed_name": concept_name,
                        "mention": mention
                    }

                    record.update(span_metadata(spans))
                    snomed_mappings.append(record)


# Create structured tables
documents_df = pd.DataFrame(documents)
entities_df = pd.DataFrame(original_annotations)
meddra_df = pd.DataFrame(meddra_mappings)
snomed_df = pd.DataFrame(snomed_mappings)


# Link mappings to original annotations using document ID and spans
annotated_df = entities_df.merge(
    meddra_df[["doc_id", "span_key", "meddra_id"]],
    on=["doc_id", "span_key"],
    how="left",
    validate="one_to_one"
)

annotated_df = annotated_df.merge(
    snomed_df[["doc_id", "span_key", "snomed_id", "snomed_name"]],
    on=["doc_id", "span_key"],
    how="left",
    validate="one_to_one"
)

# Include the original patient text
annotated_df = annotated_df.merge(
    documents_df,
    on="doc_id",
    how="left",
    validate="many_to_one"
)


# Report the results
print("DATA LOADED AND MATCHED")
print("-" * 45)

print("Patient posts:", len(documents_df))
print("Original annotations:", len(entities_df))
print("MedDRA mapping entries:", len(meddra_df))
print("SNOMED CT mapping entries:", len(snomed_df))
print(
    "Discontinuous original annotations:",
    int(entities_df["is_discontinuous"].sum())
)

print("\nOriginal annotation label counts:")
print(entities_df["entity_type"].value_counts())

print("\nExample combined annotations:")
display(
    annotated_df[
        [
            "doc_id",
            "entity_type",
            "mention",
            "meddra_id",
            "snomed_id",
            "snomed_name"
        ]
    ].head(10)
)

DATA LOADED AND MATCHED
---------------------------------------------
Patient posts: 1250
Original annotations: 9111
MedDRA mapping entries: 6318
SNOMED CT mapping entries: 9111
Discontinuous original annotations: 1066

Original annotation label counts:
entity_type
ADR        6318
Drug       1800
Finding     435
Disease     283
Symptom     275
Name: count, dtype: int64

Example combined annotations:


,doc_id,entity_type,mention,meddra_id,snomed_id,snomed_name
0,ARTHROTEC.1,ADR,bit drowsy,10013649,271782001,Drowsy
1,ARTHROTEC.1,ADR,little blurred vision,10005886,246636008,Blurred vision - hazy
2,ARTHROTEC.1,Drug,Arthrotec,NaN,3384011000036100,Arthrotec
3,ARTHROTEC.1,Disease,arthritis,NaN,3723001,Arthritis
4,ARTHROTEC.1,Symptom,agony,NaN,102498003,Agony | or 76948002|Severe pain
5,ARTHROTEC.1,ADR,gastric problems,10056819,162076009,Excessive upper gastrointestinal gas
6,ARTHROTEC.1,Symptom,pains,NaN,22253000,Pain
7,ARTHROTEC.1,ADR,feel a bit weird,10025482,367391008,Malaise
8,ARTHROTEC.10,Symptom,hip pain,NaN,49218002,Hip pain
9,ARTHROTEC.10,Symptom,walk up & down steps sideways,NaN,CONCEPT_LESS,None


#Check that the annotations match the original text

In [8]:
# Step 5: Validate annotation spans against the original text

import re
import pandas as pd

# Make patient posts easy to look up
text_lookup = documents_df.set_index("doc_id")["text"].to_dict()


def normalize_for_comparison(text):
    """Ignore differences in capitalization and whitespace."""
    return re.sub(r"\s+", " ", str(text)).strip().casefold()


validation_results = []

for row in entities_df.itertuples(index=False):

    original_text = text_lookup[row.doc_id]

    # Retrieve each annotated segment from the original post
    segments = [
        original_text[start:end]
        for start, end in row.spans
    ]

    # Join segments for comparison, including discontinuous mentions
    reconstructed = " ".join(segments)

    matches = (
        normalize_for_comparison(reconstructed)
        == normalize_for_comparison(row.mention)
    )

    validation_results.append({
        "doc_id": row.doc_id,
        "entity_type": row.entity_type,
        "mention": row.mention,
        "reconstructed": reconstructed,
        "is_discontinuous": row.is_discontinuous,
        "matches": matches
    })


validation_df = pd.DataFrame(validation_results)

print("ANNOTATION VALIDATION")
print("-" * 45)

print("Total annotations checked:", len(validation_df))
print(
    "Exact text reconstructions:",
    int(validation_df["matches"].sum())
)
print(
    "Reconstruction match rate:",
    f"{validation_df['matches'].mean() * 100:.2f}%"
)

print("\nResults by annotation type:")
display(
    validation_df.groupby("is_discontinuous")["matches"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "Annotations",
        "sum": "Matches",
        "mean": "Match rate"
    })
)

# Inspect mismatches rather than silently ignoring them
mismatches = validation_df[~validation_df["matches"]]

print("\nNumber of mismatches:", len(mismatches))

if not mismatches.empty:
    display(mismatches.head(10))

ANNOTATION VALIDATION
---------------------------------------------
Total annotations checked: 9111
Exact text reconstructions: 9064
Reconstruction match rate: 99.48%

Results by annotation type:


,Annotations,Matches,Match rate
is_discontinuous,,,
False,8045,8044,0.999876
True,1066,1020,0.956848



Number of mismatches: 47


,doc_id,entity_type,mention,reconstructed,is_discontinuous,matches
525,ARTHROTEC.35,ADR,bleeding several times a month menstrual cycle...,menstrual cycle is completely out of whack ble...,True,False
932,ARTHROTEC.95,ADR,swelling feet,feet swelling,True,False
1083,DICLOFENAC-SODIUM.7,ADR,renal failure,rena failure,True,False
1084,DICLOFENAC-SODIUM.7,ADR,renal and respiratory failure,respiratory failure,False,False
1817,LIPITOR.18,ADR,Severe pain shoulder,Severe shoulder pain,True,False
1946,LIPITOR.189,ADR,feet numbing,numbing feet,True,False
2620,LIPITOR.264,ADR,in both legs pain,pain in both legs,True,False
2621,LIPITOR.264,ADR,in both legs weakness,weakness in both legs,True,False
2732,LIPITOR.273,ADR,pain muscle,muscle pain,True,False
2735,LIPITOR.273,ADR,legs tingling in,tingling in legs,True,False


#Inspecting the mismatches

In [9]:
# Step 6: Investigate annotation mismatches

# Inspect up to 15 mismatches
mismatch_indices = mismatches.index[:15]

with ZipFile(ZIP_PATH, "r") as dataset:

    for idx in mismatch_indices:

        # Retrieve the corresponding annotation
        row = entities_df.iloc[idx]
        validation_row = validation_df.loc[idx]

        print("\n" + "=" * 70)
        print(f"Document: {row.doc_id}")
        print(f"Entity ID: {row.entity_id}")
        print(f"Label: {row.entity_type}")
        print(f"Discontinuous: {row.is_discontinuous}")

        # Original annotation as recorded in CADEC
        ann_path = f"cadec/original/{row.doc_id}.ann"
        annotation_file = dataset.read(ann_path).decode(
            "utf-8", errors="replace"
        )

        for line in annotation_file.splitlines():
            if line.startswith(row.entity_id + "\t"):
                print("Original annotation line:", line)
                break

        print("Annotated phrase:", repr(row.mention))

        # Show what each character span retrieves from the post
        original_text = text_lookup[row.doc_id]

        print("Extracted text segments:")
        for start, end in row.spans:
            print(
                f"  Characters {start}:{end} -> "
                f"{repr(original_text[start:end])}"
            )

        print(
            "Reconstructed phrase:",
            repr(validation_row["reconstructed"])
        )


Document: ARTHROTEC.35
Entity ID: T8
Label: ADR
Discontinuous: True
Original annotation line: T8	ADR 135 165;87 129	bleeding several times a month menstrual cycle is completely out of whack
Annotated phrase: 'bleeding several times a month menstrual cycle is completely out of whack'
Extracted text segments:
  Characters 87:129 -> 'menstrual cycle is completely out of whack'
  Characters 135:165 -> 'bleeding several times a month'
Reconstructed phrase: 'menstrual cycle is completely out of whack bleeding several times a month'

Document: ARTHROTEC.95
Entity ID: T2
Label: ADR
Discontinuous: True
Original annotation line: T2	ADR 115 123;91 95	swelling feet
Annotated phrase: 'swelling feet'
Extracted text segments:
  Characters 91:95 -> 'feet'
  Characters 115:123 -> 'swelling'
Reconstructed phrase: 'feet swelling'

Document: DICLOFENAC-SODIUM.7
Entity ID: T7
Label: ADR
Discontinuous: True
Original annotation line: T7	ADR 411 415;432 440	renal failure
Annotated phrase: 'renal failure'
Ext

#Separate word-order differences from possible errors

In [10]:
from collections import Counter
import re

# Tokenize phrases while preserving contractions
def get_tokens(text):
    return re.findall(
        r"\b[\w]+(?:['’][\w]+)*\b",
        str(text).casefold()
    )


# Compare the words, ignoring their order
validation_df["same_words"] = validation_df.apply(
    lambda row: Counter(get_tokens(row["mention"]))
    == Counter(get_tokens(row["reconstructed"])),
    axis=1
)

print("WORD-LEVEL VALIDATION")
print("-" * 45)

print("Total annotations:", len(validation_df))
print(
    "Annotations containing the same words:",
    int(validation_df["same_words"].sum())
)
print(
    "Word-level agreement:",
    f"{validation_df['same_words'].mean() * 100:.2f}%"
)

print("\nResults by annotation type:")
display(
    validation_df.groupby("is_discontinuous")["same_words"]
    .agg(["count", "sum", "mean"])
)

# Inspect cases where even the words do not match
possible_errors = validation_df[~validation_df["same_words"]]

print(
    "\nAnnotations needing further inspection:",
    len(possible_errors)
)

display(
    possible_errors[
        [
            "doc_id",
            "entity_type",
            "mention",
            "reconstructed",
            "is_discontinuous"
        ]
    ].head(20)
)

WORD-LEVEL VALIDATION
---------------------------------------------
Total annotations: 9111
Annotations containing the same words: 9107
Word-level agreement: 99.96%

Results by annotation type:


,count,sum,mean
is_discontinuous,,,
False,8045,8044,0.999876
True,1066,1063,0.997186



Annotations needing further inspection: 4


,doc_id,entity_type,mention,reconstructed,is_discontinuous
1083,DICLOFENAC-SODIUM.7,ADR,renal failure,rena failure,True
1084,DICLOFENAC-SODIUM.7,ADR,renal and respiratory failure,respiratory failure,False
8285,LIPITOR.944,ADR,eyes microabrasions,eyes microabrasion,True
8968,VOLTAREN.34,ADR,pain in stomach,pain i stomach,True


#Inspect the four remaining cases

In [11]:
# Step 8: Inspect the four remaining annotation discrepancies

with ZipFile(ZIP_PATH, "r") as dataset:

    for idx in possible_errors.index:

        row = entities_df.iloc[idx]
        original_text = text_lookup[row.doc_id]

        start = min(s for s, e in row.spans)
        end = max(e for s, e in row.spans)

        print("\n" + "=" * 70)
        print("Document:", row.doc_id)
        print("Entity ID:", row.entity_id)
        print("Label:", row.entity_type)
        print("Annotated phrase:", repr(row.mention))

        print("\nPatient text around the annotation:")
        print(repr(original_text[max(0, start - 60):min(len(original_text), end + 60)]))

        print("\nIndividual character spans:")
        for s, e in row.spans:
            print(f"{s}:{e} -> {repr(original_text[s:e])}")

        print("\nOriginal annotation record:")
        ann_path = f"cadec/original/{row.doc_id}.ann"
        annotation_text = dataset.read(ann_path).decode(
            "utf-8", errors="replace"
        )

        for line in annotation_text.splitlines():
            if line.startswith(row.entity_id + "\t"):
                print(line)
                break


Document: DICLOFENAC-SODIUM.7
Entity ID: T7
Label: ADR
Annotated phrase: 'renal failure'

Patient text around the annotation:
't operatively I suffered bleeding, peritonitis, septicaemia,renal and respiratory failure.\nI was sent home wearing 2 stoma bags and 3 wound dressings'

Individual character spans:
411:415 -> 'rena'
432:440 -> ' failure'

Original annotation record:
T7	ADR 411 415;432 440	renal failure

Document: DICLOFENAC-SODIUM.7
Entity ID: T8
Label: ADR
Annotated phrase: 'renal and respiratory failure'

Patient text around the annotation:
'ely I suffered bleeding, peritonitis, septicaemia,renal and respiratory failure.\nI was sent home wearing 2 stoma bags and 3 wound dressings'

Individual character spans:
421:440 -> 'respiratory failure'

Original annotation record:
T8	ADR 421 440	renal and respiratory failure

Document: LIPITOR.944
Entity ID: T5
Label: ADR
Annotated phrase: 'eyes microabrasions'

Patient text around the annotation:
' and noticed extreme dehydration and 

#Correct the four confirmed offset issues

In [12]:
# Step 9: Correct confirmed character offsets in a working copy

working_entities_df = entities_df.copy(deep=True)

# Corrected spans, verified against the original patient text
corrections = {
    ("DICLOFENAC-SODIUM.7", "T7"): [
        (411, 416), (432, 440)
    ],
    ("DICLOFENAC-SODIUM.7", "T8"): [
        (411, 440)
    ],
    ("LIPITOR.944", "T5"): [
        (86, 90), (159, 173)
    ],
    ("VOLTAREN.34", "T5"): [
        (215, 222), (249, 256)
    ]
}

# Apply corrections only to the working copy
for (doc_id, entity_id), spans in corrections.items():

    mask = (
        (working_entities_df["doc_id"] == doc_id)
        & (working_entities_df["entity_id"] == entity_id)
    )

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one annotation for {doc_id}, {entity_id}"
        )

    row_index = working_entities_df.index[mask][0]

    # Update spans and their derived metadata
    for column, value in span_metadata(spans).items():
        working_entities_df.at[row_index, column] = value

print("Confirmed offset corrections applied:", len(corrections))
print("Original annotations preserved:", len(entities_df))
print("Working annotations:", len(working_entities_df))

Confirmed offset corrections applied: 4
Original annotations preserved: 9111
Working annotations: 9111


In [13]:
# Revalidate all corrected annotations

corrected_results = []

for row in working_entities_df.itertuples(index=False):

    text = text_lookup[row.doc_id]

    extracted_segments = [
        text[start:end]
        for start, end in row.spans
    ]

    reconstructed = " ".join(extracted_segments)

    exact_match = (
        normalize_for_comparison(reconstructed)
        == normalize_for_comparison(row.mention)
    )

    same_words = (
        Counter(get_tokens(reconstructed))
        == Counter(get_tokens(row.mention))
    )

    corrected_results.append({
        "doc_id": row.doc_id,
        "entity_id": row.entity_id,
        "mention": row.mention,
        "reconstructed": reconstructed,
        "exact_match": exact_match,
        "same_words": same_words
    })

corrected_validation_df = pd.DataFrame(corrected_results)

print("ANNOTATION VALIDATION AFTER CORRECTIONS")
print("-" * 45)
print("Total annotations:", len(corrected_validation_df))
print(
    "Exact reconstructions:",
    int(corrected_validation_df["exact_match"].sum())
)
print(
    "Exact reconstruction rate:",
    f"{corrected_validation_df['exact_match'].mean() * 100:.2f}%"
)
print(
    "Word-level matches:",
    int(corrected_validation_df["same_words"].sum())
)
print(
    "Word-level agreement:",
    f"{corrected_validation_df['same_words'].mean() * 100:.2f}%"
)

print("\nRemaining word-level mismatches:")
display(
    corrected_validation_df[
        ~corrected_validation_df["same_words"]
    ]
)

ANNOTATION VALIDATION AFTER CORRECTIONS
---------------------------------------------
Total annotations: 9111
Exact reconstructions: 9068
Exact reconstruction rate: 99.53%
Word-level matches: 9111
Word-level agreement: 100.00%

Remaining word-level mismatches:


,doc_id,entity_id,mention,reconstructed,exact_match,same_words


#Exploratory data analysis

In [14]:
# Step 10: Exploratory Data Analysis

# 1. Explore the length of patient posts
documents_df["word_count"] = documents_df["text"].apply(
    lambda text: len(text.split())
)

print("1. PATIENT POST STATISTICS")
print("-" * 45)
print("Total posts:", len(documents_df))
print("Average words per post:", round(documents_df["word_count"].mean(), 2))
print("Median words per post:", documents_df["word_count"].median())
print("Shortest post:", documents_df["word_count"].min(), "words")
print("Longest post:", documents_df["word_count"].max(), "words")


# 2. Explore the existing medical labels
print("\n2. MEDICAL ENTITY DISTRIBUTION")
print("-" * 45)

entity_counts = (
    working_entities_df["entity_type"]
    .value_counts()
    .rename_axis("Entity type")
    .reset_index(name="Count")
)

entity_counts["Percentage"] = (
    entity_counts["Count"]
    / len(working_entities_df)
    * 100
).round(2)

display(entity_counts)


# 3. Examine the existing SNOMED CT mappings
print("\n3. SNOMED CT MAPPING STATISTICS")
print("-" * 45)

snomed_valid = snomed_df.copy()

# Exclude missing concepts and unassigned concept labels
snomed_valid = snomed_valid[
    snomed_valid["snomed_id"].notna()
    & (snomed_valid["snomed_id"].astype(str).str.upper()
       != "CONCEPT_LESS")
    & snomed_valid["snomed_name"].notna()
].copy()

snomed_valid["normalized_mention"] = (
    snomed_valid["mention"]
    .map(normalize_for_comparison)
)

print("Total SNOMED mapping entries:", len(snomed_df))
print("Entries with assigned concept names:", len(snomed_valid))
print("Unique assigned concept IDs:", snomed_valid["snomed_id"].nunique())


# 4. Find concepts associated with different patient expressions
concept_diversity = (
    snomed_valid.groupby("snomed_id")
    .agg(
        concept_name=("snomed_name", "first"),
        mapped_mentions=("mention", "size"),
        distinct_patient_phrases=("normalized_mention", "nunique"),
        example_phrases=(
            "mention",
            lambda phrases: list(pd.unique(phrases))[:5]
        )
    )
    .reset_index()
    .sort_values(
        ["distinct_patient_phrases", "mapped_mentions"],
        ascending=False
    )
)

multiple_expression_concepts = concept_diversity[
    concept_diversity["distinct_patient_phrases"] > 1
]

print("\n4. PATIENT LANGUAGE VARIATION")
print("-" * 45)

print(
    "Concepts with multiple distinct patient expressions:",
    len(multiple_expression_concepts)
)

print("\nConcepts with the most distinct expressions:")
display(
    multiple_expression_concepts[
        [
            "snomed_id",
            "concept_name",
            "mapped_mentions",
            "distinct_patient_phrases",
            "example_phrases"
        ]
    ].head(15)
)

1. PATIENT POST STATISTICS
---------------------------------------------
Total posts: 1250
Average words per post: 83.12
Median words per post: 67.0
Shortest post: 0 words
Longest post: 694 words

2. MEDICAL ENTITY DISTRIBUTION
---------------------------------------------


,Entity type,Count,Percentage
0,ADR,6318,69.34
1,Drug,1800,19.76
2,Finding,435,4.77
3,Disease,283,3.11
4,Symptom,275,3.02



3. SNOMED CT MAPPING STATISTICS
---------------------------------------------
Total SNOMED mapping entries: 9111
Entries with assigned concept names: 8665
Unique assigned concept IDs: 1037

4. PATIENT LANGUAGE VARIATION
---------------------------------------------
Concepts with multiple distinct patient expressions: 523

Concepts with the most distinct expressions:


,snomed_id,concept_name,mapped_mentions,distinct_patient_phrases,example_phrases
945,76948002,Severe pain |+ 21522001 | Abdominal pain,195,141,"[horrible stomach pain, extreme pain, gut wren..."
906,68962001,Myalgia,265,78,"[muscular pains, muscle pain, Muscle pain, Mus..."
96,162076009,Excessive upper gastrointestinal gas,118,55,"[gastric problems, Alot of gas, gassy, gas, ga..."
882,62507009,Pins and needles,73,44,"[finger tingling, tingling, Pins and needles, ..."
226,228158008,Walking disability,64,40,"[the heel I couldn't walk on very well, could ..."
839,55300003,Muscle cramp,139,39,"[cramping, mild cramps, cramps, major cramps, ..."
810,49218002,Hip pain,70,38,"[hip pain, Spine & hip pain, pain in my hips, ..."
552,309774006,Weakness of limb,53,37,"[weakness in my legs, weaknesses in my legs, w..."
855,57676002,Arthralgia,136,36,"[arthritis pain, arthritic pain, athritic pain..."
23,10601006,Pain in lower limb,84,36,"[Developed sore legs, leg soreness, LEG PAIN, ..."


# Inspect empty posts and compound SNOMED mappings


In [15]:
# Step 11: Inspect empty posts and compound SNOMED mappings

# 1. Find empty patient posts
empty_posts = documents_df[
    documents_df["text"].fillna("").str.strip().eq("")
]

print("1. EMPTY PATIENT POSTS")
print("-" * 45)
print("Number of empty posts:", len(empty_posts))

if not empty_posts.empty:
    display(empty_posts[["doc_id", "text"]])


# 2. Inspect SNOMED entries containing multiple concepts
compound_rows = []

with ZipFile(ZIP_PATH, "r") as dataset:
    for file_path in dataset.namelist():

        if not (
            file_path.startswith("cadec/sct/")
            and file_path.endswith(".ann")
        ):
            continue

        content = dataset.read(file_path).decode(
            "utf-8", errors="replace"
        )

        for line in content.splitlines():
            if "|+" in line:
                compound_rows.append({
                    "file": file_path,
                    "raw_annotation": line
                })

compound_df = pd.DataFrame(compound_rows)

print("\n2. POSSIBLE COMPOUND SNOMED MAPPINGS")
print("-" * 45)
print("Rows containing '|+':", len(compound_df))

if not compound_df.empty:
    display(compound_df.head(10))

1. EMPTY PATIENT POSTS
---------------------------------------------
Number of empty posts: 2


,doc_id,text
505,LIPITOR.40,
1198,VOLTAREN-XR.9,



2. POSSIBLE COMPOUND SNOMED MAPPINGS
---------------------------------------------
Rows containing '|+': 78


,file,raw_annotation
0,cadec/sct/ARTHROTEC.112.ann,TT1\t76948002 | Severe pain |+ 21522001 | Abdo...
1,cadec/sct/ARTHROTEC.137.ann,TT6\t225014007 | Feeling empty |+ 271681002 | ...
2,cadec/sct/ARTHROTEC.137.ann,TT11\t162059005 | Upset stomach |+ 271681002 |...
3,cadec/sct/ARTHROTEC.65.ann,TT1\t76948002 | Severe pain |+ 21522001 | Abdo...
4,cadec/sct/ARTHROTEC.68.ann,TT2\t8708008 | Sharp pain |+ 271681002|Stomach...
5,cadec/sct/ARTHROTEC.76.ann,TT4\t16386004 | Dry skin |+ 418363000 | Itchin...
6,cadec/sct/ARTHROTEC.94.ann,TT3\t67849003 | Excruciating pain |+ 21522001 ...
7,cadec/sct/LIPITOR.114.ann,TT9\t271870002 | Low blood pressure reading |+...
8,cadec/sct/LIPITOR.114.ann,TT18\t271870002 | Low blood pressure reading |...
9,cadec/sct/LIPITOR.117.ann,TT7\t47933007 | Foot pain |+247373008 | Ankle ...


#Inspect the original mapping formats

In [16]:
# Step 12: Inspect compound SNOMED mappings and empty documents

from collections import Counter
import re

empty_ids = set(empty_posts["doc_id"])

with ZipFile(ZIP_PATH, "r") as dataset:
    files = dataset.namelist()

    # Part 1: Check whether the empty posts have annotations
    print("1. EMPTY DOCUMENT AUDIT")
    print("=" * 60)

    for doc_id in sorted(empty_ids):
        print(f"\nDocument: {doc_id}")

        for category in ["original", "meddra", "sct"]:
            path = f"cadec/{category}/{doc_id}.ann"

            if path in files:
                content = dataset.read(path).decode(
                    "utf-8", errors="replace"
                ).strip()

                print(f"{category}:")
                print(content if content else "[Empty annotation file]")
            else:
                print(f"{category}: [File not found]")

    # Part 2: Inspect the different compound-mapping formats
    print("\n\n2. SNOMED MAPPING FORMATS")
    print("=" * 60)

    counts = Counter()
    examples = {
        "plus": [],
        "or": [],
        "other": []
    }

    for path in files:
        if not (
            path.startswith("cadec/sct/")
            and path.endswith(".ann")
        ):
            continue

        content = dataset.read(path).decode(
            "utf-8", errors="replace"
        )

        for line in content.splitlines():
            has_plus = "|+" in line
            has_or = bool(
                re.search(r"\|\s*or\s+\d+\s*\|", line, re.I)
            )

            if has_plus:
                counts["Contains |+"] += 1
                if len(examples["plus"]) < 8:
                    examples["plus"].append((path, line))

            if has_or:
                counts["Contains | or concept"] += 1
                if len(examples["or"]) < 8:
                    examples["or"].append((path, line))

            if has_plus and has_or:
                counts["Contains both formats"] += 1

    print("Mapping-format counts:")
    for label, count in counts.items():
        print(f"{label}: {count}")

    for label, rows in examples.items():
        if rows:
            print(f"\nExamples: {label.upper()}")

            for path, line in rows:
                print(f"\nFile: {path}")
                print(line)

1. EMPTY DOCUMENT AUDIT

Document: LIPITOR.40
original:
[Empty annotation file]
meddra:
[Empty annotation file]
sct:
[Empty annotation file]

Document: VOLTAREN-XR.9
original:
[Empty annotation file]
meddra:
[Empty annotation file]
sct:
[Empty annotation file]


2. SNOMED MAPPING FORMATS
Mapping-format counts:
Contains | or concept: 3
Contains |+: 78

Examples: PLUS

File: cadec/sct/ARTHROTEC.112.ann
TT1	76948002 | Severe pain |+ 21522001 | Abdominal pain | 21 42	horrible stomach pain

File: cadec/sct/ARTHROTEC.137.ann
TT6	225014007 | Feeling empty |+ 271681002 | Stomch ache |  249 275	stomach emptiness and pain

File: cadec/sct/ARTHROTEC.137.ann
TT11	162059005 | Upset stomach |+ 271681002 | Stomach ache | 602 634	stomach stayed upset and painful

File: cadec/sct/ARTHROTEC.65.ann
TT1	76948002 | Severe pain |+ 21522001 | Abdominal pain | 8 35	gut-renching abdominal pain

File: cadec/sct/ARTHROTEC.68.ann
TT2	8708008 | Sharp pain |+ 271681002|Stomach ache| 229 253	sharp pain in my stomach

#Fix the SNOMED CT parser

In [17]:
# Step 13: Parse all SNOMED CT concepts, including alternatives

snomed_concept_records = []
unparsed_snomed_lines = []
mapping_line_count = 0

with ZipFile(ZIP_PATH, "r") as dataset:

    for path in dataset.namelist():

        if not (
            path.startswith("cadec/sct/")
            and path.endswith(".ann")
        ):
            continue

        doc_id = Path(path).stem

        content = dataset.read(path).decode(
            "utf-8", errors="replace"
        )

        for line in content.splitlines():

            if not line.strip():
                continue

            parts = split_ann_line(line)

            if parts is None:
                unparsed_snomed_lines.append((path, line))
                continue

            annotation_id, details, mention = parts

            parsed = parse_span_suffix(details)

            if parsed is None:
                unparsed_snomed_lines.append((path, line))
                continue

            concept_block, spans = parsed
            mapping_line_count += 1

            # Identify whether concepts are joined by "+" or "or"
            if re.search(r"\|\s*\+", concept_block):
                relation = "plus"
            elif re.search(r"\|\s*or\s+", concept_block, re.I):
                relation = "or"
            else:
                relation = "single"

            # Split the line into its individual concept expressions
            concept_parts = re.split(
                r"\s*\|\s*\+\s*|\s*\|\s*or\s+",
                concept_block,
                flags=re.IGNORECASE
            )

            extracted = []

            for concept_part in concept_parts:

                match = re.match(
                    r"^\s*(\d+)\s*\|\s*(.*?)\s*$",
                    concept_part
                )

                if match:
                    concept_id = match.group(1).strip()
                    concept_name = (
                        match.group(2).strip().strip("|").strip()
                        or None
                    )

                    extracted.append(
                        (concept_id, concept_name)
                    )

            # Preserve annotations that explicitly have no concept
            if not extracted and "CONCEPT_LESS" in concept_block.upper():
                extracted.append(("CONCEPT_LESS", None))

            # Record any format we have not handled
            if not extracted:
                unparsed_snomed_lines.append((path, line))
                continue

            span_info = span_metadata(spans)

            for concept_index, (concept_id, concept_name) in enumerate(
                extracted, start=1
            ):
                record = {
                    "doc_id": doc_id,
                    "annotation_id": annotation_id,
                    "mention": mention,
                    "snomed_id": concept_id,
                    "snomed_name": concept_name,
                    "mapping_relation": relation,
                    "concept_index": concept_index,
                    "raw_line": line
                }

                record.update(span_info)
                snomed_concept_records.append(record)


# One row per individual SNOMED concept assignment
snomed_concepts_df = pd.DataFrame(snomed_concept_records)

print("SNOMED PARSING RESULTS")
print("-" * 50)

print("Original mapping lines parsed:", mapping_line_count)
print("Individual concept assignments:", len(snomed_concepts_df))
print(
    "Unparsed lines requiring inspection:",
    len(unparsed_snomed_lines)
)

print("\nMapping relationships:")
print(snomed_concepts_df["mapping_relation"].value_counts())

print("\nExamples with multiple concepts:")
display(
    snomed_concepts_df[
        snomed_concepts_df["mapping_relation"].isin(["plus", "or"])
    ][
        [
            "doc_id",
            "mention",
            "snomed_id",
            "snomed_name",
            "mapping_relation"
        ]
    ].head(15)
)

if unparsed_snomed_lines:
    print("\nFirst 10 unparsed lines:")
    for path, line in unparsed_snomed_lines[:10]:
        print(path, ":", line)

SNOMED PARSING RESULTS
--------------------------------------------------
Original mapping lines parsed: 9111
Individual concept assignments: 9372
Unparsed lines requiring inspection: 1

Mapping relationships:
mapping_relation
single    8855
plus       511
or           6
Name: count, dtype: int64

Examples with multiple concepts:


,doc_id,mention,snomed_id,snomed_name,mapping_relation
5,ARTHROTEC.1,agony,102498003,Agony,or
6,ARTHROTEC.1,agony,76948002,Severe pain,or
27,ARTHROTEC.103,feel weak and almost fainted,271795006,Malaise and fatigue,plus
28,ARTHROTEC.103,feel weak and almost fainted,248223005,Feeling faint,plus
42,ARTHROTEC.105,forgetfulnes and confussion,55533009,Forgetful,plus
43,ARTHROTEC.105,forgetfulnes and confussion,40917007,Confusion,plus
79,ARTHROTEC.112,horrible stomach pain,76948002,Severe pain,plus
80,ARTHROTEC.112,horrible stomach pain,21522001,Abdominal pain,plus
98,ARTHROTEC.113,uterus pain and cramping,289669005,Pain of uterus,plus
99,ARTHROTEC.113,uterus pain and cramping,29542008,Uterine spasm,plus



First 10 unparsed lines:
cadec/sct/ARTHROTEC.91.ann : TT2	31460011000036109  (substance) 34 46	acidophilous


#Handle the remaining SNOMED format and audit the mappings

In [18]:
# Step 15: Recover the remaining SNOMED record and audit mappings

recovered_records = []
still_unparsed = []

for path, line in unparsed_snomed_lines:

    parts = split_ann_line(line)

    if parts is None:
        still_unparsed.append((path, line))
        continue

    annotation_id, details, mention = parts

    # Handle: concept_id (semantic_tag) start end
    match = re.fullmatch(
        r"\s*(\d+)\s+\(([^)]+)\)\s+"
        r"(\d+\s+\d+(?:;\s*\d+\s+\d+)*)\s*",
        details.strip()
    )

    if not match:
        still_unparsed.append((path, line))
        continue

    concept_id, semantic_tag, offset_text = match.groups()
    spans = parse_offsets(offset_text)

    if not spans:
        still_unparsed.append((path, line))
        continue

    record = {
        "doc_id": Path(path).stem,
        "annotation_id": annotation_id,
        "mention": mention,
        "snomed_id": concept_id,
        "snomed_name": None,
        "semantic_tag": semantic_tag,
        "mapping_relation": "single",
        "concept_index": 1,
        "raw_line": line
    }

    record.update(span_metadata(spans))
    recovered_records.append(record)


# Add the recovered records without changing the original ZIP
if recovered_records:
    snomed_concepts_df = pd.concat(
        [snomed_concepts_df, pd.DataFrame(recovered_records)],
        ignore_index=True
    )

print("RECOVERY RESULTS")
print("-" * 45)
print("Records recovered:", len(recovered_records))
print("Records still unparsed:", len(still_unparsed))

if recovered_records:
    display(pd.DataFrame(recovered_records))


# Audit one source annotation at a time
source_audit = (
    snomed_concepts_df
    .groupby(["doc_id", "annotation_id"], as_index=False)
    .agg(
        mention=("mention", "first"),
        mapping_relation=("mapping_relation", "first"),
        concept_count=("snomed_id", "nunique")
    )
)

print("\nMAPPING AUDIT")
print("-" * 45)
print("Individual concept assignments:", len(snomed_concepts_df))
print("Unique source annotations:", len(source_audit))

print("\nSource annotations by relationship:")
display(
    source_audit["mapping_relation"]
    .value_counts()
    .rename_axis("Relationship")
    .reset_index(name="Source annotations")
)

print("\nNumber of concepts per source annotation:")
display(
    source_audit.groupby("mapping_relation")["concept_count"]
    .describe()
    .round(2)
)

print("\nExamples with multiple concepts:")
display(
    source_audit[source_audit["concept_count"] > 1].head(10)
)

RECOVERY RESULTS
---------------------------------------------
Records recovered: 1
Records still unparsed: 0


,doc_id,annotation_id,mention,snomed_id,snomed_name,semantic_tag,mapping_relation,concept_index,raw_line,spans,span_key,start,end,is_discontinuous
0,ARTHROTEC.91,TT2,acidophilous,31460011000036109,None,substance,single,1,TT2\t31460011000036109 (substance) 34 46\taci...,"((34, 46),)",34:46,34,46,False



MAPPING AUDIT
---------------------------------------------
Individual concept assignments: 9373
Unique source annotations: 9111

Source annotations by relationship:


,Relationship,Source annotations
0,single,8856
1,plus,252
2,or,3



Number of concepts per source annotation:


,count,mean,std,min,25%,50%,75%,max
mapping_relation,,,,,,,,
or,3.0,2.00,0.00,2.0,2.0,2.0,2.0,2.0
plus,252.0,2.02,0.23,1.0,2.0,2.0,2.0,4.0
single,8856.0,1.00,0.00,1.0,1.0,1.0,1.0,1.0



Examples with multiple concepts:


,doc_id,annotation_id,mention,mapping_relation,concept_count
5,ARTHROTEC.1,TT6,agony,or,2
26,ARTHROTEC.103,TT5,feel weak and almost fainted,plus,2
37,ARTHROTEC.105,TT2,forgetfulnes and confussion,plus,2
76,ARTHROTEC.112,TT1,horrible stomach pain,plus,2
93,ARTHROTEC.113,TT7,uterus pain and cramping,plus,2
129,ARTHROTEC.120,TT5,"cold sweat, shaking uncontrollably",plus,2
243,ARTHROTEC.133,TT9,Spine & hip pain,plus,2
264,ARTHROTEC.137,TT11,stomach stayed upset and painful,plus,2
295,ARTHROTEC.137,TT6,stomach emptiness and pain,plus,2
307,ARTHROTEC.140,TT3,agony,or,2


#Finalize preliminary concept-diversity statistics

In [19]:
# Step 16: Finalize preliminary concept-diversity statistics

# Keep entries with an assigned concept ID and concept name
valid_concepts = snomed_concepts_df[
    snomed_concepts_df["snomed_id"].notna()
    & (snomed_concepts_df["snomed_id"] != "CONCEPT_LESS")
    & snomed_concepts_df["snomed_name"].notna()
].copy()

# Normalize only for comparing surface-form variations.
# Preserve the original phrases in the dataset.
valid_concepts["normalized_mention"] = (
    valid_concepts["mention"].map(normalize_for_comparison)
)

# Summarize the expressions associated with each concept
concept_diversity = (
    valid_concepts.groupby("snomed_id")
    .agg(
        concept_name=("snomed_name", "first"),
        mapped_mentions=("mention", "size"),
        distinct_patient_phrases=("normalized_mention", "nunique"),
        example_phrases=(
            "mention",
            lambda phrases: list(pd.unique(phrases))[:5]
        )
    )
    .reset_index()
    .sort_values(
        ["distinct_patient_phrases", "mapped_mentions"],
        ascending=False
    )
)

# Concepts associated with more than one distinct expression
multiple_expression_concepts = concept_diversity[
    concept_diversity["distinct_patient_phrases"] > 1
]

print("UPDATED CONCEPT-DIVERSITY ANALYSIS")
print("=" * 50)

print("Individual concept assignments:", len(snomed_concepts_df))
print("Assignments with named concepts:", len(valid_concepts))
print(
    "Unique concept IDs:",
    valid_concepts["snomed_id"].nunique()
)
print(
    "Concepts with multiple distinct patient expressions:",
    len(multiple_expression_concepts)
)

print("\nTop 15 concepts by expression diversity:")
display(
    multiple_expression_concepts[
        [
            "snomed_id",
            "concept_name",
            "mapped_mentions",
            "distinct_patient_phrases",
            "example_phrases"
        ]
    ].head(15)
)

UPDATED CONCEPT-DIVERSITY ANALYSIS
Individual concept assignments: 9373
Assignments with named concepts: 8927
Unique concept IDs: 1046
Concepts with multiple distinct patient expressions: 531

Top 15 concepts by expression diversity:


,snomed_id,concept_name,mapped_mentions,distinct_patient_phrases,example_phrases
954,76948002,Severe pain,199,143,"[agony, horrible stomach pain, extreme pain, g..."
915,68962001,Myalgia,288,88,"[muscular pains, muscle pain, Muscle pain, Mus..."
96,162076009,Excessive upper gastrointestinal gas,118,55,"[gastric problems, Alot of gas, gassy, gas, ga..."
23,10601006,Pain in lower limb,104,52,"[pain in feet and legs, Developed sore legs, l..."
796,45326000,Shoulder pain,82,50,"[shoulder pain, Shoulder pain, pain in shoulde..."
819,49218002,Hip pain,84,47,"[hip pain, Spine & hip pain, severe hip pain, ..."
891,62507009,Pins and needles,73,44,"[finger tingling, tingling, Pins and needles, ..."
810,47933007,Foot pain,67,44,"[pain in foot, foot pain, foot was hurting, pa..."
560,30989003,Knee pain,76,42,"[knee pain, sore knee, chronic knee pain, KNEE..."
227,228158008,Walking disability,66,42,"[the heel I couldn't walk on very well, could ..."


#Check overlapping annotations before training

In [20]:
# Step 17: Audit overlapping entity spans

overlap_pairs = {}

for doc_id, group in working_entities_df.groupby("doc_id"):

    segments = []

    for row in group.itertuples(index=False):
        for start, end in row.spans:
            segments.append({
                "start": start,
                "end": end,
                "entity_id": row.entity_id,
                "entity_type": row.entity_type,
                "mention": row.mention
            })

    segments.sort(key=lambda x: (x["start"], x["end"]))

    active = []

    for current in segments:
        start = current["start"]
        end = current["end"]

        # Keep only segments that may still overlap
        active = [
            previous for previous in active
            if previous["end"] > start
        ]

        for previous in active:

            # Ignore segments belonging to the same annotation
            if previous["entity_id"] == current["entity_id"]:
                continue

            # Positive-length character overlap
            if max(previous["start"], start) < min(
                previous["end"], end
            ):

                key = (
                    doc_id,
                    *sorted([
                        previous["entity_id"],
                        current["entity_id"]
                    ])
                )

                overlap_pairs.setdefault(key, {
                    "doc_id": doc_id,
                    "entity_1": previous["entity_id"],
                    "label_1": previous["entity_type"],
                    "mention_1": previous["mention"],
                    "entity_2": current["entity_id"],
                    "label_2": current["entity_type"],
                    "mention_2": current["mention"]
                })

        active.append(current)


overlaps_df = pd.DataFrame(list(overlap_pairs.values()))

print("ANNOTATION STRUCTURE AUDIT")
print("-" * 50)

print("Total annotations:", len(working_entities_df))
print(
    "Discontinuous annotations:",
    int(working_entities_df["is_discontinuous"].sum())
)
print("Documents:", len(documents_df))
print("Overlapping annotation pairs:", len(overlaps_df))

if not overlaps_df.empty:
    print("\nExamples of overlapping annotations:")
    display(overlaps_df.head(15))

ANNOTATION STRUCTURE AUDIT
--------------------------------------------------
Total annotations: 9111
Discontinuous annotations: 1066
Documents: 1250
Overlapping annotation pairs: 851

Examples of overlapping annotations:


,doc_id,entity_1,label_1,mention_1,entity_2,label_2,mention_2
0,ARTHROTEC.104,T7,Disease,severe osteoarthritis in the hands,T5,Disease,severe osteoarthritis in the knees
1,ARTHROTEC.105,T1,ADR,body swelling,T8,ADR,swelling face
2,ARTHROTEC.105,T1,ADR,body swelling,T9,ADR,swelling wrists
3,ARTHROTEC.105,T8,ADR,swelling face,T9,ADR,swelling wrists
4,ARTHROTEC.105,T1,ADR,body swelling,T10,ADR,swelling abdomen
5,ARTHROTEC.105,T8,ADR,swelling face,T10,ADR,swelling abdomen
6,ARTHROTEC.105,T9,ADR,swelling wrists,T10,ADR,swelling abdomen
7,ARTHROTEC.105,T1,ADR,body swelling,T11,ADR,swelling thighs
8,ARTHROTEC.105,T8,ADR,swelling face,T11,ADR,swelling thighs
9,ARTHROTEC.105,T9,ADR,swelling wrists,T11,ADR,swelling thighs


#Measure how much overlap affects our ADR task

In [21]:
# Step 18: Analyze overlapping annotations by entity type

# Categorize each overlapping pair regardless of label order
overlaps_df["label_pair"] = overlaps_df.apply(
    lambda row: " + ".join(
        sorted([row["label_1"], row["label_2"]])
    ),
    axis=1
)

# Overall overlap statistics
print("OVERLAP ANALYSIS")
print("=" * 50)

print("Overlapping annotation pairs:", len(overlaps_df))
print("Documents affected:", overlaps_df["doc_id"].nunique())

print("\nOverlapping pairs by entity-label combination:")
display(
    overlaps_df["label_pair"]
    .value_counts()
    .rename_axis("Entity-label combination")
    .reset_index(name="Overlapping pairs")
)

# Pairs where both annotations are ADRs
adr_adr_overlaps = overlaps_df[
    (overlaps_df["label_1"] == "ADR")
    & (overlaps_df["label_2"] == "ADR")
]

# Pairs where at least one annotation is an ADR
adr_involved_overlaps = overlaps_df[
    (overlaps_df["label_1"] == "ADR")
    | (overlaps_df["label_2"] == "ADR")
]

print("\nADR-SPECIFIC STATISTICS")
print("-" * 50)

print("ADR–ADR overlapping pairs:", len(adr_adr_overlaps))
print(
    "Documents with ADR–ADR overlaps:",
    adr_adr_overlaps["doc_id"].nunique()
)
print(
    "Pairs involving at least one ADR:",
    len(adr_involved_overlaps)
)
print(
    "Documents with at least one ADR-related overlap:",
    adr_involved_overlaps["doc_id"].nunique()
)

print("\nExamples of ADR-related overlaps:")
display(
    adr_involved_overlaps[
        [
            "doc_id",
            "entity_1",
            "label_1",
            "mention_1",
            "entity_2",
            "label_2",
            "mention_2"
        ]
    ].head(15)
)

OVERLAP ANALYSIS
Overlapping annotation pairs: 851
Documents affected: 312

Overlapping pairs by entity-label combination:


,Entity-label combination,Overlapping pairs
0,ADR + ADR,809
1,Finding + Finding,32
2,Symptom + Symptom,8
3,Disease + Disease,1
4,Drug + Drug,1



ADR-SPECIFIC STATISTICS
--------------------------------------------------
ADR–ADR overlapping pairs: 809
Documents with ADR–ADR overlaps: 299
Pairs involving at least one ADR: 809
Documents with at least one ADR-related overlap: 299

Examples of ADR-related overlaps:


,doc_id,entity_1,label_1,mention_1,entity_2,label_2,mention_2
1,ARTHROTEC.105,T1,ADR,body swelling,T8,ADR,swelling face
2,ARTHROTEC.105,T1,ADR,body swelling,T9,ADR,swelling wrists
3,ARTHROTEC.105,T8,ADR,swelling face,T9,ADR,swelling wrists
4,ARTHROTEC.105,T1,ADR,body swelling,T10,ADR,swelling abdomen
5,ARTHROTEC.105,T8,ADR,swelling face,T10,ADR,swelling abdomen
6,ARTHROTEC.105,T9,ADR,swelling wrists,T10,ADR,swelling abdomen
7,ARTHROTEC.105,T1,ADR,body swelling,T11,ADR,swelling thighs
8,ARTHROTEC.105,T8,ADR,swelling face,T11,ADR,swelling thighs
9,ARTHROTEC.105,T9,ADR,swelling wrists,T11,ADR,swelling thighs
10,ARTHROTEC.105,T10,ADR,swelling abdomen,T11,ADR,swelling thighs


#Create our train, validation and test splits

In [22]:
# Step 19: Create reproducible document-level data splits

import numpy as np
import pandas as pd

# Exclude empty posts from model experiments
model_documents = documents_df[
    documents_df["text"].fillna("").str.strip().ne("")
].copy()

# Use a fixed seed so results can be reproduced
doc_ids = np.array(
    sorted(model_documents["doc_id"].unique())
)

rng = np.random.default_rng(42)
rng.shuffle(doc_ids)

# Create 70% / 15% / 15% splits
n_total = len(doc_ids)
n_train = round(0.70 * n_total)
n_val = round(0.15 * n_total)

train_doc_ids = set(doc_ids[:n_train])
val_doc_ids = set(doc_ids[n_train:n_train + n_val])
test_doc_ids = set(doc_ids[n_train + n_val:])

# Store the document split assignments
document_splits_df = pd.DataFrame({
    "doc_id": (
        list(train_doc_ids)
        + list(val_doc_ids)
        + list(test_doc_ids)
    ),
    "split": (
        ["train"] * len(train_doc_ids)
        + ["validation"] * len(val_doc_ids)
        + ["test"] * len(test_doc_ids)
    )
})

# Create document tables for each split
train_documents_df = documents_df[
    documents_df["doc_id"].isin(train_doc_ids)
].copy()

val_documents_df = documents_df[
    documents_df["doc_id"].isin(val_doc_ids)
].copy()

test_documents_df = documents_df[
    documents_df["doc_id"].isin(test_doc_ids)
].copy()

# Split the original entity annotations
train_entities_df = working_entities_df[
    working_entities_df["doc_id"].isin(train_doc_ids)
].copy()

val_entities_df = working_entities_df[
    working_entities_df["doc_id"].isin(val_doc_ids)
].copy()

test_entities_df = working_entities_df[
    working_entities_df["doc_id"].isin(test_doc_ids)
].copy()

# Split the SNOMED concept assignments
train_snomed_df = snomed_concepts_df[
    snomed_concepts_df["doc_id"].isin(train_doc_ids)
].copy()

val_snomed_df = snomed_concepts_df[
    snomed_concepts_df["doc_id"].isin(val_doc_ids)
].copy()

test_snomed_df = snomed_concepts_df[
    snomed_concepts_df["doc_id"].isin(test_doc_ids)
].copy()


# Report split sizes
print("DOCUMENT-LEVEL SPLITS")
print("=" * 50)

split_summary = pd.DataFrame([
    {
        "Split": "Train",
        "Documents": len(train_documents_df),
        "Entity annotations": len(train_entities_df),
        "SNOMED assignments": len(train_snomed_df)
    },
    {
        "Split": "Validation",
        "Documents": len(val_documents_df),
        "Entity annotations": len(val_entities_df),
        "SNOMED assignments": len(val_snomed_df)
    },
    {
        "Split": "Test",
        "Documents": len(test_documents_df),
        "Entity annotations": len(test_entities_df),
        "SNOMED assignments": len(test_snomed_df)
    }
])

display(split_summary)

# Verify that no document appears in multiple splits
assert train_doc_ids.isdisjoint(val_doc_ids)
assert train_doc_ids.isdisjoint(test_doc_ids)
assert val_doc_ids.isdisjoint(test_doc_ids)

# Verify that all annotations and mappings are accounted for
assert (
    len(train_entities_df)
    + len(val_entities_df)
    + len(test_entities_df)
    == len(working_entities_df)
)

assert (
    len(train_snomed_df)
    + len(val_snomed_df)
    + len(test_snomed_df)
    == len(snomed_concepts_df)
)

print("\nValidation passed!")
print("No document leakage between splits.")
print("All annotations and SNOMED assignments accounted for.")
print("Empty posts excluded from modelling:", 1250 - len(model_documents))

DOCUMENT-LEVEL SPLITS


,Split,Documents,Entity annotations,SNOMED assignments
0,Train,874,6365,6547
1,Validation,187,1283,1321
2,Test,187,1463,1505



Validation passed!
No document leakage between splits.
All annotations and SNOMED assignments accounted for.
Empty posts excluded from modelling: 2


#Check the split distribution

In [23]:
# Step 20: Check entity-label distribution across splits

split_frames = {
    "Train": train_entities_df,
    "Validation": val_entities_df,
    "Test": test_entities_df
}

label_counts = pd.DataFrame({
    split: frame["entity_type"].value_counts()
    for split, frame in split_frames.items()
}).fillna(0).astype(int)

# Calculate percentages within each split
label_percentages = (
    label_counts.div(label_counts.sum(axis=0), axis=1) * 100
).round(2)

print("ENTITY COUNTS BY SPLIT")
print("=" * 45)
display(label_counts)

print("\nENTITY PERCENTAGES BY SPLIT")
print("=" * 45)
display(label_percentages)

# Check concept coverage between training and test sets
train_concepts = set(
    train_snomed_df.loc[
        train_snomed_df["snomed_id"].notna()
        & (train_snomed_df["snomed_id"] != "CONCEPT_LESS"),
        "snomed_id"
    ].astype(str)
)

test_concepts = set(
    test_snomed_df.loc[
        test_snomed_df["snomed_id"].notna()
        & (test_snomed_df["snomed_id"] != "CONCEPT_LESS"),
        "snomed_id"
    ].astype(str)
)

unseen_test_concepts = test_concepts - train_concepts

print("\nSNOMED CONCEPT COVERAGE")
print("=" * 45)
print("Unique concepts in training:", len(train_concepts))
print("Unique concepts in test:", len(test_concepts))
print("Test concepts not represented in training:", len(unseen_test_concepts))

ENTITY COUNTS BY SPLIT


,Train,Validation,Test
entity_type,,,
ADR,4417,896,1005
Disease,200,41,42
Drug,1220,256,324
Finding,316,56,63
Symptom,212,34,29



ENTITY PERCENTAGES BY SPLIT


,Train,Validation,Test
entity_type,,,
ADR,69.40,69.84,68.69
Disease,3.14,3.20,2.87
Drug,19.17,19.95,22.15
Finding,4.96,4.36,4.31
Symptom,3.33,2.65,1.98



SNOMED CONCEPT COVERAGE
Unique concepts in training: 898
Unique concepts in test: 405
Test concepts not represented in training: 73


#Measure unseen patient expressions

In [24]:
# Step 21: Measure unseen expressions and unseen concepts

# Keep mappings with an assigned concept
train_norm = train_snomed_df[
    train_snomed_df["snomed_id"].notna()
    & (train_snomed_df["snomed_id"] != "CONCEPT_LESS")
].copy()

test_norm = test_snomed_df[
    test_snomed_df["snomed_id"].notna()
    & (test_snomed_df["snomed_id"] != "CONCEPT_LESS")
].copy()

# Normalize capitalization and whitespace only
train_norm["normalized_mention"] = (
    train_norm["mention"].map(normalize_for_comparison)
)

test_norm["normalized_mention"] = (
    test_norm["mention"].map(normalize_for_comparison)
)

# What concepts and surface phrases appear in training?
train_concepts = set(train_norm["snomed_id"].astype(str))

train_phrases = set(train_norm["normalized_mention"])

# Exact phrase + concept combinations seen during training
train_phrase_concept_pairs = set(
    zip(
        train_norm["normalized_mention"],
        train_norm["snomed_id"].astype(str)
    )
)

# Check the test annotations
test_norm["concept_seen_in_train"] = (
    test_norm["snomed_id"].astype(str).isin(train_concepts)
)

test_norm["phrase_seen_in_train"] = (
    test_norm["normalized_mention"].isin(train_phrases)
)

test_norm["phrase_concept_pair_seen"] = [
    (phrase, str(concept)) in train_phrase_concept_pairs
    for phrase, concept in zip(
        test_norm["normalized_mention"],
        test_norm["snomed_id"]
    )
]

# Assign an interpretable category to each test mapping
def classify_mapping(row):
    if not row["concept_seen_in_train"]:
        return "Concept unseen in training"

    if not row["phrase_seen_in_train"]:
        return "New phrase, known concept"

    if not row["phrase_concept_pair_seen"]:
        return "Known phrase, new concept mapping"

    return "Known phrase and concept mapping"


test_norm["novelty_category"] = test_norm.apply(
    classify_mapping,
    axis=1
)

print("TEST-SET NOVELTY ANALYSIS")
print("=" * 55)

display(
    test_norm["novelty_category"]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="Test concept assignments")
)

print("\nDistinct phrase statistics")
print("-" * 55)

print("Distinct test phrases:", test_norm["normalized_mention"].nunique())

print(
    "Test phrases absent from training:",
    len(
        set(test_norm["normalized_mention"])
        - train_phrases
    )
)

# Inspect examples from the key research category
unseen_phrase_known_concept = test_norm[
    (test_norm["novelty_category"] == "New phrase, known concept")
]

print("\nEXAMPLES: NEW PHRASE, KNOWN CONCEPT")
print("-" * 55)

display(
    unseen_phrase_known_concept[
        [
            "doc_id",
            "mention",
            "snomed_id",
            "snomed_name"
        ]
    ].drop_duplicates().head(20)
)

TEST-SET NOVELTY ANALYSIS


,Category,Test concept assignments
0,Known phrase and concept mapping,865
1,"New phrase, known concept",486
2,Concept unseen in training,83
3,"Known phrase, new concept mapping",6



Distinct phrase statistics
-------------------------------------------------------
Distinct test phrases: 820
Test phrases absent from training: 519

EXAMPLES: NEW PHRASE, KNOWN CONCEPT
-------------------------------------------------------


,doc_id,mention,snomed_id,snomed_name
25,ARTHROTEC.103,pain and stiffness,22253000,Pain
26,ARTHROTEC.103,so much pain,22253000,Pain
68,ARTHROTEC.110,uterine cramps,29542008,Uterine spasm
70,ARTHROTEC.110,Alot of gas,162076009,Excessive upper gastrointestinal gas
193,ARTHROTEC.128,mild cramps,55300003,Muscle cramp
203,ARTHROTEC.13,severe stomach upset,162059005,Upset stomach
206,ARTHROTEC.13,increased menstruation,386692008,Menorrhagia
207,ARTHROTEC.13,2-3 periods a month instead of once a month,266601003,Excessive and frequent menstruation
208,ARTHROTEC.13,menstrual cramps with vaginal bleeding,431416001,Menstrual cramp
209,ARTHROTEC.13,menstrual cramps without vaginal bleeding,431416001,Menstrual cramp


#Prepare clean medical concept normalization examples

In [25]:
# Step 22: Prepare clean medical concept normalization examples

# Start with mappings that have exactly one assigned concept
normalization_examples = snomed_concepts_df[
    (snomed_concepts_df["mapping_relation"] == "single")
    & snomed_concepts_df["snomed_id"].notna()
    & (snomed_concepts_df["snomed_id"] != "CONCEPT_LESS")
    & snomed_concepts_df["snomed_name"].notna()
    & snomed_concepts_df["mention"].notna()
].copy()

# Remove duplicate representations of the same source annotation
normalization_examples = normalization_examples.drop_duplicates(
    subset=["doc_id", "annotation_id", "span_key", "snomed_id"]
).copy()

# Use the document splits already established
doc_split = {
    **{doc_id: "train" for doc_id in train_doc_ids},
    **{doc_id: "validation" for doc_id in val_doc_ids},
    **{doc_id: "test" for doc_id in test_doc_ids}
}

normalization_examples["split"] = (
    normalization_examples["doc_id"].map(doc_split)
)

# Keep only rows belonging to our established experiment splits
normalization_examples = normalization_examples[
    normalization_examples["split"].notna()
].copy()

# Normalize case and whitespace for phrase comparisons only
normalization_examples["normalized_mention"] = (
    normalization_examples["mention"]
    .map(normalize_for_comparison)
)

train_norm_examples = normalization_examples[
    normalization_examples["split"] == "train"
].copy()

val_norm_examples = normalization_examples[
    normalization_examples["split"] == "validation"
].copy()

test_norm_examples = normalization_examples[
    normalization_examples["split"] == "test"
].copy()

# Determine which phrases and concepts the training set contains
train_phrases = set(
    train_norm_examples["normalized_mention"]
)

train_concepts = set(
    train_norm_examples["snomed_id"].astype(str)
)

# Measure novelty for each test example
test_norm_examples["phrase_seen_in_train"] = (
    test_norm_examples["normalized_mention"].isin(train_phrases)
)

test_norm_examples["concept_seen_in_train"] = (
    test_norm_examples["snomed_id"].astype(str).isin(train_concepts)
)


def classify_novelty(row):
    phrase_seen = row["phrase_seen_in_train"]
    concept_seen = row["concept_seen_in_train"]

    if phrase_seen and concept_seen:
        return "Known phrase + known concept"
    elif not phrase_seen and concept_seen:
        return "New phrase + known concept"
    elif phrase_seen and not concept_seen:
        return "Known phrase + unseen concept"
    else:
        return "New phrase + unseen concept"


test_norm_examples["novelty_category"] = (
    test_norm_examples.apply(classify_novelty, axis=1)
)

print("NORMALIZATION BENCHMARK")
print("=" * 50)

print("Training examples:", len(train_norm_examples))
print("Validation examples:", len(val_norm_examples))
print("Test examples:", len(test_norm_examples))

print("\nTest-set novelty categories:")
display(
    test_norm_examples["novelty_category"]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="Examples")
)

# Count unique unseen surface forms in the target category
new_phrase_known = test_norm_examples[
    test_norm_examples["novelty_category"]
    == "New phrase + known concept"
]

print(
    "\nNew phrases for known concepts:",
    len(new_phrase_known)
)

print(
    "Distinct new phrases for known concepts:",
    new_phrase_known["normalized_mention"].nunique()
)

print("\nExamples to examine:")
display(
    new_phrase_known[
        ["mention", "snomed_id", "snomed_name"]
    ].drop_duplicates().head(15)
)

NORMALIZATION BENCHMARK
Training examples: 5879
Validation examples: 1175
Test examples: 1356

Test-set novelty categories:


,Category,Examples
0,Known phrase + known concept,847
1,New phrase + known concept,428
2,New phrase + unseen concept,81



New phrases for known concepts: 428
Distinct new phrases for known concepts: 413

Examples to examine:


,mention,snomed_id,snomed_name
25,pain and stiffness,22253000,Pain
26,so much pain,22253000,Pain
68,uterine cramps,29542008,Uterine spasm
70,Alot of gas,162076009,Excessive upper gastrointestinal gas
193,mild cramps,55300003,Muscle cramp
203,severe stomach upset,162059005,Upset stomach
206,increased menstruation,386692008,Menorrhagia
207,2-3 periods a month instead of once a month,266601003,Excessive and frequent menstruation
208,menstrual cramps with vaginal bleeding,431416001,Menstrual cramp
209,menstrual cramps without vaginal bleeding,431416001,Menstrual cramp


#TF-IDF baseline for medical concept normalization


In [26]:
# Step 23: TF-IDF baseline for medical concept normalization

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

# --------------------------------------------------
# 1. Prepare the training and test phrases
# --------------------------------------------------

train_retrieval = (
    train_norm_examples[
        ["mention", "normalized_mention", "snomed_id", "snomed_name"]
    ]
    .drop_duplicates(
        subset=["normalized_mention", "snomed_id"]
    )
    .reset_index(drop=True)
)

evaluation_df = test_norm_examples.reset_index(drop=True).copy()

train_texts = train_retrieval["mention"].astype(str).tolist()
test_texts = evaluation_df["mention"].astype(str).tolist()

train_labels = train_retrieval["snomed_id"].astype(str).to_numpy()
test_labels = evaluation_df["snomed_id"].astype(str).to_numpy()

# Candidate concepts must occur in the training data
candidate_concepts = np.array(sorted(set(train_labels)))

print("TF-IDF NORMALIZATION BASELINE")
print("=" * 50)
print("Unique training phrase-concept pairs:", len(train_retrieval))
print("Candidate medical concepts:", len(candidate_concepts))
print("Test examples:", len(evaluation_df))


# --------------------------------------------------
# 2. Represent phrases using word-level TF-IDF
# --------------------------------------------------

word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    lowercase=True,
    strip_accents="unicode",
    sublinear_tf=True,
    dtype=np.float32
)

X_train_word = word_vectorizer.fit_transform(train_texts)
X_test_word = word_vectorizer.transform(test_texts)


# --------------------------------------------------
# 3. Represent phrases using character-level TF-IDF
# --------------------------------------------------

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    lowercase=True,
    sublinear_tf=True,
    dtype=np.float32
)

X_train_char = char_vectorizer.fit_transform(train_texts)
X_test_char = char_vectorizer.transform(test_texts)


# --------------------------------------------------
# 4. Combine word and character similarity
# --------------------------------------------------

word_similarity = (
    X_test_word @ X_train_word.T
).toarray()

char_similarity = (
    X_test_char @ X_train_char.T
).toarray()

# Fixed, equal weights for the initial baseline
similarity = (
    0.5 * word_similarity
    + 0.5 * char_similarity
)

print("\nSimilarity matrices created successfully.")


# --------------------------------------------------
# 5. Rank concepts by their best-matching phrase
# --------------------------------------------------

concept_scores = np.zeros(
    (len(evaluation_df), len(candidate_concepts)),
    dtype=np.float32
)

for j, concept_id in enumerate(candidate_concepts):

    matching_rows = np.flatnonzero(
        train_labels == concept_id
    )

    concept_scores[:, j] = similarity[
        :, matching_rows
    ].max(axis=1)

# Highest-scoring concept first
ranked_indices = np.argsort(
    -concept_scores, axis=1
)

ranked_concepts = candidate_concepts[ranked_indices]

evaluation_df["predicted_snomed_id"] = ranked_concepts[:, 0]
evaluation_df["top1_similarity"] = concept_scores[
    np.arange(len(evaluation_df)),
    ranked_indices[:, 0]
]

print("Concept ranking completed.")


# --------------------------------------------------
# 6. Evaluate only concepts represented in training
# --------------------------------------------------

def calculate_metrics(row_indices):

    ranks = []

    for i in row_indices:

        target = test_labels[i]

        matching_positions = np.flatnonzero(
            ranked_concepts[i] == target
        )

        # A target absent from the candidate list cannot
        # receive a correct rank in this closed-set baseline.
        rank = (
            int(matching_positions[0]) + 1
            if len(matching_positions)
            else len(candidate_concepts) + 1
        )

        ranks.append(rank)

    ranks = np.asarray(ranks)

    return {
        "Examples": len(ranks),
        "Accuracy@1": round(float(np.mean(ranks == 1)), 4),
        "Recall@5": round(float(np.mean(ranks <= 5)), 4),
        "MRR": round(float(np.mean(1 / ranks)), 4)
    }


print("\nBASELINE RESULTS BY NOVELTY CATEGORY")
print("=" * 65)

results = []

categories_to_evaluate = [
    "Known phrase + known concept",
    "New phrase + known concept"
]

for category in categories_to_evaluate:

    indices = np.flatnonzero(
        evaluation_df["novelty_category"].to_numpy() == category
    )

    if len(indices) == 0:
        continue

    metrics = calculate_metrics(indices)
    metrics["Category"] = category
    results.append(metrics)

results_df = pd.DataFrame(results)

display(
    results_df[
        ["Category", "Examples", "Accuracy@1", "Recall@5", "MRR"]
    ]
)

# Concepts absent from training are outside the baseline's
# closed candidate vocabulary, so report them separately.
unseen_indices = np.flatnonzero(
    evaluation_df["novelty_category"].to_numpy()
    == "New phrase + unseen concept"
)

print(
    "\nTest examples with concepts unseen during training:",
    len(unseen_indices)
)


# --------------------------------------------------
# 7. Inspect predictions on unseen phrases
# --------------------------------------------------

new_phrase_mask = (
    evaluation_df["novelty_category"]
    == "New phrase + known concept"
)

example_columns = [
    "mention",
    "snomed_name",
    "snomed_id",
    "predicted_snomed_id",
    "top1_similarity"
]

print("\nEXAMPLE PREDICTIONS: NEW PHRASE, KNOWN CONCEPT")
display(
    evaluation_df.loc[
        new_phrase_mask, example_columns
    ].head(20)
)

TF-IDF NORMALIZATION BASELINE
Unique training phrase-concept pairs: 2678
Candidate medical concepts: 885
Test examples: 1356

Similarity matrices created successfully.
Concept ranking completed.

BASELINE RESULTS BY NOVELTY CATEGORY


,Category,Examples,Accuracy@1,Recall@5,MRR
0,Known phrase + known concept,847,0.9835,0.9988,0.9913
1,New phrase + known concept,428,0.4836,0.7874,0.6202



Test examples with concepts unseen during training: 81

EXAMPLE PREDICTIONS: NEW PHRASE, KNOWN CONCEPT


,mention,snomed_name,snomed_id,predicted_snomed_id,top1_similarity
2,pain and stiffness,Pain,22253000,271587009,0.690190
3,so much pain,Pain,22253000,10601006,0.360480
5,uterine cramps,Uterine spasm,29542008,29542008,0.514986
7,Alot of gas,Excessive upper gastrointestinal gas,162076009,77692006,0.514025
12,mild cramps,Muscle cramp,55300003,55300003,0.661040
16,severe stomach upset,Upset stomach,162059005,162059005,0.821050
18,increased menstruation,Menorrhagia,386692008,161722004,0.417803
19,2-3 periods a month instead of once a month,Excessive and frequent menstruation,266601003,80182007,0.310516
20,menstrual cramps with vaginal bleeding,Menstrual cramp,431416001,431416001,0.677342
21,menstrual cramps without vaginal bleeding,Menstrual cramp,431416001,431416001,0.636600


#SapBERT, our biomedical semantic model

#Install the required library

In [27]:
%pip -q install transformers

#Load SapBERT and prepare the candidate concepts

In [28]:
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

MODEL_NAME = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

# Load the pretrained biomedical model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
sapbert = AutoModel.from_pretrained(MODEL_NAME).to(device)
sapbert.eval()

print("SapBERT loaded successfully!")


# Build the candidate concept dictionary using training data only
def most_common_name(names):
    names = names.dropna().astype(str).str.strip()
    names = names[names.ne("")]

    if names.empty:
        return None

    return names.value_counts().index[0]


candidate_table = (
    train_norm_examples.groupby("snomed_id")
    .agg(concept_name=("snomed_name", most_common_name))
    .reset_index()
)

candidate_table = candidate_table[
    candidate_table["concept_name"].notna()
].copy()

candidate_table["snomed_id"] = (
    candidate_table["snomed_id"].astype(str)
)

candidate_table = candidate_table.reset_index(drop=True)

print("Candidate concepts:", len(candidate_table))
display(candidate_table.head())

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

SapBERT loaded successfully!
Candidate concepts: 885


,snomed_id,concept_name
0,10001005,Bacterial sepsis
1,10085004,Metatarsalgia
2,101000119102,Numbness and tingling sensation of skin
3,102480002,Generalised acute body pains
4,102491009,Immobile


#Encode phrases and medical concepts

In [29]:
# Create SapBERT embeddings using the CLS representation

def embed_texts(texts, batch_size=32, max_length=64):
    embeddings = []

    for start in tqdm(range(0, len(texts), batch_size)):
        batch = [
            str(text) if pd.notna(text) else ""
            for text in texts[start:start + batch_size]
        ]

        tokens = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        tokens = {
            key: value.to(device)
            for key, value in tokens.items()
        }

        with torch.inference_mode():
            outputs = sapbert(**tokens)

            # SapBERT model card specifies the CLS representation
            batch_embeddings = outputs.last_hidden_state[:, 0, :]

            # Normalize so dot product equals cosine similarity
            batch_embeddings = torch.nn.functional.normalize(
                batch_embeddings, p=2, dim=1
            )

        embeddings.append(batch_embeddings.cpu().numpy())

    return np.concatenate(embeddings, axis=0)


# Encode the test patient phrases and the training-derived concept labels
sap_eval_df = test_norm_examples.reset_index(drop=True).copy()

test_embeddings = embed_texts(
    sap_eval_df["mention"].fillna("").tolist()
)

concept_embeddings = embed_texts(
    candidate_table["concept_name"].tolist()
)

print("Test embeddings:", test_embeddings.shape)
print("Concept embeddings:", concept_embeddings.shape)

  0%|          | 0/43 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

Test embeddings: (1356, 768)
Concept embeddings: (885, 768)


#Rank concepts and evaluate SapBERT

In [30]:
# Calculate cosine similarities between every test phrase and concept
similarities = test_embeddings @ concept_embeddings.T

candidate_ids = candidate_table["snomed_id"].to_numpy()
candidate_names = candidate_table["concept_name"].to_numpy()

ranked_positions = np.argsort(-similarities, axis=1)
ranked_candidate_ids = candidate_ids[ranked_positions]

sap_eval_df["predicted_snomed_id"] = ranked_candidate_ids[:, 0]
sap_eval_df["predicted_snomed_name"] = candidate_names[
    ranked_positions[:, 0]
]

sap_eval_df["top1_similarity"] = similarities[
    np.arange(len(sap_eval_df)),
    ranked_positions[:, 0]
]


def evaluate_sapbert(indices):
    ranks = []

    for i in indices:
        gold_id = str(sap_eval_df.iloc[i]["snomed_id"])

        matches = np.flatnonzero(ranked_candidate_ids[i] == gold_id)

        if len(matches):
            ranks.append(int(matches[0]) + 1)
        else:
            ranks.append(len(candidate_ids) + 1)

    ranks = np.asarray(ranks)

    return {
        "Examples": len(ranks),
        "Accuracy@1": round(float(np.mean(ranks == 1)), 4),
        "Recall@5": round(float(np.mean(ranks <= 5)), 4),
        "MRR": round(float(np.mean(1 / ranks)), 4)
    }


# Evaluate the same two groups as our TF-IDF baseline
sapbert_results = []

for category in [
    "Known phrase + known concept",
    "New phrase + known concept"
]:
    indices = np.flatnonzero(
        sap_eval_df["novelty_category"].to_numpy() == category
    )

    metrics = evaluate_sapbert(indices)
    metrics["Category"] = category
    sapbert_results.append(metrics)

sapbert_results_df = pd.DataFrame(sapbert_results)

print("SAPBERT RESULTS")
print("=" * 60)

display(
    sapbert_results_df[
        ["Category", "Examples", "Accuracy@1", "Recall@5", "MRR"]
    ]
)

print("\nExample predictions for new phrases:")
display(
    sap_eval_df[
        sap_eval_df["novelty_category"]
        == "New phrase + known concept"
    ][
        [
            "mention",
            "snomed_name",
            "predicted_snomed_name",
            "top1_similarity"
        ]
    ].head(20)
)

SAPBERT RESULTS


,Category,Examples,Accuracy@1,Recall@5,MRR
0,Known phrase + known concept,847,0.8512,0.9457,0.8935
1,New phrase + known concept,428,0.5327,0.7921,0.6485



Example predictions for new phrases:


,mention,snomed_name,predicted_snomed_name,top1_similarity
2,pain and stiffness,Pain,Joint stiffness,0.816684
3,so much pain,Pain,Severe pain,0.876837
5,uterine cramps,Uterine spasm,Uterine spasm,0.834981
7,Alot of gas,Excessive upper gastrointestinal gas,Excessive upper gastrointestinal gas,0.582230
12,mild cramps,Muscle cramp,Muscle cramp,0.747890
16,severe stomach upset,Upset stomach,Upset stomach,0.895832
18,increased menstruation,Menorrhagia,Excessive and frequent menstruation,0.918632
19,2-3 periods a month instead of once a month,Excessive and frequent menstruation,Prolonged periods,0.598827
20,menstrual cramps with vaginal bleeding,Menstrual cramp,Menstrual cramp,0.830217
21,menstrual cramps without vaginal bleeding,Menstrual cramp,Menstrual cramp,0.839662


#Compare SapBERT against TF-IDF

In [31]:
# Compare the two models on the same evaluation groups

tfidf_comparison = results_df[
    ["Category", "Accuracy@1", "Recall@5", "MRR"]
].copy()

tfidf_comparison = tfidf_comparison.rename(columns={
    "Accuracy@1": "TF-IDF Accuracy@1",
    "Recall@5": "TF-IDF Recall@5",
    "MRR": "TF-IDF MRR"
})

sapbert_comparison = sapbert_results_df[
    ["Category", "Accuracy@1", "Recall@5", "MRR"]
].rename(columns={
    "Accuracy@1": "SapBERT Accuracy@1",
    "Recall@5": "SapBERT Recall@5",
    "MRR": "SapBERT MRR"
})

comparison_df = tfidf_comparison.merge(
    sapbert_comparison,
    on="Category",
    how="inner"
)

display(comparison_df)

,Category,TF-IDF Accuracy@1,TF-IDF Recall@5,TF-IDF MRR,SapBERT Accuracy@1,SapBERT Recall@5,SapBERT MRR
0,Known phrase + known concept,0.9835,0.9988,0.9913,0.8512,0.9457,0.8935
1,New phrase + known concept,0.4836,0.7874,0.6202,0.5327,0.7921,0.6485


#Create a fairer TF-IDF comparison

In [32]:
# Step 26: TF-IDF concept-name retrieval baseline
# Uses the same candidate concepts as SapBERT.

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Same candidate concepts and names used by SapBERT
candidate_ids = candidate_table["snomed_id"].astype(str).to_numpy()
candidate_names = candidate_table["concept_name"].astype(str).tolist()

query_texts = evaluation_df["mention"].fillna("").astype(str).tolist()
gold_ids = evaluation_df["snomed_id"].astype(str).to_numpy()
categories = evaluation_df["novelty_category"].to_numpy()


# Word-based TF-IDF: fit on candidate concept names
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    lowercase=True,
    strip_accents="unicode",
    sublinear_tf=True
)

concept_word = word_vectorizer.fit_transform(candidate_names)
query_word = word_vectorizer.transform(query_texts)

word_scores = (query_word @ concept_word.T).toarray()


# Character-based TF-IDF: fit on the same candidate names
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    lowercase=True,
    sublinear_tf=True
)

concept_char = char_vectorizer.fit_transform(candidate_names)
query_char = char_vectorizer.transform(query_texts)

char_scores = (query_char @ concept_char.T).toarray()


# Combine word and character similarity
concept_name_scores = 0.5 * word_scores + 0.5 * char_scores

# Rank the same candidate IDs for each query
tfidf_concept_positions = np.argsort(
    -concept_name_scores, axis=1
)

tfidf_concept_ranked_ids = candidate_ids[
    tfidf_concept_positions
]


# Evaluate the two known-concept categories
def evaluate_rankings(ranked_ids, gold, mask):
    indices = np.flatnonzero(mask)
    ranks = []

    for i in indices:
        matches = np.flatnonzero(ranked_ids[i] == gold[i])

        rank = (
            int(matches[0]) + 1
            if len(matches)
            else len(candidate_ids) + 1
        )

        ranks.append(rank)

    ranks = np.asarray(ranks)

    return {
        "Examples": len(ranks),
        "Accuracy@1": round(float(np.mean(ranks == 1)), 4),
        "Recall@5": round(float(np.mean(ranks <= 5)), 4),
        "MRR": round(float(np.mean(1 / ranks)), 4)
    }


matched_results = []

for category in [
    "Known phrase + known concept",
    "New phrase + known concept"
]:

    mask = categories == category

    # Matched TF-IDF concept-name baseline
    tfidf_metrics = evaluate_rankings(
        tfidf_concept_ranked_ids, gold_ids, mask
    )

    # SapBERT on exactly the same examples and candidate list
    sapbert_metrics = evaluate_rankings(
        ranked_candidate_ids, gold_ids, mask
    )

    matched_results.append({
        "Category": category,
        "Examples": tfidf_metrics["Examples"],
        "TF-IDF Accuracy@1": tfidf_metrics["Accuracy@1"],
        "SapBERT Accuracy@1": sapbert_metrics["Accuracy@1"],
        "TF-IDF Recall@5": tfidf_metrics["Recall@5"],
        "SapBERT Recall@5": sapbert_metrics["Recall@5"],
        "TF-IDF MRR": tfidf_metrics["MRR"],
        "SapBERT MRR": sapbert_metrics["MRR"]
    })

matched_comparison_df = pd.DataFrame(matched_results)

print("MATCHED CANDIDATE-SET COMPARISON")
print("=" * 65)
display(matched_comparison_df)

MATCHED CANDIDATE-SET COMPARISON


,Category,Examples,TF-IDF Accuracy@1,SapBERT Accuracy@1,TF-IDF Recall@5,SapBERT Recall@5,TF-IDF MRR,SapBERT MRR
0,Known phrase + known concept,847,0.7166,0.8512,0.7816,0.9457,0.7464,0.8935
1,New phrase + known concept,428,0.3294,0.5327,0.5584,0.7921,0.4347,0.6485


#Check whether the improvement is reliable

In [33]:
# Step 27: Document-level bootstrap confidence intervals

import numpy as np
import pandas as pd

# Focus on our main research group
eligible_mask = (
    evaluation_df["novelty_category"].to_numpy()
    == "New phrase + known concept"
)

eligible_indices = np.flatnonzero(eligible_mask)

subset_df = (
    evaluation_df.iloc[eligible_indices]
    .reset_index(drop=True)
)

gold = gold_ids[eligible_indices]

tfidf_ranked = tfidf_concept_ranked_ids[eligible_indices]
sapbert_ranked = ranked_candidate_ids[eligible_indices]


def get_gold_ranks(ranked_ids, gold_ids):
    ranks = []

    for i, target in enumerate(gold_ids):
        positions = np.flatnonzero(ranked_ids[i] == target)

        rank = (
            int(positions[0]) + 1
            if len(positions)
            else ranked_ids.shape[1] + 1
        )

        ranks.append(rank)

    return np.asarray(ranks)


tfidf_ranks = get_gold_ranks(tfidf_ranked, gold)
sapbert_ranks = get_gold_ranks(sapbert_ranked, gold)


def metrics_from_ranks(ranks):
    return np.array([
        np.mean(ranks == 1),       # Accuracy@1
        np.mean(ranks <= 5),       # Recall@5
        np.mean(1 / ranks)         # MRR
    ])


# Observed differences: SapBERT minus TF-IDF
observed_delta = (
    metrics_from_ranks(sapbert_ranks)
    - metrics_from_ranks(tfidf_ranks)
)

# Group rows by document to account for within-document dependence
doc_ids = subset_df["doc_id"].to_numpy()
unique_docs = np.unique(doc_ids)

rows_by_doc = {
    doc: np.flatnonzero(doc_ids == doc)
    for doc in unique_docs
}

# Bootstrap whole documents with replacement
rng = np.random.default_rng(42)
n_bootstrap = 2000
bootstrap_deltas = []

for _ in range(n_bootstrap):

    sampled_docs = rng.choice(
        unique_docs,
        size=len(unique_docs),
        replace=True
    )

    sampled_rows = np.concatenate([
        rows_by_doc[doc] for doc in sampled_docs
    ])

    delta = (
        metrics_from_ranks(sapbert_ranks[sampled_rows])
        - metrics_from_ranks(tfidf_ranks[sampled_rows])
    )

    bootstrap_deltas.append(delta)

bootstrap_deltas = np.asarray(bootstrap_deltas)

# Percentile confidence intervals
lower = np.percentile(bootstrap_deltas, 2.5, axis=0)
upper = np.percentile(bootstrap_deltas, 97.5, axis=0)

metric_names = ["Accuracy@1", "Recall@5", "MRR"]

bootstrap_results = pd.DataFrame({
    "Metric": metric_names,
    "Observed improvement": observed_delta,
    "95% CI lower": lower,
    "95% CI upper": upper
})

print("DOCUMENT-LEVEL BOOTSTRAP")
print("=" * 55)
print("Test examples:", len(subset_df))
print("Distinct documents:", len(unique_docs))
print("Bootstrap repetitions:", n_bootstrap)

display(bootstrap_results.round(4))

DOCUMENT-LEVEL BOOTSTRAP
Test examples: 428
Distinct documents: 145
Bootstrap repetitions: 2000


,Metric,Observed improvement,95% CI lower,95% CI upper
0,Accuracy@1,0.2033,0.1432,0.2608
1,Recall@5,0.2336,0.1752,0.2911
2,MRR,0.2138,0.1630,0.2623


#Investigate where SapBERT wins and loses

In [34]:
# Step 28: Compare TF-IDF and SapBERT predictions

import numpy as np
import pandas as pd

# The target group for our main research question
new_phrase_mask = (
    evaluation_df["novelty_category"].to_numpy()
    == "New phrase + known concept"
)

indices = np.flatnonzero(new_phrase_mask)

# Gold labels and top-ranked predictions
gold_ids = evaluation_df["snomed_id"].astype(str).to_numpy()

tfidf_predictions = tfidf_concept_ranked_ids[:, 0].astype(str)
sapbert_predictions = ranked_candidate_ids[:, 0].astype(str)

# Candidate ID -> standardized concept name
concept_name_lookup = dict(
    zip(
        candidate_table["snomed_id"].astype(str),
        candidate_table["concept_name"].astype(str)
    )
)

# Build a comparison table
error_analysis_df = evaluation_df[
    ["doc_id", "mention", "snomed_id", "snomed_name"]
].copy()

error_analysis_df["gold_id"] = gold_ids
error_analysis_df["tfidf_prediction_id"] = tfidf_predictions
error_analysis_df["sapbert_prediction_id"] = sapbert_predictions

error_analysis_df["tfidf_prediction"] = (
    error_analysis_df["tfidf_prediction_id"]
    .map(concept_name_lookup)
)

error_analysis_df["sapbert_prediction"] = (
    error_analysis_df["sapbert_prediction_id"]
    .map(concept_name_lookup)
)

error_analysis_df["tfidf_correct"] = (
    error_analysis_df["tfidf_prediction_id"]
    == error_analysis_df["gold_id"]
)

error_analysis_df["sapbert_correct"] = (
    error_analysis_df["sapbert_prediction_id"]
    == error_analysis_df["gold_id"]
)

error_analysis_df["comparison"] = np.select(
    [
        error_analysis_df["tfidf_correct"]
        & error_analysis_df["sapbert_correct"],

        ~error_analysis_df["tfidf_correct"]
        & error_analysis_df["sapbert_correct"],

        error_analysis_df["tfidf_correct"]
        & ~error_analysis_df["sapbert_correct"]
    ],
    [
        "Both correct",
        "SapBERT only correct",
        "TF-IDF only correct"
    ],
    default="Both wrong"
)

# Focus only on new phrases linked to known concepts
new_phrase_results = error_analysis_df.loc[indices].copy()

print("ERROR ANALYSIS: NEW PHRASES, KNOWN CONCEPTS")
print("=" * 60)

display(
    new_phrase_results["comparison"]
    .value_counts()
    .rename_axis("Outcome")
    .reset_index(name="Examples")
)

columns_to_show = [
    "mention",
    "snomed_name",
    "tfidf_prediction",
    "sapbert_prediction"
]

print("\nCases where SapBERT succeeds and TF-IDF fails:")
display(
    new_phrase_results[
        new_phrase_results["comparison"]
        == "SapBERT only correct"
    ][columns_to_show].head(15)
)

print("\nCases where TF-IDF succeeds and SapBERT fails:")
display(
    new_phrase_results[
        new_phrase_results["comparison"]
        == "TF-IDF only correct"
    ][columns_to_show].head(15)
)

print("\nCases where both models fail:")
display(
    new_phrase_results[
        new_phrase_results["comparison"] == "Both wrong"
    ][columns_to_show].head(15)
)

ERROR ANALYSIS: NEW PHRASES, KNOWN CONCEPTS


,Outcome,Examples
0,Both wrong,167
1,SapBERT only correct,120
2,Both correct,108
3,TF-IDF only correct,33



Cases where SapBERT succeeds and TF-IDF fails:


,mention,snomed_name,tfidf_prediction,sapbert_prediction
5,uterine cramps,Uterine spasm,Hand cramps,Uterine spasm
12,mild cramps,Muscle cramp,Hand cramps,Muscle cramp
20,menstrual cramps with vaginal bleeding,Menstrual cramp,Vaginal bleeding,Menstrual cramp
21,menstrual cramps without vaginal bleeding,Menstrual cramp,Vaginal bleeding,Menstrual cramp
22,menopausal phase,Menopause present,Postmenopausal bleeding,Menopause present
47,"""back labor"" pains",Backache,Low back pain,Backache
49,increase in cramping,Muscle cramp,Cramp in limb,Muscle cramp
78,so much pain,Severe pain,Pain,Severe pain
89,stomach gas,Excessive upper gastrointestinal gas,Stomach ache,Excessive upper gastrointestinal gas
90,feel like a bubble of air is stuck in my chest,Tight chest,Chest pain,Tight chest



Cases where TF-IDF succeeds and SapBERT fails:


,mention,snomed_name,tfidf_prediction,sapbert_prediction
3,so much pain,Pain,Pain,Severe pain
113,rash on my wrist,Rash,Rash,Pain in wrist
165,tummy bloating,Abdominal bloating,Abdominal bloating,Bloating symptom
256,blured vision,Blurred vision - hazy,Blurred vision - hazy,Blurring of visual image
436,burning sensations in calves,Burning sensation,Burning sensation,Pain in calf
462,could barely walk up stairs,Difficulty walking up stairs,Difficulty walking up stairs,Unable to climb stairs
512,high liver enzyme,Liver enzymes abnormal,Liver enzymes abnormal,Increased liver function
534,arthrites,Arthritis,Arthritis,Arthritis/arthrosis
605,memory lapses,Memory impairment,Memory impairment,Transient memory loss
638,Blured vision,Blurred vision - hazy,Blurred vision - hazy,Blurring of visual image



Cases where both models fail:


,mention,snomed_name,tfidf_prediction,sapbert_prediction
2,pain and stiffness,Pain,Stiffness,Joint stiffness
18,increased menstruation,Menorrhagia,Irregular menstruation,Excessive and frequent menstruation
19,2-3 periods a month instead of once a month,Excessive and frequent menstruation,Prolonged periods,Prolonged periods
28,blood in my stools,Haematochezia,Dark stools,Dark stools
29,major cramps,Muscle cramp,Hand cramps,Cramp in limb
33,diahorea,Diarrhoea,Pain in forearm,Avandia
34,passing blood and clots with stoals,Haematochezia,Blood clot in eye,Syncope
45,pain in the pit of my stomach,Stomach ache,Pain in the coccyx,Stomach cramps
46,deep ache from my waist down to the heels of m...,Generalised aches and pains,Unable to lie down,Heel pain
56,excessive mestrual bleeding,Menorrhagia,Bleeding,Excessive and frequent menstruation


#Test a hybrid model

In [35]:
# Step 29: Tune a hybrid retrieval model on validation data

import numpy as np
import pandas as pd

# Candidate IDs and names established from training data
candidate_ids = candidate_table["snomed_id"].astype(str).to_numpy()
candidate_names = candidate_table["concept_name"].astype(str).tolist()

train_phrases = set(train_norm_examples["normalized_mention"])
train_concepts = set(train_norm_examples["snomed_id"].astype(str))


# 1. Prepare validation examples
val_eval_df = val_norm_examples.reset_index(drop=True).copy()

val_eval_df["concept_seen_in_train"] = (
    val_eval_df["snomed_id"].astype(str).isin(train_concepts)
)

val_eval_df["phrase_seen_in_train"] = (
    val_eval_df["normalized_mention"].isin(train_phrases)
)

val_eval_df["novelty_category"] = np.select(
    [
        val_eval_df["concept_seen_in_train"]
        & val_eval_df["phrase_seen_in_train"],

        val_eval_df["concept_seen_in_train"]
        & ~val_eval_df["phrase_seen_in_train"]
    ],
    [
        "Known phrase + known concept",
        "New phrase + known concept"
    ],
    default="Concept unseen in training"
)

val_texts = val_eval_df["mention"].fillna("").astype(str).tolist()


# 2. Create TF-IDF rankings using the same concept-name vocabulary

val_word = word_vectorizer.transform(val_texts)
val_char = char_vectorizer.transform(val_texts)

val_tfidf_scores = (
    0.5 * (val_word @ concept_word.T).toarray()
    + 0.5 * (val_char @ concept_char.T).toarray()
)

val_tfidf_positions = np.argsort(-val_tfidf_scores, axis=1)


# 3. Create SapBERT rankings over the same concepts

val_embeddings = embed_texts(val_texts)

val_sapbert_scores = val_embeddings @ concept_embeddings.T

val_sapbert_positions = np.argsort(
    -val_sapbert_scores, axis=1
)


# 4. Convert rankings into reciprocal-rank-fusion scores

def rrf_scores(rank_positions, k=60):
    n_rows, n_candidates = rank_positions.shape

    scores = np.zeros(
        (n_rows, n_candidates),
        dtype=np.float32
    )

    rank_values = np.arange(1, n_candidates + 1)

    scores[
        np.arange(n_rows)[:, None],
        rank_positions
    ] = 1.0 / (k + rank_values[None, :])

    return scores


val_tfidf_rrf = rrf_scores(val_tfidf_positions)
val_sapbert_rrf = rrf_scores(val_sapbert_positions)


# 5. Evaluate rankings consistently

def evaluate_rankings(rank_positions, gold_ids, mask):
    indices = np.flatnonzero(mask)
    ranks = []

    for i in indices:
        target_positions = np.flatnonzero(
            candidate_ids == str(gold_ids[i])
        )

        if len(target_positions) == 0:
            ranks.append(len(candidate_ids) + 1)
            continue

        target_position = target_positions[0]

        rank = np.flatnonzero(
            rank_positions[i] == target_position
        )[0] + 1

        ranks.append(int(rank))

    ranks = np.asarray(ranks)

    if len(ranks) == 0:
        return {
            "Examples": 0,
            "Accuracy@1": np.nan,
            "Recall@5": np.nan,
            "MRR": np.nan
        }

    return {
        "Examples": len(ranks),
        "Accuracy@1": float(np.mean(ranks == 1)),
        "Recall@5": float(np.mean(ranks <= 5)),
        "MRR": float(np.mean(1.0 / ranks))
    }


gold_val = val_eval_df["snomed_id"].astype(str).to_numpy()

new_phrase_mask = (
    val_eval_df["novelty_category"].to_numpy()
    == "New phrase + known concept"
)


# 6. Select the weight using validation Accuracy@1
# Higher weight means greater emphasis on SapBERT.

weights = [0.0, 0.25, 0.5, 0.75, 1.0]
validation_results = []

for weight in weights:

    hybrid_scores = (
        weight * val_sapbert_rrf
        + (1 - weight) * val_tfidf_rrf
    )

    hybrid_positions = np.argsort(
        -hybrid_scores, axis=1
    )

    metrics = evaluate_rankings(
        hybrid_positions,
        gold_val,
        new_phrase_mask
    )

    validation_results.append({
        "SapBERT weight": weight,
        **metrics
    })

validation_results_df = pd.DataFrame(validation_results)

print("VALIDATION RESULTS: NEW PHRASES, KNOWN CONCEPTS")
display(validation_results_df.round(4))

# Select the weight with best validation Accuracy@1,
# using MRR as the tie-breaker.
best_row = validation_results_df.sort_values(
    ["Accuracy@1", "MRR"],
    ascending=False
).iloc[0]

best_weight = float(best_row["SapBERT weight"])

print("\nSelected SapBERT weight:", best_weight)
print("Selection uses validation data only.")

  0%|          | 0/37 [00:00<?, ?it/s]

VALIDATION RESULTS: NEW PHRASES, KNOWN CONCEPTS


,SapBERT weight,Examples,Accuracy@1,Recall@5,MRR
0,0.00,325,0.3200,0.5354,0.4224
1,0.25,325,0.3723,0.5938,0.4801
2,0.50,325,0.4492,0.6708,0.5547
3,0.75,325,0.4923,0.7385,0.6070
4,1.00,325,0.5323,0.8246,0.6594



Selected SapBERT weight: 1.0
Selection uses validation data only.


#Evaluate the selected hybrid on the test set

In [36]:
# Step 30: Final hybrid test evaluation

# These rankings use the test examples and candidate names
# already prepared in the earlier experiment.
test_tfidf_rrf = rrf_scores(tfidf_concept_positions)
test_sapbert_rrf = rrf_scores(ranked_positions)

test_eval_df = evaluation_df.reset_index(drop=True).copy()

gold_test = test_eval_df["snomed_id"].astype(str).to_numpy()

test_new_phrase_mask = (
    test_eval_df["novelty_category"].to_numpy()
    == "New phrase + known concept"
)

test_known_phrase_mask = (
    test_eval_df["novelty_category"].to_numpy()
    == "Known phrase + known concept"
)

hybrid_test_scores = (
    best_weight * test_sapbert_rrf
    + (1 - best_weight) * test_tfidf_rrf
)

hybrid_test_positions = np.argsort(
    -hybrid_test_scores, axis=1
)

final_results = []

for category, mask in [
    ("Known phrase + known concept", test_known_phrase_mask),
    ("New phrase + known concept", test_new_phrase_mask)
]:

    # TF-IDF and SapBERT rankings
    tfidf_metrics = evaluate_rankings(
        tfidf_concept_positions, gold_test, mask
    )

    sapbert_metrics = evaluate_rankings(
        ranked_positions, gold_test, mask
    )

    hybrid_metrics = evaluate_rankings(
        hybrid_test_positions, gold_test, mask
    )

    final_results.append({
        "Category": category,
        "Examples": hybrid_metrics["Examples"],
        "TF-IDF Accuracy@1": tfidf_metrics["Accuracy@1"],
        "SapBERT Accuracy@1": sapbert_metrics["Accuracy@1"],
        "Hybrid Accuracy@1": hybrid_metrics["Accuracy@1"],
        "TF-IDF MRR": tfidf_metrics["MRR"],
        "SapBERT MRR": sapbert_metrics["MRR"],
        "Hybrid MRR": hybrid_metrics["MRR"]
    })

final_comparison_df = pd.DataFrame(final_results)

print("FINAL TEST COMPARISON")
display(final_comparison_df.round(4))

FINAL TEST COMPARISON


,Category,Examples,TF-IDF Accuracy@1,SapBERT Accuracy@1,Hybrid Accuracy@1,TF-IDF MRR,SapBERT MRR,Hybrid MRR
0,Known phrase + known concept,847,0.7166,0.8512,0.8512,0.7464,0.8935,0.8935
1,New phrase + known concept,428,0.3294,0.5327,0.5327,0.4347,0.6485,0.6485


In [37]:
print("Selected SapBERT weight:", best_weight)

Selected SapBERT weight: 1.0


#Prepare for adverse-reaction extraction

In [38]:
# Step 31: Audit ADR spans for the extraction task

adr_annotations = working_entities_df[
    working_entities_df["entity_type"] == "ADR"
].copy()

adr_continuous = adr_annotations[
    ~adr_annotations["is_discontinuous"]
].copy()

adr_discontinuous = adr_annotations[
    adr_annotations["is_discontinuous"]
].copy()

print("ADR EXTRACTION DATA AUDIT")
print("=" * 50)

print("Total ADR annotations:", len(adr_annotations))
print("Continuous ADR spans:", len(adr_continuous))
print("Discontinuous ADR annotations:", len(adr_discontinuous))

print("\nBy document split:")

for split_name, doc_ids in [
    ("Train", train_doc_ids),
    ("Validation", val_doc_ids),
    ("Test", test_doc_ids)
]:
    split_adr = adr_annotations[
        adr_annotations["doc_id"].isin(doc_ids)
    ]

    print(
        f"{split_name}: {len(split_adr)} total ADR annotations; "
        f"{(~split_adr['is_discontinuous']).sum()} continuous; "
        f"{split_adr['is_discontinuous'].sum()} discontinuous"
    )

# Inspect actual continuous spans in their original context
sample_rows = []

for row in adr_continuous.head(10).itertuples(index=False):
    original_text = text_lookup[row.doc_id]
    start, end = row.spans[0]

    sample_rows.append({
        "doc_id": row.doc_id,
        "label": row.entity_type,
        "annotated_phrase": row.mention,
        "text_at_offsets": original_text[start:end]
    })

print("\nExample continuous ADR spans:")
display(pd.DataFrame(sample_rows))

ADR EXTRACTION DATA AUDIT
Total ADR annotations: 6318
Continuous ADR spans: 5316
Discontinuous ADR annotations: 1002

By document split:
Train: 4417 total ADR annotations; 3713 continuous; 704 discontinuous
Validation: 896 total ADR annotations; 767 continuous; 129 discontinuous
Test: 1005 total ADR annotations; 836 continuous; 169 discontinuous

Example continuous ADR spans:


,doc_id,label,annotated_phrase,text_at_offsets
0,ARTHROTEC.1,ADR,bit drowsy,bit drowsy
1,ARTHROTEC.1,ADR,little blurred vision,little blurred vision
2,ARTHROTEC.1,ADR,gastric problems,gastric problems
3,ARTHROTEC.1,ADR,feel a bit weird,feel a bit weird
4,ARTHROTEC.10,ADR,Hunger pangs,Hunger pangs
5,ARTHROTEC.100,ADR,vaginal bleeding,vaginal bleeding
6,ARTHROTEC.100,ADR,stomach pain,stomach pain
7,ARTHROTEC.100,ADR,canker sores,canker sores
8,ARTHROTEC.100,ADR,headache,headache
9,ARTHROTEC.101,ADR,shortness of breath,shortness of breath


#Check overlapping continuous ADR spans

In [39]:
# Step 32: Find overlaps between continuous ADR annotations

continuous_segments = []

for row in adr_continuous.itertuples(index=False):
    start, end = row.spans[0]

    continuous_segments.append({
        "doc_id": row.doc_id,
        "entity_id": row.entity_id,
        "start": start,
        "end": end,
        "mention": row.mention
    })

segments_df = pd.DataFrame(continuous_segments)

overlap_records = []
affected_annotations = set()

for doc_id, group in segments_df.groupby("doc_id"):

    rows = group.sort_values(
        ["start", "end"]
    ).to_dict("records")

    active = []

    for current in rows:
        start = current["start"]
        end = current["end"]

        # Retain spans that may overlap the current span
        active = [
            previous for previous in active
            if previous["end"] > start
        ]

        for previous in active:
            if max(previous["start"], start) < min(
                previous["end"], end
            ):
                overlap_records.append({
                    "doc_id": doc_id,
                    "entity_1": previous["entity_id"],
                    "mention_1": previous["mention"],
                    "entity_2": current["entity_id"],
                    "mention_2": current["mention"]
                })

                affected_annotations.add(
                    (doc_id, previous["entity_id"])
                )
                affected_annotations.add(
                    (doc_id, current["entity_id"])
                )

        active.append(current)


continuous_overlap_df = pd.DataFrame(overlap_records)

print("CONTINUOUS ADR OVERLAP AUDIT")
print("=" * 50)
print("Continuous ADR annotations:", len(adr_continuous))
print("Overlapping pairs:", len(continuous_overlap_df))
print(
    "Distinct annotations involved in overlaps:",
    len(affected_annotations)
)

if not continuous_overlap_df.empty:
    print(
        "Documents affected:",
        continuous_overlap_df["doc_id"].nunique()
    )

    print("\nExample overlapping spans:")
    display(continuous_overlap_df.head(15))

CONTINUOUS ADR OVERLAP AUDIT
Continuous ADR annotations: 5316
Overlapping pairs: 0
Distinct annotations involved in overlaps: 0


#Check one final overlap case

In [40]:
# Step 33: Check continuous ADRs against discontinuous ADR segments

adr_annotations = working_entities_df[
    working_entities_df["entity_type"] == "ADR"
].copy()

continuous_adrs = adr_annotations[
    ~adr_annotations["is_discontinuous"]
]

discontinuous_adrs = adr_annotations[
    adr_annotations["is_discontinuous"]
]

cross_overlap_records = []

for doc_id in continuous_adrs["doc_id"].unique():

    continuous_group = continuous_adrs[
        continuous_adrs["doc_id"] == doc_id
    ]

    discontinuous_group = discontinuous_adrs[
        discontinuous_adrs["doc_id"] == doc_id
    ]

    for continuous_row in continuous_group.itertuples(index=False):

        c_start, c_end = continuous_row.spans[0]

        for discontinuous_row in discontinuous_group.itertuples(index=False):

            for d_start, d_end in discontinuous_row.spans:

                if max(c_start, d_start) < min(c_end, d_end):

                    cross_overlap_records.append({
                        "doc_id": doc_id,
                        "continuous_entity": continuous_row.entity_id,
                        "continuous_mention": continuous_row.mention,
                        "discontinuous_entity": discontinuous_row.entity_id,
                        "discontinuous_mention": discontinuous_row.mention,
                        "overlapping_text": text_lookup[doc_id][
                            max(c_start, d_start):min(c_end, d_end)
                        ]
                    })


cross_overlap_df = pd.DataFrame(cross_overlap_records)

print("CONTINUOUS / DISCONTINUOUS ADR OVERLAP")
print("=" * 50)
print("Continuous ADR annotations:", len(continuous_adrs))
print("Discontinuous ADR annotations:", len(discontinuous_adrs))
print("Cross-type overlapping span pairs:", len(cross_overlap_df))

if not cross_overlap_df.empty:
    display(cross_overlap_df.head(15))

CONTINUOUS / DISCONTINUOUS ADR OVERLAP
Continuous ADR annotations: 5316
Discontinuous ADR annotations: 1002
Cross-type overlapping span pairs: 5


,doc_id,continuous_entity,continuous_mention,discontinuous_entity,discontinuous_mention,overlapping_text
0,DICLOFENAC-SODIUM.7,T8,renal and respiratory failure,T7,renal failure,renal
1,DICLOFENAC-SODIUM.7,T8,renal and respiratory failure,T7,renal failure,failure
2,LIPITOR.620,T4,back of neck,T12,Tingling in back of neck,back of neck
3,LIPITOR.620,T4,back of neck,T6,burning in back of neck,back of neck
4,VOLTAREN.34,T4,pain in my kidney area,T5,pain in stomach,pain in


#Prepare the BioBERT extraction data

In [41]:
# Step 34: Prepare continuous ADR extraction data for BioBERT

import numpy as np
import pandas as pd
from transformers import AutoTokenizer

MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"

# BIO label IDs
LABEL2ID = {
    "O": 0,
    "B-ADR": 1,
    "I-ADR": 2
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# 1. Exclude documents containing continuous/discontinuous ADR overlap
conflict_docs = set(cross_overlap_df["doc_id"])

valid_documents = model_documents[
    ~model_documents["doc_id"].isin(conflict_docs)
].copy()

split_sets = {
    "train": set(train_doc_ids) - conflict_docs,
    "validation": set(val_doc_ids) - conflict_docs,
    "test": set(test_doc_ids) - conflict_docs
}

valid_doc_ids = set(valid_documents["doc_id"])

assert set.union(*split_sets.values()) == valid_doc_ids
assert not (
    split_sets["train"] & split_sets["validation"]
)
assert not (
    split_sets["train"] & split_sets["test"]
)
assert not (
    split_sets["validation"] & split_sets["test"]
)

# 2. Collect continuous ADR spans
continuous_spans_by_doc = {}

for row in adr_continuous.itertuples(index=False):
    if row.doc_id in valid_doc_ids:
        continuous_spans_by_doc.setdefault(
            row.doc_id, []
        ).append(row.spans[0])

# 3. Collect segments that belong to discontinuous ADR annotations.
# These regions will be masked from the training loss.
ignored_spans_by_doc = {}

for row in adr_discontinuous.itertuples(index=False):
    if row.doc_id in valid_doc_ids:
        ignored_spans_by_doc.setdefault(
            row.doc_id, []
        ).extend(row.spans)

# 4. Load a fast tokenizer so character offsets are available
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

assert tokenizer.is_fast, "A fast tokenizer is required."

MAX_LENGTH = 512
STRIDE = 128

# Keep every document's split assignment alongside its text
doc_records = []

for row in valid_documents.itertuples(index=False):
    if row.doc_id in split_sets["train"]:
        split = "train"
    elif row.doc_id in split_sets["validation"]:
        split = "validation"
    elif row.doc_id in split_sets["test"]:
        split = "test"
    else:
        continue

    doc_records.append({
        "doc_id": row.doc_id,
        "text": row.text,
        "split": split
    })

# 5. Tokenize long posts into overlapping windows
encoded = tokenizer(
    [row["text"] for row in doc_records],
    max_length=MAX_LENGTH,
    truncation=True,
    stride=STRIDE,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding="max_length"
)

# Each window points back to its source document
window_to_document = encoded["overflow_to_sample_mapping"]
offset_mappings = encoded["offset_mapping"]

prepared_chunks = {
    "train": [],
    "validation": [],
    "test": []
}

chunk_metadata = {
    "train": [],
    "validation": [],
    "test": []
}

covered_spans = set()

# 6. Align BIO labels using the original character offsets
for window_idx, document_idx in enumerate(window_to_document):

    doc = doc_records[int(document_idx)]
    doc_id = doc["doc_id"]
    split = doc["split"]

    offsets = [
        tuple(map(int, pair))
        for pair in offset_mappings[window_idx]
    ]

    # Ignore special and padding tokens with zero-length offsets
    real_offsets = [
        (start, end)
        for start, end in offsets
        if end > start
    ]

    if not real_offsets:
        continue

    chunk_start = min(start for start, end in real_offsets)
    chunk_end = max(end for start, end in real_offsets)

    labels = [
        -100 if end <= start else LABEL2ID["O"]
        for start, end in offsets
    ]

    gold_spans = continuous_spans_by_doc.get(doc_id, [])
    ignored_spans = ignored_spans_by_doc.get(doc_id, [])

    # A target span must be fully contained in a window before
    # we assign its BIO labels. Partially truncated target spans
    # will instead be ignored in that window.
    contained_spans = []
    partial_spans = []

    for entity_start, entity_end in gold_spans:

        overlaps_window = (
            max(entity_start, chunk_start)
            < min(entity_end, chunk_end)
        )

        if (
            entity_start >= chunk_start
            and entity_end <= chunk_end
        ):
            contained_spans.append(
                (entity_start, entity_end)
            )
        elif overlaps_window:
            partial_spans.append(
                (entity_start, entity_end)
            )

    # Ignore tokens belonging to a discontinuous ADR segment
    # or to an entity that is truncated by this particular window.
    for token_idx, (token_start, token_end) in enumerate(offsets):

        if token_end <= token_start:
            labels[token_idx] = -100
            continue

        overlaps_ignored = any(
            max(token_start, seg_start)
            < min(token_end, seg_end)
            for seg_start, seg_end in ignored_spans
        )

        overlaps_partial = any(
            max(token_start, entity_start)
            < min(token_end, entity_end)
            for entity_start, entity_end in partial_spans
        )

        if overlaps_ignored or overlaps_partial:
            labels[token_idx] = -100

    # Label the continuous ADR spans
    for entity_start, entity_end in contained_spans:

        token_indices = [
            idx
            for idx, (token_start, token_end) in enumerate(offsets)
            if (
                token_end > token_start
                and max(token_start, entity_start)
                < min(token_end, entity_end)
            )
        ]

        if not token_indices:
            continue

        # First subword token = B-ADR; remaining tokens = I-ADR
        labels[token_indices[0]] = LABEL2ID["B-ADR"]

        for token_idx in token_indices[1:]:
            labels[token_idx] = LABEL2ID["I-ADR"]

        covered_spans.add(
            (doc_id, entity_start, entity_end)
        )

    # Store model inputs. Keep character offsets separately for evaluation.
    feature = {
        "input_ids": encoded["input_ids"][window_idx],
        "attention_mask": encoded["attention_mask"][window_idx],
        "labels": labels
    }

    if "token_type_ids" in encoded:
        feature["token_type_ids"] = (
            encoded["token_type_ids"][window_idx]
        )

    prepared_chunks[split].append(feature)

    chunk_metadata[split].append({
        "doc_id": doc_id,
        "offsets": offsets
    })


# 7. Check the output
print("BIO EXTRACTION DATA PREPARATION")
print("=" * 55)

for split in ["train", "validation", "test"]:

    rows = prepared_chunks[split]

    n_b = sum(
        label == LABEL2ID["B-ADR"]
        for row in rows
        for label in row["labels"]
    )

    n_i = sum(
        label == LABEL2ID["I-ADR"]
        for row in rows
        for label in row["labels"]
    )

    n_masked = sum(
        label == -100
        for row in rows
        for label in row["labels"]
    )

    print(f"\n{split.upper()}")
    print("Documents:", len(split_sets[split]))
    print("Token windows:", len(rows))
    print("B-ADR tokens:", n_b)
    print("I-ADR tokens:", n_i)
    print("Ignored tokens:", n_masked)

total_expected_spans = sum(
    len(continuous_spans_by_doc.get(doc_id, []))
    for doc_id in valid_doc_ids
)

print("\nSPAN-COVERAGE CHECK")
print("Continuous ADR spans in eligible documents:",
      total_expected_spans)
print("Spans represented in at least one window:",
      len(covered_spans))
print("Spans not yet represented:",
      total_expected_spans - len(covered_spans))

print("\nExcluded conflict documents:", sorted(conflict_docs))
print("Preparation complete.")

BIO EXTRACTION DATA PREPARATION

TRAIN
Documents: 872
Token windows: 874
B-ADR tokens: 3701
I-ADR tokens: 8525
Ignored tokens: 354272

VALIDATION
Documents: 187
Token windows: 188
B-ADR tokens: 768
I-ADR tokens: 1770
Ignored tokens: 76626

TEST
Documents: 186
Token windows: 187
B-ADR tokens: 832
I-ADR tokens: 1929
Ignored tokens: 74040

SPAN-COVERAGE CHECK
Continuous ADR spans in eligible documents: 5300
Spans represented in at least one window: 5294
Spans not yet represented: 6

Excluded conflict documents: ['DICLOFENAC-SODIUM.7', 'LIPITOR.620', 'VOLTAREN.34']
Preparation complete.


#Verify the six-span difference

In [42]:
# Step 35: Check unique span boundaries and actual coverage

# List every expected continuous ADR span in eligible documents
expected_span_records = [
    (doc_id, start, end)
    for doc_id in valid_doc_ids
    for start, end in continuous_spans_by_doc.get(doc_id, [])
]

# Count unique document + character-span combinations
expected_unique_spans = set(expected_span_records)

# The coverage set already records spans represented in token windows
missing_unique_spans = expected_unique_spans - covered_spans

print("FINAL SPAN-COVERAGE CHECK")
print("=" * 55)

print("Expected annotation occurrences:", len(expected_span_records))
print("Unique span boundaries:", len(expected_unique_spans))
print(
    "Repeated boundary occurrences:",
    len(expected_span_records) - len(expected_unique_spans)
)
print("Unique spans represented:", len(expected_unique_spans & covered_spans))
print("Unique spans not represented:", len(missing_unique_spans))

# Show any genuinely unrepresented spans
if missing_unique_spans:
    print("\nSpans requiring investigation:")

    missing_rows = []

    for doc_id, start, end in sorted(missing_unique_spans):
        text = text_lookup[doc_id]

        missing_rows.append({
            "doc_id": doc_id,
            "start": start,
            "end": end,
            "span_length": end - start,
            "annotated_text": text[start:end]
        })

    display(pd.DataFrame(missing_rows))
else:
    print("\nAll unique continuous ADR spans are represented!")

FINAL SPAN-COVERAGE CHECK
Expected annotation occurrences: 5300
Unique span boundaries: 5300
Repeated boundary occurrences: 0
Unique spans represented: 5294
Unique spans not represented: 6

Spans requiring investigation:


,doc_id,start,end,span_length,annotated_text
0,LIPITOR.546,2298,2312,14,sinus problems
1,LIPITOR.546,2509,2528,19,high blood pressure
2,LIPITOR.546,2576,2586,10,nightmares
3,LIPITOR.546,3180,3212,32,left shoulder is almost immobile
4,LIPITOR.546,3226,3249,23,tinnitus in my left ear
5,LIPITOR.546,3316,3338,22,lost my sense of taste


#Rebuild the token windows correctly

In [43]:
# ============================================================
# STEP 36: Rebuild BioBERT extraction data correctly
# ============================================================

import numpy as np
import pandas as pd
from transformers import AutoTokenizer

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"

MAX_LENGTH = 512
STRIDE = 128

LABEL2ID = {
    "O": 0,
    "B-ADR": 1,
    "I-ADR": 2
}

ID2LABEL = {value: key for key, value in LABEL2ID.items()}

# Load the fast tokenizer so we can obtain character offsets.
# Hugging Face will use the cached files if already downloaded.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

assert tokenizer.is_fast, (
    "Please use a fast BioBERT tokenizer for offset mapping."
)

assert tokenizer.cls_token_id is not None
assert tokenizer.sep_token_id is not None
assert tokenizer.pad_token_id is not None


# ------------------------------------------------------------
# 2. Prepare the eligible documents and splits
# ------------------------------------------------------------

# Remove the two empty posts from modelling.
model_documents = documents_df[
    documents_df["text"].fillna("").str.strip().ne("")
].copy()

# Exclude the three documents containing ADR overlap conflicts.
conflict_docs = set(cross_overlap_df["doc_id"])

valid_documents = model_documents[
    ~model_documents["doc_id"].isin(conflict_docs)
].copy()

train_doc_ids_clean = set(train_doc_ids) - conflict_docs
val_doc_ids_clean = set(val_doc_ids) - conflict_docs
test_doc_ids_clean = set(test_doc_ids) - conflict_docs

split_sets = {
    "train": train_doc_ids_clean,
    "validation": val_doc_ids_clean,
    "test": test_doc_ids_clean
}

# Confirm that splits remain separate.
assert train_doc_ids_clean.isdisjoint(val_doc_ids_clean)
assert train_doc_ids_clean.isdisjoint(test_doc_ids_clean)
assert val_doc_ids_clean.isdisjoint(test_doc_ids_clean)

document_split_lookup = {}

for split_name, document_ids in split_sets.items():
    for doc_id in document_ids:
        document_split_lookup[doc_id] = split_name

valid_doc_ids = set(valid_documents["doc_id"])

assert valid_doc_ids == set(document_split_lookup.keys()), (
    "The eligible documents do not match the document split assignments."
)

text_lookup = documents_df.set_index("doc_id")["text"].to_dict()

# Re-create document records in a stable order.
doc_records = []

for row in valid_documents.sort_values("doc_id").itertuples(index=False):

    doc_id = row.doc_id

    doc_records.append({
        "doc_id": doc_id,
        "text": str(row.text),
        "split": document_split_lookup[doc_id]
    })


# ------------------------------------------------------------
# 3. Collect continuous and discontinuous ADR spans
# ------------------------------------------------------------

adr_annotations = working_entities_df[
    working_entities_df["entity_type"] == "ADR"
].copy()

adr_continuous = adr_annotations[
    ~adr_annotations["is_discontinuous"]
].copy()

adr_discontinuous = adr_annotations[
    adr_annotations["is_discontinuous"]
].copy()

continuous_spans_by_doc = {}
ignored_spans_by_doc = {}

for row in adr_continuous.itertuples(index=False):

    if row.doc_id not in valid_doc_ids:
        continue

    continuous_spans_by_doc.setdefault(
        row.doc_id, []
    ).extend(row.spans)


for row in adr_discontinuous.itertuples(index=False):

    if row.doc_id not in valid_doc_ids:
        continue

    # Each discontinuous annotation can contain multiple segments.
    ignored_spans_by_doc.setdefault(
        row.doc_id, []
    ).extend(row.spans)


# Confirm there are no continuous/discontinuous ADR overlaps
# in the documents retained for this benchmark.
remaining_cross_overlaps = cross_overlap_df[
    cross_overlap_df["doc_id"].isin(valid_doc_ids)
]

assert remaining_cross_overlaps.empty, (
    "Overlapping ADR annotations remain in eligible documents."
)


# ------------------------------------------------------------
# 4. Create the output containers
# ------------------------------------------------------------

prepared_chunks = {
    "train": [],
    "validation": [],
    "test": []
}

chunk_metadata = {
    "train": [],
    "validation": [],
    "test": []
}

# Store each document + character-span combination represented
# in at least one complete token window.
covered_spans = set()

special_token_count = tokenizer.num_special_tokens_to_add(
    pair=False
)

payload_size = MAX_LENGTH - special_token_count
step_size = payload_size - STRIDE

assert step_size > 0, "Stride must be smaller than the payload size."


# ------------------------------------------------------------
# 5. Tokenize complete documents and create windows
# ------------------------------------------------------------

for doc in doc_records:

    doc_id = doc["doc_id"]
    split = doc["split"]
    text = doc["text"]

    # Tokenize the whole document without truncation.
    # These offsets refer to the original patient text.
    document_encoding = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_offsets_mapping=True
    )

    all_ids = document_encoding["input_ids"]

    all_offsets = [
        (int(start), int(end))
        for start, end in document_encoding["offset_mapping"]
    ]

    if not all_ids:
        continue

    gold_spans = continuous_spans_by_doc.get(doc_id, [])
    ignored_spans = ignored_spans_by_doc.get(doc_id, [])

    # Build overlapping windows while guaranteeing the final
    # tokens of every document are processed.
    window_start = 0

    while window_start < len(all_ids):

        window_end = min(
            window_start + payload_size,
            len(all_ids)
        )

        sub_ids = all_ids[window_start:window_end]

        sub_offsets = all_offsets[
            window_start:window_end
        ]

        # Manually add BERT special tokens.
        # This avoids tokenizer.prepare_for_model().
        input_ids = (
            [tokenizer.cls_token_id]
            + list(sub_ids)
            + [tokenizer.sep_token_id]
        )

        offsets = (
            [(0, 0)]
            + list(sub_offsets)
            + [(0, 0)]
        )

        attention_mask = [1] * len(input_ids)
        token_type_ids = [0] * len(input_ids)

        # Pad every sequence to MAX_LENGTH.
        padding_length = MAX_LENGTH - len(input_ids)

        if padding_length < 0:
            raise ValueError(
                f"Window in {doc_id} exceeds MAX_LENGTH."
            )

        input_ids += [
            tokenizer.pad_token_id
        ] * padding_length

        attention_mask += [0] * padding_length
        token_type_ids += [0] * padding_length

        offsets += [(0, 0)] * padding_length

        assert len(input_ids) == MAX_LENGTH
        assert len(attention_mask) == MAX_LENGTH
        assert len(token_type_ids) == MAX_LENGTH
        assert len(offsets) == MAX_LENGTH

        # Determine the actual character boundaries covered by
        # this window, ignoring special tokens and padding.
        real_offsets = [
            (start, end)
            for start, end in offsets
            if end > start
        ]

        if not real_offsets:
            break

        chunk_start = min(
            start for start, end in real_offsets
        )

        chunk_end = max(
            end for start, end in real_offsets
        )

        # -100 tells the training loss to ignore special tokens,
        # padding and intentionally masked regions.
        labels = [
            -100 if end <= start else LABEL2ID["O"]
            for start, end in offsets
        ]

        contained_spans = []
        partial_spans = []

        # A target span must be fully inside a window for its
        # complete BIO sequence to be supervised there.
        for entity_start, entity_end in gold_spans:

            overlaps_window = (
                max(entity_start, chunk_start)
                < min(entity_end, chunk_end)
            )

            if (
                entity_start >= chunk_start
                and entity_end <= chunk_end
            ):
                contained_spans.append(
                    (entity_start, entity_end)
                )

            elif overlaps_window:
                partial_spans.append(
                    (entity_start, entity_end)
                )

        # Mask discontinuous ADR segments so they are not
        # incorrectly treated as confirmed negative tokens.
        # Also mask fragments of entities cut by a window.
        for token_idx, (token_start, token_end) in enumerate(offsets):

            if token_end <= token_start:
                labels[token_idx] = -100
                continue

            overlaps_discontinuous = any(
                max(token_start, seg_start)
                < min(token_end, seg_end)
                for seg_start, seg_end in ignored_spans
            )

            overlaps_partial = any(
                max(token_start, entity_start)
                < min(token_end, entity_end)
                for entity_start, entity_end in partial_spans
            )

            if overlaps_discontinuous or overlaps_partial:
                labels[token_idx] = -100

        # Assign BIO labels to every fully contained continuous ADR.
        for entity_start, entity_end in contained_spans:

            token_indices = [
                idx
                for idx, (token_start, token_end) in enumerate(offsets)
                if (
                    token_end > token_start
                    and max(token_start, entity_start)
                    < min(token_end, entity_end)
                )
            ]

            if not token_indices:
                continue

            # First token is B-ADR; remaining tokens are I-ADR.
            labels[token_indices[0]] = LABEL2ID["B-ADR"]

            for token_idx in token_indices[1:]:
                labels[token_idx] = LABEL2ID["I-ADR"]

            covered_spans.add(
                (doc_id, entity_start, entity_end)
            )

        # Store only the inputs needed by BioBERT.
        feature = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "labels": labels
        }

        prepared_chunks[split].append(feature)

        # Store offsets separately for span-level evaluation later.
        chunk_metadata[split].append({
            "doc_id": doc_id,
            "offsets": offsets
        })

        # Exit when the final token has been included.
        if window_end == len(all_ids):
            break

        window_start += step_size


# ------------------------------------------------------------
# 6. Validate span coverage
# ------------------------------------------------------------

expected_spans = {
    (doc_id, start, end)
    for doc_id in valid_doc_ids
    for start, end in continuous_spans_by_doc.get(doc_id, [])
}

missing_spans = expected_spans - covered_spans

print("BIO EXTRACTION DATA PREPARATION")
print("=" * 55)

print("Excluded conflict documents:", sorted(conflict_docs))
print("Eligible documents:", len(valid_doc_ids))

for split in ["train", "validation", "test"]:

    windows = prepared_chunks[split]

    b_count = sum(
        label == LABEL2ID["B-ADR"]
        for feature in windows
        for label in feature["labels"]
    )

    i_count = sum(
        label == LABEL2ID["I-ADR"]
        for feature in windows
        for label in feature["labels"]
    )

    ignored_count = sum(
        label == -100
        for feature in windows
        for label in feature["labels"]
    )

    print(f"\n{split.upper()}")
    print("Documents:", len(split_sets[split]))
    print("Token windows:", len(windows))
    print("B-ADR tokens:", b_count)
    print("I-ADR tokens:", i_count)
    print("Ignored tokens:", ignored_count)


print("\nSPAN-COVERAGE VALIDATION")
print("-" * 55)

print("Expected unique ADR spans:", len(expected_spans))
print("Spans represented in windows:", len(expected_spans & covered_spans))
print("Missing spans:", len(missing_spans))

if missing_spans:

    print("\nSpans requiring investigation:")

    missing_rows = []

    for doc_id, start, end in sorted(missing_spans):

        missing_rows.append({
            "doc_id": doc_id,
            "start": start,
            "end": end,
            "span_length": end - start,
            "annotated_text": text_lookup[doc_id][start:end]
        })

    display(pd.DataFrame(missing_rows))

else:
    print("All eligible continuous ADR spans are represented!")


# ------------------------------------------------------------
# 7. Final structural checks
# ------------------------------------------------------------

for split in ["train", "validation", "test"]:
    for feature in prepared_chunks[split]:

        assert len(feature["input_ids"]) == MAX_LENGTH
        assert len(feature["attention_mask"]) == MAX_LENGTH
        assert len(feature["token_type_ids"]) == MAX_LENGTH
        assert len(feature["labels"]) == MAX_LENGTH

print("\nAll structural checks completed.")

BIO EXTRACTION DATA PREPARATION
Excluded conflict documents: ['DICLOFENAC-SODIUM.7', 'LIPITOR.620', 'VOLTAREN.34']
Eligible documents: 1245

TRAIN
Documents: 872
Token windows: 875
B-ADR tokens: 3709
I-ADR tokens: 8555
Ignored tokens: 354174

VALIDATION
Documents: 187
Token windows: 188
B-ADR tokens: 768
I-ADR tokens: 1770
Ignored tokens: 76474

TEST
Documents: 186
Token windows: 187
B-ADR tokens: 832
I-ADR tokens: 1929
Ignored tokens: 73985

SPAN-COVERAGE VALIDATION
-------------------------------------------------------
Expected unique ADR spans: 5300
Spans represented in windows: 5300
Missing spans: 0
All eligible continuous ADR spans are represented!

All structural checks completed.


#Fine-tune BioBERT

#Install the training dependencies

In [44]:
%pip -q install accelerate seqeval

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [45]:
%pip install -q seqeval accelerate

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [46]:
import sys
import subprocess

print("Python version:", sys.version)
print("Python executable:", sys.executable)

result = subprocess.run(
    [
        sys.executable, "-m", "pip",
        "install", "seqeval", "accelerate"
    ],
    capture_output=True,
    text=True
)

print("\nInstallation exit code:", result.returncode)

print("\n--- INSTALLATION OUTPUT ---")
print(result.stdout[-5000:] or "No standard output.")

print("\n--- ERROR DETAILS ---")
print(result.stderr[-5000:] or "No error text captured.")

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Python executable: /usr/bin/python3

Installation exit code: 1

--- INSTALLATION OUTPUT ---
  Using cached seqeval-1.2.2.tar.gz (43 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'


--- ERROR DETAILS ---
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.



In [47]:
%pip install -q --upgrade accelerate

In [48]:
import accelerate

print("Accelerate version:", accelerate.__version__)
print("Accelerate is ready!")

Accelerate version: 1.15.0
Accelerate is ready!


#Train BioBERT

In [49]:
# ============================================================
# STEP 37: Fine-tune BioBERT for ADR extraction
# ============================================================

import inspect
import numpy as np
import torch
import accelerate

from torch.utils.data import Dataset
from transformers import (
    AutoConfig,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"

LABEL2ID = {
    "O": 0,
    "B-ADR": 1,
    "I-ADR": 2
}

ID2LABEL = {
    0: "O",
    1: "B-ADR",
    2: "I-ADR"
}

device = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 60)
print("BIOBERT ADR EXTRACTION")
print("=" * 60)
print("Device:", device)
print("PyTorch version:", torch.__version__)
print("Accelerate version:", accelerate.__version__)

if device == "cpu":
    print(
        "\nNOTE: Training on CPU can be slow. "
        "Please allow time for all four epochs."
    )

# Check the prepared data from Step 36.
for split in ["train", "validation", "test"]:
    if split not in prepared_chunks:
        raise ValueError(
            f"Missing {split} data. Please rerun Step 36."
        )

if len(prepared_chunks["train"]) == 0:
    raise ValueError("Training data is empty.")

if len(prepared_chunks["validation"]) == 0:
    raise ValueError("Validation data is empty.")

print("\nTraining windows:", len(prepared_chunks["train"]))
print("Validation windows:", len(prepared_chunks["validation"]))
print("Test windows:", len(prepared_chunks["test"]))


# ------------------------------------------------------------
# 2. CREATE PYTORCH DATASETS
# ------------------------------------------------------------

class ADRTokenDataset(Dataset):

    def __init__(self, features):
        self.features = features

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        feature = self.features[index]

        return {
            key: torch.tensor(feature[key], dtype=torch.long)
            for key in [
                "input_ids",
                "attention_mask",
                "token_type_ids",
                "labels"
            ]
        }


train_dataset = ADRTokenDataset(
    prepared_chunks["train"]
)

val_dataset = ADRTokenDataset(
    prepared_chunks["validation"]
)

print("\nPyTorch datasets created successfully.")


# ------------------------------------------------------------
# 3. LOAD BIOBERT WITH EXACTLY THREE LABELS
# ------------------------------------------------------------

config = AutoConfig.from_pretrained(MODEL_NAME)

# Set the number of labels and explicitly overwrite the
# pretrained configuration's original label mapping.
config.num_labels = 3
config.id2label = ID2LABEL.copy()
config.label2id = LABEL2ID.copy()

assert config.num_labels == 3
assert len(config.id2label) == 3
assert len(config.label2id) == 3

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    config=config,
    ignore_mismatched_sizes=True
)

print("\nBioBERT token-classification model loaded.")
print("Number of labels:", model.config.num_labels)
print("Label mapping:", model.config.id2label)


# ------------------------------------------------------------
# 4. CALCULATE TOKEN-LEVEL PRECISION, RECALL AND F1
# ------------------------------------------------------------

def compute_metrics(eval_prediction):
    """
    Calculate token-level ADR precision, recall and F1.

    Tokens with gold label -100 are ignored.
    An ADR token has either B-ADR or I-ADR as its label.
    """

    logits = eval_prediction.predictions
    gold_labels = eval_prediction.label_ids

    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=-1)

    # Ignore special tokens, padding and deliberately masked tokens.
    valid_mask = gold_labels != -100

    gold_is_adr = (
        (gold_labels == LABEL2ID["B-ADR"])
        | (gold_labels == LABEL2ID["I-ADR"])
    )

    predicted_is_adr = (
        (predictions == LABEL2ID["B-ADR"])
        | (predictions == LABEL2ID["I-ADR"])
    )

    true_positive = int(np.sum(
        valid_mask & gold_is_adr & predicted_is_adr
    ))

    false_positive = int(np.sum(
        valid_mask & ~gold_is_adr & predicted_is_adr
    ))

    false_negative = int(np.sum(
        valid_mask & gold_is_adr & ~predicted_is_adr
    ))

    precision = (
        true_positive / (true_positive + false_positive)
        if true_positive + false_positive > 0
        else 0.0
    )

    recall = (
        true_positive / (true_positive + false_negative)
        if true_positive + false_negative > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "token_precision": precision,
        "token_recall": recall,
        "token_f1": f1
    }


# ------------------------------------------------------------
# 5. CONFIGURE TRAINING
# ------------------------------------------------------------

training_kwargs = {
    "output_dir": "./cadec_biobert_adr",
    "num_train_epochs": 4,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 2,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "save_total_limit": 2,
    "seed": 42,
    "data_seed": 42,
    "fp16": torch.cuda.is_available(),
    "report_to": "none",
    "logging_steps": 25
}

# Support different versions of Transformers.
argument_names = inspect.signature(
    TrainingArguments.__init__
).parameters

if "eval_strategy" in argument_names:
    training_kwargs["eval_strategy"] = "epoch"
elif "evaluation_strategy" in argument_names:
    training_kwargs["evaluation_strategy"] = "epoch"
else:
    raise RuntimeError(
        "Could not find a supported evaluation-strategy argument "
        "in TrainingArguments. Please send me the error."
    )

# Remove unsupported arguments instead of allowing them to
# cause another unexpected-keyword error.
unsupported_arguments = [
    key for key in training_kwargs
    if key not in argument_names
]

if unsupported_arguments:
    print(
        "\nSkipping unsupported TrainingArguments:",
        unsupported_arguments
    )

training_kwargs = {
    key: value
    for key, value in training_kwargs.items()
    if key in argument_names
}

training_args = TrainingArguments(**training_kwargs)

print("\nTraining configuration created successfully.")


# ------------------------------------------------------------
# 6. INITIALIZE THE TRAINER
# ------------------------------------------------------------

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": val_dataset,
    "compute_metrics": compute_metrics
}

trainer_parameters = inspect.signature(
    Trainer.__init__
).parameters

# Handle tokenizer argument changes between Transformers versions.
if "processing_class" in trainer_parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("Trainer initialized successfully.")


# ------------------------------------------------------------
# 7. TRAIN BIOBERT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STARTING BIOBERT TRAINING")
print("=" * 60)

train_output = trainer.train()

print("\nTraining completed!")

print("\nTraining summary:")
for metric_name, metric_value in train_output.metrics.items():
    print(f"{metric_name}: {metric_value}")


# ------------------------------------------------------------
# 8. EVALUATE THE SELECTED CHECKPOINT ON VALIDATION DATA
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("BEST CHECKPOINT")
print("=" * 60)

print(
    "Selected checkpoint:",
    trainer.state.best_model_checkpoint
)

validation_metrics = trainer.evaluate(
    eval_dataset=val_dataset
)

print("\nValidation metrics:")

for metric_name, metric_value in validation_metrics.items():
    if isinstance(metric_value, (int, float)):
        print(f"{metric_name}: {metric_value:.4f}")
    else:
        print(f"{metric_name}: {metric_value}")


# ------------------------------------------------------------
# 9. SAVE THE BEST MODEL
# ------------------------------------------------------------

BEST_MODEL_DIR = "./cadec_biobert_adr/best_model"

trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("\n" + "=" * 60)
print("MODEL SAVED SUCCESSFULLY")
print("=" * 60)
print("Saved model directory:", BEST_MODEL_DIR)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

BIOBERT ADR EXTRACTION
Device: cuda
PyTorch version: 2.11.0+cu130
Accelerate version: 1.15.0

Training windows: 875
Validation windows: 188
Test windows: 187

PyTorch datasets created successfully.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	


BioBERT token-classification model loaded.
Number of labels: 3
Label mapping: {0: 'O', 1: 'B-ADR', 2: 'I-ADR'}

Training configuration created successfully.
Trainer initialized successfully.

STARTING BIOBERT TRAINING


Epoch,Training Loss,Validation Loss,Token Precision,Token Recall,Token F1
1,0.329334,0.132537,0.838119,0.793538,0.815220
2,0.226415,0.132369,0.793422,0.845942,0.818841
3,0.166571,0.139734,0.820791,0.833727,0.827209
4,0.138394,0.154718,0.814264,0.841214,0.827519


...


Training completed!

Training summary:
train_runtime: 156.1691
train_samples_per_second: 22.412
train_steps_per_second: 2.817
total_flos: 914546916864000.0
train_loss: 0.27021541920575226
epoch: 4.0

BEST CHECKPOINT
Selected checkpoint: ./cadec_biobert_adr/checkpoint-220


Training Loss,Validation Loss,Epoch,Token Precision,Token Recall,Token F1
0.138394,0.132369,4,0.793422,0.845942,0.818841



Validation metrics:
eval_loss: 0.1324
eval_token_precision: 0.7934
eval_token_recall: 0.8459
eval_token_f1: 0.8188


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


MODEL SAVED SUCCESSFULLY
Saved model directory: ./cadec_biobert_adr/best_model
Best checkpoint: ./cadec_biobert_adr/checkpoint-220


#Select BioBERT using validation F1

In [50]:
# Step 38: Retrain BioBERT and select by validation token F1

from transformers import (
    AutoConfig,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments
)
import inspect

# Use a separate output directory to preserve the first run.
f1_training_kwargs = training_kwargs.copy()

f1_training_kwargs["output_dir"] = (
    "./cadec_biobert_adr_f1select"
)

# Select the checkpoint using our extraction metric.
f1_training_kwargs["metric_for_best_model"] = "eval_token_f1"
f1_training_kwargs["greater_is_better"] = True

# Reload the pretrained model with the correct labels.
f1_config = AutoConfig.from_pretrained(MODEL_NAME)

f1_config.num_labels = 3
f1_config.id2label = ID2LABEL.copy()
f1_config.label2id = LABEL2ID.copy()

f1_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    config=f1_config,
    ignore_mismatched_sizes=True
)

# Initialize the training arguments.
f1_training_args = TrainingArguments(
    **f1_training_kwargs
)

# Set up the trainer.
f1_trainer_kwargs = {
    "model": f1_model,
    "args": f1_training_args,
    "train_dataset": train_dataset,
    "eval_dataset": val_dataset,
    "compute_metrics": compute_metrics
}

trainer_parameters = inspect.signature(
    Trainer.__init__
).parameters

if "processing_class" in trainer_parameters:
    f1_trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_parameters:
    f1_trainer_kwargs["tokenizer"] = tokenizer

trainer_f1 = Trainer(**f1_trainer_kwargs)

# Fine-tune and select the checkpoint with the best validation F1.
print("Starting BioBERT training with F1-based checkpoint selection...")

f1_train_output = trainer_f1.train()

print("\nTraining completed!")
print("Best checkpoint:", trainer_f1.state.best_model_checkpoint)

print("\nValidation results for the selected checkpoint:")
f1_validation_metrics = trainer_f1.evaluate()

for name, value in f1_validation_metrics.items():
    if isinstance(value, (int, float)):
        print(f"{name}: {value:.4f}")

# Save the selected model.
BEST_MODEL_DIR_F1 = "./cadec_biobert_adr_f1select/best_model"

trainer_f1.save_model(BEST_MODEL_DIR_F1)
tokenizer.save_pretrained(BEST_MODEL_DIR_F1)

print("\nSelected model saved to:", BEST_MODEL_DIR_F1)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	

Starting BioBERT training with F1-based checkpoint selection...


Epoch,Training Loss,Validation Loss,Token Precision,Token Recall,Token F1
1,0.316556,0.126068,0.838550,0.802206,0.819976
2,0.220953,0.125668,0.802321,0.844366,0.822807
3,0.165781,0.131289,0.830255,0.832545,0.831399
4,0.144682,0.145655,0.819194,0.840820,0.829866


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...


Training completed!
Best checkpoint: ./cadec_biobert_adr_f1select/checkpoint-330

Validation results for the selected checkpoint:


Training Loss,Validation Loss,Epoch,Token Precision,Token Recall,Token F1
0.144682,0.131289,4,0.830255,0.832545,0.831399


eval_loss: 0.1313
eval_token_precision: 0.8303
eval_token_recall: 0.8325
eval_token_f1: 0.8314


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Selected model saved to: ./cadec_biobert_adr_f1select/best_model


#Evaluate BioBERT on the held-out test set

In [51]:
# ============================================================
# STEP 39: Exact character-span evaluation on the test set
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Run inference on the held-out test windows
# ------------------------------------------------------------

test_dataset = ADRTokenDataset(
    prepared_chunks["test"]
)

print("Running BioBERT on test windows...")

test_output = trainer_f1.predict(test_dataset)

test_logits = test_output.predictions

if isinstance(test_logits, tuple):
    test_logits = test_logits[0]

test_predicted_ids = np.argmax(test_logits, axis=-1)

print("Test predictions generated:", len(test_predicted_ids))


# ------------------------------------------------------------
# 2. Define the gold-standard continuous ADR spans
# ------------------------------------------------------------

# Only score spans belonging to the held-out test documents.
test_gold_spans = {
    (doc_id, start, end)
    for doc_id in split_sets["test"]
    for start, end in continuous_spans_by_doc.get(doc_id, [])
}

print("Gold continuous ADR spans:", len(test_gold_spans))


# ------------------------------------------------------------
# 3. Convert predicted BIO tags back to character spans
# ------------------------------------------------------------

def extract_window_spans(
    predicted_ids,
    gold_labels,
    offsets,
    doc_id,
    document_text
):
    """
    Convert predicted token labels into character spans.

    Special tokens, padding and masked regions are ignored.
    Predictions truncated by a non-final window boundary
    are omitted to avoid counting incomplete edge fragments.
    """

    content_indices = [
        i for i, (start, end) in enumerate(offsets)
        if end > start
    ]

    if not content_indices:
        return set()

    first_content_idx = content_indices[0]
    last_content_idx = content_indices[-1]

    window_start_char = min(
        offsets[i][0] for i in content_indices
    )

    window_end_char = max(
        offsets[i][1] for i in content_indices
    )

    # Use the last non-whitespace character as the end of
    # the document to avoid problems with trailing whitespace.
    document_end_char = len(document_text.rstrip())

    spans = set()
    active_start_idx = None
    active_end_idx = None

    def finish_active_span():
        nonlocal active_start_idx, active_end_idx

        if active_start_idx is None:
            return

        start_char = offsets[active_start_idx][0]
        end_char = offsets[active_end_idx][1]

        # Skip an apparent entity fragment touching a window
        # boundary when more document text exists beyond it.
        truncated_at_left = (
            active_start_idx == first_content_idx
            and window_start_char > 0
        )

        truncated_at_right = (
            active_end_idx == last_content_idx
            and window_end_char < document_end_char
        )

        if (
            end_char > start_char
            and not truncated_at_left
            and not truncated_at_right
        ):
            spans.add((doc_id, start_char, end_char))

        active_start_idx = None
        active_end_idx = None

    for i, (start, end) in enumerate(offsets):

        # Ignore special tokens, padding and masked regions.
        # Masked regions break a predicted span.
        if end <= start or gold_labels[i] == -100:
            finish_active_span()
            continue

        tag = ID2LABEL[int(predicted_ids[i])]

        if tag == "B-ADR":
            finish_active_span()
            active_start_idx = i
            active_end_idx = i

        elif tag == "I-ADR":
            if active_start_idx is None:
                # Recover a malformed BIO sequence.
                active_start_idx = i

            active_end_idx = i

        else:
            finish_active_span()

    finish_active_span()

    return spans


# ------------------------------------------------------------
# 4. Aggregate predictions across windows
# ------------------------------------------------------------

test_predicted_spans = set()

for window_idx, predicted_ids in enumerate(test_predicted_ids):

    metadata = chunk_metadata["test"][window_idx]
    feature = prepared_chunks["test"][window_idx]

    doc_id = metadata["doc_id"]
    offsets = metadata["offsets"]
    gold_labels = feature["labels"]
    document_text = text_lookup[doc_id]

    window_spans = extract_window_spans(
        predicted_ids=predicted_ids,
        gold_labels=gold_labels,
        offsets=offsets,
        doc_id=doc_id,
        document_text=document_text
    )

    # A set removes identical predictions from overlapping windows.
    test_predicted_spans.update(window_spans)


# ------------------------------------------------------------
# 5. Calculate exact character-span precision, recall and F1
# ------------------------------------------------------------

true_positive_spans = (
    test_predicted_spans & test_gold_spans
)

false_positive_spans = (
    test_predicted_spans - test_gold_spans
)

false_negative_spans = (
    test_gold_spans - test_predicted_spans
)

tp = len(true_positive_spans)
fp = len(false_positive_spans)
fn = len(false_negative_spans)

precision = (
    tp / (tp + fp)
    if tp + fp > 0
    else 0.0
)

recall = (
    tp / (tp + fn)
    if tp + fn > 0
    else 0.0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0
    else 0.0
)

print("\n" + "=" * 60)
print("BIOBERT — EXACT CHARACTER-SPAN TEST RESULTS")
print("=" * 60)

print("Test documents:", len(split_sets["test"]))
print("Gold ADR spans:", len(test_gold_spans))
print("Predicted ADR spans:", len(test_predicted_spans))
print("Exact true positives:", tp)
print("False positives:", fp)
print("False negatives:", fn)

print(f"\nExact-span precision: {precision:.4f} ({precision * 100:.2f}%)")
print(f"Exact-span recall:    {recall:.4f} ({recall * 100:.2f}%)")
print(f"Exact-span F1:        {f1:.4f} ({f1 * 100:.2f}%)")


# ------------------------------------------------------------
# 6. Inspect example predictions
# ------------------------------------------------------------

def span_to_text(span):
    doc_id, start, end = span
    return text_lookup[doc_id][start:end]


print("\nEXACT MATCH EXAMPLES")
print("-" * 60)

for span in sorted(true_positive_spans)[:15]:
    print(
        f"{span[0]} | Gold/prediction: "
        f"{span_to_text(span)!r}"
    )


print("\nFALSE POSITIVE EXAMPLES")
print("-" * 60)

for span in sorted(false_positive_spans)[:15]:
    print(
        f"{span[0]} | Predicted: {span_to_text(span)!r}"
    )


print("\nFALSE NEGATIVE EXAMPLES")
print("-" * 60)

for span in sorted(false_negative_spans)[:15]:
    print(
        f"{span[0]} | Missed gold ADR: {span_to_text(span)!r}"
    )


# ------------------------------------------------------------
# 7. Save the test results
# ------------------------------------------------------------

exact_span_results = pd.DataFrame([
    {
        "Metric": "Exact-span precision",
        "Value": precision
    },
    {
        "Metric": "Exact-span recall",
        "Value": recall
    },
    {
        "Metric": "Exact-span F1",
        "Value": f1
    },
    {
        "Metric": "Gold ADR spans",
        "Value": len(test_gold_spans)
    },
    {
        "Metric": "Predicted ADR spans",
        "Value": len(test_predicted_spans)
    }
])

display(exact_span_results)

Running BioBERT on test windows...


Test predictions generated: 187
Gold continuous ADR spans: 831

BIOBERT — EXACT CHARACTER-SPAN TEST RESULTS
Test documents: 186
Gold ADR spans: 831
Predicted ADR spans: 1010
Exact true positives: 624
False positives: 386
False negatives: 207

Exact-span precision: 0.6178 (61.78%)
Exact-span recall:    0.7509 (75.09%)
Exact-span F1:        0.6779 (67.79%)

EXACT MATCH EXAMPLES
------------------------------------------------------------
ARTHROTEC.103 | Gold/prediction: 'bleeding'
ARTHROTEC.110 | Gold/prediction: 'diarrhea'
ARTHROTEC.110 | Gold/prediction: 'heavy menstrual bleeding'
ARTHROTEC.110 | Gold/prediction: 'Alot of gas'
ARTHROTEC.110 | Gold/prediction: 'nausea'
ARTHROTEC.128 | Gold/prediction: 'mild cramps'
ARTHROTEC.128 | Gold/prediction: 'bloating'
ARTHROTEC.13 | Gold/prediction: 'severe stomach upset'
ARTHROTEC.13 | Gold/prediction: 'false full feeling'
ARTHROTEC.13 | Gold/prediction: 'very poor appetite'
ARTHROTEC.13 | Gold/prediction: 'increased menstruation'
ARTHROTEC.132 

,Metric,Value
0,Exact-span precision,0.617822
1,Exact-span recall,0.750903
2,Exact-span F1,0.677892
3,Gold ADR spans,831.000000
4,Predicted ADR spans,1010.000000


#Find potential boundary errors

In [52]:
# Step 40A: Analyze partial matches and boundary errors

def spans_overlap(span_a, span_b):
    """Return True if two character spans overlap in the same document."""
    doc_a, start_a, end_a = span_a
    doc_b, start_b, end_b = span_b

    return (
        doc_a == doc_b
        and max(start_a, start_b) < min(end_a, end_b)
    )


# False-positive predictions that overlap a gold span
partial_fp = {
    pred for pred in false_positive_spans
    if any(
        spans_overlap(pred, gold)
        for gold in test_gold_spans
    )
}

# Missed gold spans that overlap at least one prediction
partial_fn = {
    gold for gold in false_negative_spans
    if any(
        spans_overlap(gold, pred)
        for pred in test_predicted_spans
    )
}

print("POTENTIAL BOUNDARY ERRORS")
print("=" * 55)
print("False-positive spans overlapping gold:", len(partial_fp))
print("Missed gold spans overlapping a prediction:", len(partial_fn))

# Show examples of overlapping predictions and gold spans
examples = []

for pred in sorted(partial_fp):

    matches = [
        gold for gold in test_gold_spans
        if spans_overlap(pred, gold)
        and gold not in true_positive_spans
    ]

    for gold in matches[:3]:

        pred_text = text_lookup[pred[0]][pred[1]:pred[2]]
        gold_text = text_lookup[gold[0]][gold[1]:gold[2]]

        examples.append({
            "doc_id": pred[0],
            "predicted_phrase": pred_text,
            "gold_phrase": gold_text,
            "predicted_start": pred[1],
            "predicted_end": pred[2],
            "gold_start": gold[1],
            "gold_end": gold[2]
        })

print("\nExamples of partial matches:")
display(pd.DataFrame(examples).head(20))

POTENTIAL BOUNDARY ERRORS
False-positive spans overlapping gold: 207
Missed gold spans overlapping a prediction: 164

Examples of partial matches:


,doc_id,predicted_phrase,gold_phrase,predicted_start,predicted_end,gold_start,gold_end
0,ARTHROTEC.103,pain,so much pain,211,215,203,215
1,ARTHROTEC.103,feel,feel weak and almost fainted,285,289,285,313
2,ARTHROTEC.103,weak,feel weak and almost fainted,290,294,285,313
3,ARTHROTEC.103,fainted,feel weak and almost fainted,306,313,285,313
4,ARTHROTEC.110,bad uterine cramps,uterine cramps,32,50,36,50
5,ARTHROTEC.13,NSAID gastritis,gastritis,0,15,6,15
6,ARTHROTEC.134,blood,passing blood and clots with stoals,71,76,63,98
7,ARTHROTEC.134,clots,passing blood and clots with stoals,81,86,63,98
8,ARTHROTEC.134,stoals,passing blood and clots with stoals,92,98,63,98
9,ARTHROTEC.137,stomach felt empty and hollow,my stomach felt empty and hollow,94,123,91,123


#Restrict normalization to ADR annotations

In [53]:
# ============================================================
# STEP 41: Build an ADR-only normalization benchmark
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Join SNOMED mappings to the original entity labels
# ------------------------------------------------------------

original_labels = entities_df[
    [
        "doc_id",
        "span_key",
        "entity_type",
        "mention",
        "is_discontinuous"
    ]
].copy()

adr_norm = snomed_concepts_df.merge(
    original_labels,
    on=["doc_id", "span_key"],
    how="inner",
    suffixes=("_mapping", "_original")
)

# ------------------------------------------------------------
# 2. Keep only continuous ADRs with one assigned concept
# ------------------------------------------------------------

adr_norm = adr_norm[
    (adr_norm["entity_type"] == "ADR")
    & (~adr_norm["is_discontinuous_original"])
    & (adr_norm["mapping_relation"] == "single")
    & adr_norm["snomed_id"].notna()
    & (adr_norm["snomed_id"].astype(str) != "CONCEPT_LESS")
    & adr_norm["snomed_name"].notna()
    & adr_norm["mention_mapping"].notna()
].copy()

# Exclude the three documents identified as having conflicting
# ADR spans, so this benchmark uses our clean eligible documents.
clean_doc_ids = (
    train_doc_ids_clean
    | val_doc_ids_clean
    | test_doc_ids_clean
)

adr_norm = adr_norm[
    adr_norm["doc_id"].isin(clean_doc_ids)
].copy()

# Use the patient phrase recorded in the SNOMED annotation.
adr_norm["mention"] = adr_norm["mention_mapping"].astype(str)

# Avoid counting the same source phrase-concept mapping twice.
adr_norm = adr_norm.drop_duplicates(
    subset=["doc_id", "span_key", "snomed_id"]
).copy()

# ------------------------------------------------------------
# 3. Assign the established document-level splits
# ------------------------------------------------------------

doc_split_lookup = {
    **{doc_id: "train" for doc_id in train_doc_ids_clean},
    **{doc_id: "validation" for doc_id in val_doc_ids_clean},
    **{doc_id: "test" for doc_id in test_doc_ids_clean}
}

adr_norm["split"] = adr_norm["doc_id"].map(doc_split_lookup)

adr_norm = adr_norm[
    adr_norm["split"].notna()
].copy()

adr_norm["normalized_mention"] = (
    adr_norm["mention"].map(normalize_for_comparison)
)

# ------------------------------------------------------------
# 4. Create split-specific tables
# ------------------------------------------------------------

train_adr_norm = adr_norm[
    adr_norm["split"] == "train"
].copy()

val_adr_norm = adr_norm[
    adr_norm["split"] == "validation"
].copy()

test_adr_norm = adr_norm[
    adr_norm["split"] == "test"
].copy()

train_adr_phrases = set(
    train_adr_norm["normalized_mention"]
)

train_adr_concepts = set(
    train_adr_norm["snomed_id"].astype(str)
)

# ------------------------------------------------------------
# 5. Categorize test examples by phrase and concept novelty
# ------------------------------------------------------------

test_adr_norm["phrase_seen_in_train"] = (
    test_adr_norm["normalized_mention"].isin(train_adr_phrases)
)

test_adr_norm["concept_seen_in_train"] = (
    test_adr_norm["snomed_id"].astype(str).isin(train_adr_concepts)
)

test_adr_norm["novelty_category"] = np.select(
    [
        test_adr_norm["phrase_seen_in_train"]
        & test_adr_norm["concept_seen_in_train"],

        ~test_adr_norm["phrase_seen_in_train"]
        & test_adr_norm["concept_seen_in_train"],

        ~test_adr_norm["concept_seen_in_train"]
    ],
    [
        "Known phrase + known concept",
        "New phrase + known concept",
        "Concept unseen in training"
    ],
    default="Other"
)

# ------------------------------------------------------------
# 6. Report the corrected benchmark size
# ------------------------------------------------------------

print("ADR-ONLY NORMALIZATION BENCHMARK")
print("=" * 55)

print("Total eligible ADR mappings:", len(adr_norm))
print("Training examples:", len(train_adr_norm))
print("Validation examples:", len(val_adr_norm))
print("Test examples:", len(test_adr_norm))

print(
    "\nUnique training concepts:",
    train_adr_norm["snomed_id"].nunique()
)

print(
    "Unique test phrases:",
    test_adr_norm["normalized_mention"].nunique()
)

print("\nTest-set novelty categories:")

display(
    test_adr_norm["novelty_category"]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="Examples")
)

print("\nExample ADR normalization records:")

display(
    adr_norm[
        [
            "doc_id",
            "mention",
            "snomed_id",
            "snomed_name",
            "split"
        ]
    ].head(15)
)

ADR-ONLY NORMALIZATION BENCHMARK
Total eligible ADR mappings: 4928
Training examples: 3446
Validation examples: 705
Test examples: 777

Unique training concepts: 612
Unique test phrases: 509

Test-set novelty categories:


,Category,Examples
0,Known phrase + known concept,447
1,New phrase + known concept,268
2,Concept unseen in training,62



Example ADR normalization records:


,doc_id,mention,snomed_id,snomed_name,split
0,ARTHROTEC.1,bit drowsy,271782001,Drowsy,train
1,ARTHROTEC.1,little blurred vision,246636008,Blurred vision - hazy,train
2,ARTHROTEC.1,gastric problems,162076009,Excessive upper gastrointestinal gas,train
8,ARTHROTEC.1,feel a bit weird,367391008,Malaise,train
9,ARTHROTEC.10,Hunger pangs,162043005,Hunger pain,train
12,ARTHROTEC.100,vaginal bleeding,289530006,Vaginal bleeding,train
13,ARTHROTEC.100,stomach pain,271681002,Stomach ache,train
14,ARTHROTEC.100,canker sores,426965005,Aphthous ulcer,train
15,ARTHROTEC.100,headache,25064002,Headache,train
17,ARTHROTEC.101,shortness of breath,267036007,Dyspnoea,train


#Compare TF-IDF and SapBERT on ADRs only

In [54]:
# ============================================================
# STEP 42: Fair TF-IDF vs SapBERT comparison on ADRs only
# ============================================================

import numpy as np
import pandas as pd
import torch

from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. Build candidate concepts from TRAINING ADRs ONLY
# ------------------------------------------------------------

def most_common_concept_name(names):
    names = names.dropna().astype(str).str.strip()
    names = names[names.ne("")]

    if names.empty:
        return None

    return names.value_counts().index[0]


candidate_adr_table = (
    train_adr_norm.groupby("snomed_id")
    .agg(
        concept_name=("snomed_name", most_common_concept_name)
    )
    .reset_index()
)

candidate_adr_table = candidate_adr_table[
    candidate_adr_table["concept_name"].notna()
].copy()

candidate_adr_table["snomed_id"] = (
    candidate_adr_table["snomed_id"].astype(str)
)

candidate_adr_table = candidate_adr_table.reset_index(drop=True)

candidate_ids_adr = (
    candidate_adr_table["snomed_id"].to_numpy()
)

candidate_names_adr = (
    candidate_adr_table["concept_name"].astype(str).tolist()
)

print("ADR-ONLY CANDIDATE VOCABULARY")
print("=" * 50)
print("Training ADR examples:", len(train_adr_norm))
print("Candidate concepts:", len(candidate_ids_adr))


# ------------------------------------------------------------
# 2. Prepare the ADR-only test set
# ------------------------------------------------------------

adr_test_eval = test_adr_norm.reset_index(drop=True).copy()

query_texts = (
    adr_test_eval["mention"].fillna("").astype(str).tolist()
)

gold_ids_adr = (
    adr_test_eval["snomed_id"].astype(str).to_numpy()
)

novelty_adr = (
    adr_test_eval["novelty_category"].to_numpy()
)

candidate_name_lookup = dict(
    zip(
        candidate_adr_table["snomed_id"],
        candidate_adr_table["concept_name"]
    )
)

print("Test examples:", len(adr_test_eval))

print("\nTest novelty groups:")
display(
    adr_test_eval["novelty_category"]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="Examples")
)


# ------------------------------------------------------------
# 3. TF-IDF: retrieve the closest standardized concept name
# ------------------------------------------------------------

word_vectorizer_adr = TfidfVectorizer(
    ngram_range=(1, 2),
    lowercase=True,
    strip_accents="unicode",
    sublinear_tf=True
)

char_vectorizer_adr = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    lowercase=True,
    sublinear_tf=True
)

concept_word_adr = word_vectorizer_adr.fit_transform(
    candidate_names_adr
)

concept_char_adr = char_vectorizer_adr.fit_transform(
    candidate_names_adr
)

query_word_adr = word_vectorizer_adr.transform(query_texts)
query_char_adr = char_vectorizer_adr.transform(query_texts)

tfidf_scores_adr = (
    0.5 * (query_word_adr @ concept_word_adr.T).toarray()
    + 0.5 * (query_char_adr @ concept_char_adr.T).toarray()
)

tfidf_positions_adr = np.argsort(
    -tfidf_scores_adr, axis=1
)

tfidf_ranked_ids_adr = candidate_ids_adr[
    tfidf_positions_adr
]


# ------------------------------------------------------------
# 4. SapBERT: use a separate tokenizer and model
# ------------------------------------------------------------
# Important: "tokenizer" elsewhere in the notebook is now
# the BioBERT tokenizer. We use separate SapBERT variables.

SAPBERT_MODEL_NAME = (
    "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

sapbert_tokenizer_adr = AutoTokenizer.from_pretrained(
    SAPBERT_MODEL_NAME
)

sapbert_model_adr = AutoModel.from_pretrained(
    SAPBERT_MODEL_NAME
).to(device)

sapbert_model_adr.eval()

print("\nSapBERT device:", device)


def embed_adr_texts(texts, batch_size=32, max_length=64):
    """Create L2-normalized SapBERT CLS embeddings."""

    all_embeddings = []

    for start in tqdm(range(0, len(texts), batch_size)):

        batch = [
            str(text) if pd.notna(text) else ""
            for text in texts[start:start + batch_size]
        ]

        encoded = sapbert_tokenizer_adr(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        with torch.inference_mode():
            output = sapbert_model_adr(**encoded)

            # CLS representation, L2-normalized for cosine similarity
            embeddings = output.last_hidden_state[:, 0, :]
            embeddings = torch.nn.functional.normalize(
                embeddings, p=2, dim=1
            )

        all_embeddings.append(
            embeddings.cpu().numpy()
        )

    return np.concatenate(all_embeddings, axis=0)


# Embed the test phrases and exactly the same candidate concepts
test_embeddings_adr = embed_adr_texts(query_texts)

concept_embeddings_adr = embed_adr_texts(
    candidate_names_adr
)

sapbert_scores_adr = (
    test_embeddings_adr @ concept_embeddings_adr.T
)

sapbert_positions_adr = np.argsort(
    -sapbert_scores_adr, axis=1
)

sapbert_ranked_ids_adr = candidate_ids_adr[
    sapbert_positions_adr
]


# ------------------------------------------------------------
# 5. Evaluate the same examples for both models
# ------------------------------------------------------------

def evaluate_ranked_concepts(
    ranked_ids,
    gold_ids,
    example_mask,
    candidate_ids
):
    """Calculate Accuracy@1, Recall@5 and MRR."""

    indices = np.flatnonzero(example_mask)
    ranks = []

    for i in indices:

        target = str(gold_ids[i])

        # Position of the gold concept in the ranked predictions
        candidate_positions = np.flatnonzero(
            ranked_ids[i] == target
        )

        if len(candidate_positions):
            rank = int(candidate_positions[0]) + 1
        else:
            rank = len(candidate_ids) + 1

        ranks.append(rank)

    ranks = np.asarray(ranks)

    if len(ranks) == 0:
        return {
            "Examples": 0,
            "Accuracy@1": np.nan,
            "Recall@5": np.nan,
            "MRR": np.nan
        }

    return {
        "Examples": len(ranks),
        "Accuracy@1": float(np.mean(ranks == 1)),
        "Recall@5": float(np.mean(ranks <= 5)),
        "MRR": float(np.mean(1.0 / ranks))
    }


comparison_rows = []

for category in [
    "Known phrase + known concept",
    "New phrase + known concept"
]:

    category_mask = novelty_adr == category

    tfidf_metrics = evaluate_ranked_concepts(
        tfidf_ranked_ids_adr,
        gold_ids_adr,
        category_mask,
        candidate_ids_adr
    )

    sapbert_metrics = evaluate_ranked_concepts(
        sapbert_ranked_ids_adr,
        gold_ids_adr,
        category_mask,
        candidate_ids_adr
    )

    comparison_rows.append({
        "Category": category,
        "Examples": tfidf_metrics["Examples"],
        "TF-IDF Accuracy@1": tfidf_metrics["Accuracy@1"],
        "SapBERT Accuracy@1": sapbert_metrics["Accuracy@1"],
        "TF-IDF Recall@5": tfidf_metrics["Recall@5"],
        "SapBERT Recall@5": sapbert_metrics["Recall@5"],
        "TF-IDF MRR": tfidf_metrics["MRR"],
        "SapBERT MRR": sapbert_metrics["MRR"]
    })


adr_comparison_df = pd.DataFrame(comparison_rows)

print("\nADR-ONLY NORMALIZATION RESULTS")
print("=" * 70)

display(adr_comparison_df.round(4))


# ------------------------------------------------------------
# 6. Inspect the new-phrase predictions
# ------------------------------------------------------------

new_phrase_mask = (
    novelty_adr == "New phrase + known concept"
)

example_predictions_df = adr_test_eval.loc[
    new_phrase_mask,
    ["doc_id", "mention", "snomed_id", "snomed_name"]
].copy()

example_predictions_df["TF-IDF prediction ID"] = (
    tfidf_ranked_ids_adr[new_phrase_mask, 0]
)

example_predictions_df["SapBERT prediction ID"] = (
    sapbert_ranked_ids_adr[new_phrase_mask, 0]
)

example_predictions_df["TF-IDF prediction"] = (
    example_predictions_df["TF-IDF prediction ID"]
    .map(candidate_name_lookup)
)

example_predictions_df["SapBERT prediction"] = (
    example_predictions_df["SapBERT prediction ID"]
    .map(candidate_name_lookup)
)

print("\nNEW PHRASES — EXAMPLE PREDICTIONS")
display(example_predictions_df.head(20))


# ------------------------------------------------------------
# 7. Record the key numbers for the research report
# ------------------------------------------------------------

known_concept_test = adr_test_eval[
    adr_test_eval["novelty_category"]
    == "New phrase + known concept"
]

print("\nRESEARCH SUMMARY")
print("=" * 50)
print("ADR-only candidate concepts:", len(candidate_ids_adr))
print(
    "New-phrase/known-concept test assignments:",
    len(known_concept_test)
)
print(
    "Distinct new phrases in this group:",
    known_concept_test["normalized_mention"].nunique()
)
print(
    "Concepts unseen in training:",
    int(
        (
            adr_test_eval["novelty_category"]
            == "Concept unseen in training"
        ).sum()
    )
)

ADR-ONLY CANDIDATE VOCABULARY
Training ADR examples: 3446
Candidate concepts: 612
Test examples: 777

Test novelty groups:


,Category,Examples
0,Known phrase + known concept,447
1,New phrase + known concept,268
2,Concept unseen in training,62


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


SapBERT device: cuda


  0%|          | 0/25 [00:00<?, ?it/s]

...


ADR-ONLY NORMALIZATION RESULTS


,Category,Examples,TF-IDF Accuracy@1,SapBERT Accuracy@1,TF-IDF Recall@5,SapBERT Recall@5,TF-IDF MRR,SapBERT MRR
0,Known phrase + known concept,447,0.6488,0.7942,0.7271,0.9195,0.6850,0.8526
1,New phrase + known concept,268,0.3657,0.5000,0.5896,0.7873,0.4635,0.6342



NEW PHRASES — EXAMPLE PREDICTIONS


,doc_id,mention,snomed_id,snomed_name,TF-IDF prediction ID,SapBERT prediction ID,TF-IDF prediction,SapBERT prediction
1,ARTHROTEC.103,so much pain,22253000,Pain,22253000,76948002,Pain,Severe pain
3,ARTHROTEC.110,uterine cramps,29542008,Uterine spasm,29542008,29542008,Uterine spasm,Uterine spasm
5,ARTHROTEC.110,Alot of gas,162076009,Excessive upper gastrointestinal gas,162076009,162076009,Excessive upper gastrointestinal gas,Excessive upper gastrointestinal gas
8,ARTHROTEC.128,mild cramps,55300003,Muscle cramp,250082003,55300003,Hand cramps,Muscle cramp
10,ARTHROTEC.13,gastritis,162076009,Excessive upper gastrointestinal gas,397825006,271681002,Gastric ulcer,Stomach ache
11,ARTHROTEC.13,severe stomach upset,162059005,Upset stomach,162059005,162059005,Upset stomach,Upset stomach
13,ARTHROTEC.13,increased menstruation,386692008,Menorrhagia,80182007,266601003,Irregular menstruation,Excessive and frequent menstruation
14,ARTHROTEC.13,2-3 periods a month instead of once a month,266601003,Excessive and frequent menstruation,248966006,248966006,Prolonged periods,Prolonged periods
17,ARTHROTEC.132,blood in my stools,405729008,Haematochezia,35064005,35064005,Dark stools,Dark stools
18,ARTHROTEC.132,major cramps,55300003,Muscle cramp,250082003,55300003,Hand cramps,Muscle cramp



RESEARCH SUMMARY
ADR-only candidate concepts: 612
New-phrase/known-concept test assignments: 268
Distinct new phrases in this group: 262
Concepts unseen in training: 62


#Calculate confidence intervals for the ADR-only results

In [55]:
# ============================================================
# STEP 43: Document-level bootstrap for ADR-only normalization
# ============================================================

import numpy as np
import pandas as pd

# Use only new patient expressions for known concepts.
target_mask = (
    novelty_adr == "New phrase + known concept"
)

target_df = (
    adr_test_eval.loc[target_mask]
    .reset_index(drop=True)
)

target_gold = gold_ids_adr[target_mask]

target_tfidf_rankings = (
    tfidf_ranked_ids_adr[target_mask]
)

target_sapbert_rankings = (
    sapbert_ranked_ids_adr[target_mask]
)

# Calculate the rank assigned to each gold concept.
def get_gold_ranks(ranked_ids, gold_ids):

    ranks = []

    for row, gold_id in zip(ranked_ids, gold_ids):

        matches = np.flatnonzero(
            row.astype(str) == str(gold_id)
        )

        rank = (
            int(matches[0]) + 1
            if len(matches)
            else ranked_ids.shape[1] + 1
        )

        ranks.append(rank)

    return np.asarray(ranks)


tfidf_ranks = get_gold_ranks(
    target_tfidf_rankings, target_gold
)

sapbert_ranks = get_gold_ranks(
    target_sapbert_rankings, target_gold
)


def ranking_metrics(ranks):

    return np.array([
        np.mean(ranks == 1),   # Accuracy@1
        np.mean(ranks <= 5),   # Recall@5
        np.mean(1.0 / ranks)   # MRR
    ])


# Observed improvement: SapBERT minus TF-IDF.
observed_delta = (
    ranking_metrics(sapbert_ranks)
    - ranking_metrics(tfidf_ranks)
)

# Group test examples by source document.
doc_ids = target_df["doc_id"].to_numpy()
unique_docs = np.unique(doc_ids)

rows_by_doc = {
    doc_id: np.flatnonzero(doc_ids == doc_id)
    for doc_id in unique_docs
}

print("ADR-ONLY DOCUMENT-LEVEL BOOTSTRAP")
print("=" * 55)
print("Test assignments:", len(target_df))
print("Distinct phrases:", target_df["normalized_mention"].nunique())
print("Distinct documents:", len(unique_docs))

# Resample complete documents with replacement.
rng = np.random.default_rng(42)
n_bootstrap = 2000

bootstrap_deltas = []

for _ in range(n_bootstrap):

    sampled_docs = rng.choice(
        unique_docs,
        size=len(unique_docs),
        replace=True
    )

    sampled_rows = np.concatenate([
        rows_by_doc[doc_id]
        for doc_id in sampled_docs
    ])

    delta = (
        ranking_metrics(sapbert_ranks[sampled_rows])
        - ranking_metrics(tfidf_ranks[sampled_rows])
    )

    bootstrap_deltas.append(delta)

bootstrap_deltas = np.asarray(bootstrap_deltas)

# Percentile 95% confidence intervals.
lower = np.percentile(
    bootstrap_deltas, 2.5, axis=0
)

upper = np.percentile(
    bootstrap_deltas, 97.5, axis=0
)

results = pd.DataFrame({
    "Metric": ["Accuracy@1", "Recall@5", "MRR"],
    "SapBERT improvement": observed_delta,
    "95% CI lower": lower,
    "95% CI upper": upper
})

print("\nObserved improvements and confidence intervals:")
display(results.round(4))

ADR-ONLY DOCUMENT-LEVEL BOOTSTRAP
Test assignments: 268
Distinct phrases: 262
Distinct documents: 125

Observed improvements and confidence intervals:


,Metric,SapBERT improvement,95% CI lower,95% CI upper
0,Accuracy@1,0.1343,0.0741,0.1963
1,Recall@5,0.1978,0.1303,0.2617
2,MRR,0.1707,0.1204,0.2207


#Examine errors in the ADR-only benchmark

In [56]:
# ============================================================
# STEP 45: Error analysis of ADR-only normalization
# ============================================================

import numpy as np
import pandas as pd

# Focus on new phrases whose concepts are known to the model.
target_mask = (
    novelty_adr == "New phrase + known concept"
)

error_df = adr_test_eval.loc[
    target_mask,
    ["doc_id", "mention", "snomed_id", "snomed_name"]
].copy().reset_index(drop=True)

# Gold labels and top-ranked predictions from both models.
gold = error_df["snomed_id"].astype(str).to_numpy()

tfidf_pred = (
    tfidf_ranked_ids_adr[target_mask, 0].astype(str)
)

sapbert_pred = (
    sapbert_ranked_ids_adr[target_mask, 0].astype(str)
)

# Look up the standardized names of the predicted concepts.
concept_names = dict(
    zip(
        candidate_adr_table["snomed_id"].astype(str),
        candidate_adr_table["concept_name"].astype(str)
    )
)

error_df["TF-IDF prediction ID"] = tfidf_pred
error_df["SapBERT prediction ID"] = sapbert_pred

error_df["TF-IDF prediction"] = (
    error_df["TF-IDF prediction ID"].map(concept_names)
)

error_df["SapBERT prediction"] = (
    error_df["SapBERT prediction ID"].map(concept_names)
)

error_df["TF-IDF correct"] = (
    error_df["TF-IDF prediction ID"] == gold
)

error_df["SapBERT correct"] = (
    error_df["SapBERT prediction ID"] == gold
)

error_df["Outcome"] = np.select(
    [
        error_df["TF-IDF correct"]
        & error_df["SapBERT correct"],

        ~error_df["TF-IDF correct"]
        & error_df["SapBERT correct"],

        error_df["TF-IDF correct"]
        & ~error_df["SapBERT correct"]
    ],
    [
        "Both correct",
        "SapBERT only correct",
        "TF-IDF only correct"
    ],
    default="Both wrong"
)

print("ADR-ONLY ERROR ANALYSIS")
print("=" * 55)

display(
    error_df["Outcome"]
    .value_counts()
    .rename_axis("Outcome")
    .reset_index(name="Examples")
)

columns = [
    "mention",
    "snomed_name",
    "TF-IDF prediction",
    "SapBERT prediction"
]

print("\nExamples where SapBERT is correct and TF-IDF is wrong:")
display(
    error_df[
        (error_df["Outcome"] == "SapBERT only correct")
    ][columns].head(15)
)

print("\nExamples where TF-IDF is correct and SapBERT is wrong:")
display(
    error_df[
        (error_df["Outcome"] == "TF-IDF only correct")
    ][columns].head(15)
)

print("\nExamples where both models are wrong:")
display(
    error_df[
        (error_df["Outcome"] == "Both wrong")
    ][columns].head(15)
)

ADR-ONLY ERROR ANALYSIS


,Outcome,Examples
0,Both wrong,111
1,Both correct,75
2,SapBERT only correct,59
3,TF-IDF only correct,23



Examples where SapBERT is correct and TF-IDF is wrong:


,mention,snomed_name,TF-IDF prediction,SapBERT prediction
3,mild cramps,Muscle cramp,Hand cramps,Muscle cramp
9,major cramps,Muscle cramp,Hand cramps,Muscle cramp
16,"""back labor"" pains",Backache,Low back pain,Backache
17,increase in cramping,Muscle cramp,Cramp in foot,Muscle cramp
20,so much pain,Severe pain,Pain,Severe pain
22,stomach gas,Excessive upper gastrointestinal gas,Stomach ache,Excessive upper gastrointestinal gas
23,feel like a bubble of air is stuck in my chest,Tight chest,Chest pain,Tight chest
42,burning back pain,Backache,Burning pain,Backache
43,drive is knocked down,Loss of motivation,Unable to lie down,Loss of motivation
46,sore back,Backache,Low back pain,Backache



Examples where TF-IDF is correct and SapBERT is wrong:


,mention,snomed_name,TF-IDF prediction,SapBERT prediction
0,so much pain,Pain,Pain,Severe pain
10,diahorea,Diarrhoea,Diarrhoea,Polymenorrhoea
14,pain in the pit of my stomach,Stomach ache,Stomach ache,Stomach cramps
29,rash on my wrist,Rash,Rash,Pain in wrist
38,tummy bloating,Abdominal bloating,Abdominal bloating,Bloating symptom
56,blured vision,Blurred vision - hazy,Blurred vision - hazy,Blurring of visual image
83,burning sensations in calves,Burning sensation,Burning sensation,Pain in calf
97,high liver enzyme,Liver enzymes abnormal,Liver enzymes abnormal,Increased liver function
114,memory lapses,Memory impairment,Memory impairment,Transient memory loss
120,Blured vision,Blurred vision - hazy,Blurred vision - hazy,Blurring of visual image



Examples where both models are wrong:


,mention,snomed_name,TF-IDF prediction,SapBERT prediction
4,gastritis,Excessive upper gastrointestinal gas,Gastric ulcer,Stomach ache
6,increased menstruation,Menorrhagia,Irregular menstruation,Excessive and frequent menstruation
7,2-3 periods a month instead of once a month,Excessive and frequent menstruation,Prolonged periods,Prolonged periods
8,blood in my stools,Haematochezia,Dark stools,Dark stools
11,passing blood and clots with stoals,Haematochezia,Blood clot in eye,Syncope
15,deep ache from my waist down to the heels of m...,Generalised aches and pains,Unable to lie down,Heel pain
18,excessive mestrual bleeding,Menorrhagia,Bleeding,Excessive and frequent menstruation
19,severe mentrual cramps,Menstrual cramp,Hand cramps,Hand cramps
24,feel dreadful,Malaise,Abnormal dreams,Discomfort
26,feel sick,Malaise,Feeling tired,Illness


#Verify the ADR-to-SNOMED mappings

In [57]:
# ============================================================
# STEP 46: Audit ADR phrase-to-SNOMED mappings
# ============================================================

import pandas as pd

mapping_audit_df = adr_norm.copy()

# Retrieve the phrase directly from the original patient text.
mapping_audit_df["text_at_offsets"] = mapping_audit_df.apply(
    lambda row: text_lookup[row["doc_id"]][
        int(row["start"]):int(row["end"])
    ],
    axis=1
)

mapping_audit_df["offset_matches_mapping_phrase"] = (
    mapping_audit_df.apply(
        lambda row: normalize_for_comparison(
            row["text_at_offsets"]
        ) == normalize_for_comparison(row["mention"]),
        axis=1
    )
)

# Compare against the phrase in the original entity annotations,
# when that column is available.
if "mention_original" in mapping_audit_df.columns:
    mapping_audit_df["mapping_matches_original_mention"] = (
        mapping_audit_df.apply(
            lambda row: normalize_for_comparison(
                row["mention"]
            ) == normalize_for_comparison(
                row["mention_original"]
            ),
            axis=1
        )
    )

print("ADR MAPPING AUDIT")
print("=" * 55)

print("ADR normalization records:", len(mapping_audit_df))

print(
    "Phrases matching original text offsets:",
    int(mapping_audit_df["offset_matches_mapping_phrase"].sum())
)

print(
    "Offset mismatch count:",
    int((~mapping_audit_df["offset_matches_mapping_phrase"]).sum())
)

if "mapping_matches_original_mention" in mapping_audit_df.columns:
    print(
        "Phrases matching original entity annotations:",
        int(
            mapping_audit_df[
                "mapping_matches_original_mention"
            ].sum()
        )
    )

print("\nOffset mismatches, if any:")

display(
    mapping_audit_df[
        ~mapping_audit_df["offset_matches_mapping_phrase"]
    ][
        [
            "doc_id",
            "mention",
            "text_at_offsets",
            "snomed_id",
            "snomed_name"
        ]
    ].head(20)
)

print("\nReview of the gastritis example:")

display(
    mapping_audit_df[
        mapping_audit_df["mention"]
        .str.contains("gastritis", case=False, na=False)
    ][
        [
            "doc_id",
            "mention",
            "text_at_offsets",
            "snomed_id",
            "snomed_name",
            "split"
        ]
    ].head(10)
)

ADR MAPPING AUDIT
ADR normalization records: 4928
Phrases matching original text offsets: 4928
Offset mismatch count: 0
Phrases matching original entity annotations: 4928

Offset mismatches, if any:


,doc_id,mention,text_at_offsets,snomed_id,snomed_name



Review of the gastritis example:


,doc_id,mention,text_at_offsets,snomed_id,snomed_name,split
202,ARTHROTEC.13,gastritis,gastritis,162076009,Excessive upper gastrointestinal gas,test


#Inspect the original records

In [58]:
# Inspect the original annotation and parsed mapping for ARTHROTEC.13

doc_id_to_check = "ARTHROTEC.13"

print("ENTITIES DATAFRAME COLUMNS:")
print(entities_df.columns.tolist())

print("\nSNOMED MAPPING DATAFRAME COLUMNS:")
print(snomed_concepts_df.columns.tolist())

print("\nORIGINAL ANNOTATIONS FOR THIS DOCUMENT:")
display(
    entities_df[
        entities_df["doc_id"] == doc_id_to_check
    ].to_string(index=False)
)

print("\nPARSED SNOMED MAPPINGS FOR THIS DOCUMENT:")
display(
    snomed_concepts_df[
        snomed_concepts_df["doc_id"] == doc_id_to_check
    ].to_string(index=False)
)

ENTITIES DATAFRAME COLUMNS:
['doc_id', 'entity_id', 'entity_type', 'mention', 'spans', 'span_key', 'start', 'end', 'is_discontinuous']

SNOMED MAPPING DATAFRAME COLUMNS:
['doc_id', 'annotation_id', 'mention', 'snomed_id', 'snomed_name', 'mapping_relation', 'concept_index', 'raw_line', 'spans', 'span_key', 'start', 'end', 'is_discontinuous', 'semantic_tag']

ORIGINAL ANNOTATIONS FOR THIS DOCUMENT:


'      doc_id entity_id entity_type                                     mention                                spans                span_key  start  end  is_discontinuous\nARTHROTEC.13        T1         ADR                                   gastritis                           ((6, 15),)                    6:15      6   15             False\nARTHROTEC.13        T2         ADR                        severe stomach upset                          ((17, 37),)                   17:37     17   37             False\nARTHROTEC.13        T3         ADR                          false full feeling                          ((59, 77),)                   59:77     59   77             False\nARTHROTEC.13        T4         ADR                          very poor appetite                          ((79, 97),)                   79:97     79   97             False\nARTHROTEC.13        T5         ADR                      increased menstruation                         ((99, 121),)                  99:121     


PARSED SNOMED MAPPINGS FOR THIS DOCUMENT:


'      doc_id annotation_id                                     mention    snomed_id                          snomed_name mapping_relation  concept_index                                                                                                     raw_line                                spans                span_key  start  end  is_discontinuous semantic_tag\nARTHROTEC.13           TT1                                   gastritis    162076009 Excessive upper gastrointestinal gas           single              1                                      TT1\\t162076009 | Excessive upper gastrointestinal gas | 6 15\\tgastritis                           ((6, 15),)                    6:15      6   15             False          NaN\nARTHROTEC.13           TT2                        severe stomach upset    162059005                        Upset stomach           single              1                                                 TT2\\t162059005 | Upset stomach | 17 37\\tsevere stomach upset

#Find other records that may need review

In [59]:
# STEP 48: Investigate potentially questionable source mappings

print("1. ALL MAPPINGS FOR MENTIONS CONTAINING 'GASTRITIS'")
gastritis_records = snomed_concepts_df[
    snomed_concepts_df["mention"].str.contains(
        "gastritis", case=False, na=False
    )
]

display(
    gastritis_records[
        ["doc_id", "mention", "snomed_id",
         "snomed_name", "mapping_relation", "raw_line"]
    ].to_string(index=False)
)

print("\n2. ALL MENTIONS ASSIGNED TO THE GAS-RELATED CONCEPT")
gas_concept_records = snomed_concepts_df[
    snomed_concepts_df["snomed_id"].astype(str) == "162076009"
]

display(
    gas_concept_records[
        ["doc_id", "mention", "snomed_id",
         "snomed_name", "mapping_relation"]
    ].to_string(index=False)
)

print("\n3. SUMMARY")
print("Mentions containing 'gastritis':", len(gastritis_records))
print("Records assigned to concept 162076009:", len(gas_concept_records))

1. ALL MAPPINGS FOR MENTIONS CONTAINING 'GASTRITIS'


'      doc_id         mention snomed_id                          snomed_name mapping_relation                                                                   raw_line\nARTHROTEC.13       gastritis 162076009 Excessive upper gastrointestinal gas           single    TT1\\t162076009 | Excessive upper gastrointestinal gas | 6 15\\tgastritis\n  LIPITOR.20       gastritis 162076009 Excessive upper gastrointestinal gas           single   TT1\\t162076009 | Excessive upper gastrointestinal gas | 28 37\\tgastritis\n VOLTAREN.21 acute gastritis  25458004                      Acute gastritis           single                 TT7\\t25458004 | Acute gastritis | 403 418\\tacute gastritis\n VOLTAREN.24       gastritis 162076009 Excessive upper gastrointestinal gas           single TT3\\t162076009 | Excessive upper gastrointestinal gas | 134 143\\tgastritis'


2. ALL MENTIONS ASSIGNED TO THE GAS-RELATED CONCEPT


'        doc_id                                                      mention snomed_id                          snomed_name mapping_relation\n   ARTHROTEC.1                                             gastric problems 162076009 Excessive upper gastrointestinal gas           single\n ARTHROTEC.110                                                  Alot of gas 162076009 Excessive upper gastrointestinal gas           single\n ARTHROTEC.115                                                        gassy 162076009 Excessive upper gastrointestinal gas           single\n ARTHROTEC.124                                                          gas 162076009 Excessive upper gastrointestinal gas           single\n ARTHROTEC.126                                    gastrointestinal cramping 162076009 Excessive upper gastrointestinal gas           single\n  ARTHROTEC.13                                                    gastritis 162076009 Excessive upper gastrointestinal gas           single\n ARTHROTEC.1


3. SUMMARY
Mentions containing 'gastritis': 4
Records assigned to concept 162076009: 118


#Check how this affects our benchmark

In [60]:
# STEP 49: Check gastritis mappings in the ADR normalization benchmark

gastritis_benchmark = adr_norm[
    adr_norm["mention"].str.strip().str.lower().eq("gastritis")
].copy()

print("Exact 'gastritis' records in the benchmark:", len(gastritis_benchmark))

display(
    gastritis_benchmark[
        [
            "doc_id",
            "mention",
            "snomed_id",
            "snomed_name",
            "split"
        ]
    ].sort_values("split").to_string(index=False)
)

print("\nCounts by split:")
print(
    gastritis_benchmark["split"]
    .value_counts()
    .reindex(["train", "validation", "test"], fill_value=0)
)

Exact 'gastritis' records in the benchmark: 1


'      doc_id   mention snomed_id                          snomed_name split\nARTHROTEC.13 gastritis 162076009 Excessive upper gastrointestinal gas  test'


Counts by split:
split
train         0
validation    0
test          1
Name: count, dtype: int64


#Inspect the patient context

In [61]:
# STEP 50: Inspect the context of gastritis-related mappings

for _, row in gastritis_records.drop_duplicates(
    subset=["doc_id", "mention", "snomed_id"]
).iterrows():

    doc_id = row["doc_id"]
    text = text_lookup.get(doc_id, "")

    start = int(row["start"])
    end = int(row["end"])

    # Show 200 characters before and after the mention
    context_start = max(0, start - 200)
    context_end = min(len(text), end + 200)

    print("=" * 90)
    print("Document:", doc_id)
    print("Mention:", row["mention"])
    print("SNOMED CT ID:", row["snomed_id"])
    print("Mapped concept:", row["snomed_name"])
    print("\nOriginal patient text around the mention:")
    print(text[context_start:context_end])

    print("\n" + "-" * 90)

Document: ARTHROTEC.13
Mention: gastritis
SNOMED CT ID: 162076009
Mapped concept: Excessive upper gastrointestinal gas

Original patient text around the mention:
NSAID gastritis, severe stomach upset after prolonged use.
false full feeling, very poor appetite.
increased menstruation, 2-3 periods a month instead of once a month, menstrual cramps present with or without vagina

------------------------------------------------------------------------------------------
Document: LIPITOR.20
Mention: gastritis
SNOMED CT ID: 162076009
Mapped concept: Excessive upper gastrointestinal gas

Original patient text around the mention:
First the doc thought I had gastritis from pain in my abdomen, headaches, flushing - I had an abdominal ultrasound, endoscopy, then CT scans (2) and then yesterday an MRI - all were normal.
The fatigue, muscle pain, headaches, and genera

------------------------------------------------------------------------------------------
Document: VOLTAREN.21
Mention: acute gas

#Check saved results

In [62]:
# FINAL CODING — STEP 1: Check existing models and evaluation results

print("=" * 70)
print("1. NORMALIZATION DATA")
print("=" * 70)

for name in ["adr_norm", "test_adr_norm", "candidate_adr_table"]:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name}:")
        print("  Type:", type(obj).__name__)
        if hasattr(obj, "shape"):
            print("  Shape:", obj.shape)
        if hasattr(obj, "columns"):
            print("  Columns:", list(obj.columns))
    else:
        print(f"\n{name}: NOT FOUND")

print("\n" + "=" * 70)
print("2. NORMALIZATION MODEL RESULTS")
print("=" * 70)

for name in [
    "tfidf_ranked_ids_adr",
    "tfidf_positions_adr",
    "sapbert_ranked_ids_adr",
    "sapbert_positions_adr",
    "tfidf_scores_adr",
    "sapbert_scores_adr",
    "test_embeddings_adr",
    "concept_embeddings_adr",
    "sapbert_model_adr",
    "sapbert_tokenizer_adr",
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: available", end="")
        if hasattr(obj, "shape"):
            print(f" | shape={obj.shape}")
        else:
            print(f" | type={type(obj).__name__}")
    else:
        print(f"{name}: not found")

print("\n" + "=" * 70)
print("3. EXTRACTION MODEL")
print("=" * 70)

for name in [
    "trainer_f1",
    "tokenizer",
    "prepared_chunks",
    "text_lookup",
    "test_gold_spans",
    "test_predicted_spans",
    "BEST_MODEL_DIR_F1",
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: available", end="")
        if name == "prepared_chunks":
            print(" | splits:", list(obj.keys()))
        elif name == "text_lookup":
            print(f" | documents={len(obj)}")
        elif name == "BEST_MODEL_DIR_F1":
            print(f" | path={obj}")
        else:
            print(f" | type={type(obj).__name__}")
    else:
        print(f"{name}: not found")

print("\n" + "=" * 70)
print("4. FLAGGED GASTRITIS TEST RECORD")
print("=" * 70)

if "test_adr_norm" in globals():
    flagged = test_adr_norm[
        (test_adr_norm["doc_id"] == "ARTHROTEC.13")
        & (test_adr_norm["mention"].str.lower() == "gastritis")
    ]
    if len(flagged):
        display(flagged)
    else:
        print("Record not found in test_adr_norm; we will locate it another way.")
else:
    print("test_adr_norm is unavailable; checking the full benchmark instead.")

print("\nDiagnostic complete.")

1. NORMALIZATION DATA

adr_norm:
  Type: DataFrame
  Shape: (4928, 20)
  Columns: ['doc_id', 'annotation_id', 'mention_mapping', 'snomed_id', 'snomed_name', 'mapping_relation', 'concept_index', 'raw_line', 'spans', 'span_key', 'start', 'end', 'is_discontinuous_mapping', 'semantic_tag', 'entity_type', 'mention_original', 'is_discontinuous_original', 'mention', 'split', 'normalized_mention']

test_adr_norm:
  Type: DataFrame
  Shape: (777, 23)
  Columns: ['doc_id', 'annotation_id', 'mention_mapping', 'snomed_id', 'snomed_name', 'mapping_relation', 'concept_index', 'raw_line', 'spans', 'span_key', 'start', 'end', 'is_discontinuous_mapping', 'semantic_tag', 'entity_type', 'mention_original', 'is_discontinuous_original', 'mention', 'split', 'normalized_mention', 'phrase_seen_in_train', 'concept_seen_in_train', 'novelty_category']

candidate_adr_table:
  Type: DataFrame
  Shape: (612, 2)
  Columns: ['snomed_id', 'concept_name']

2. NORMALIZATION MODEL RESULTS
tfidf_ranked_ids_adr: available 

,doc_id,annotation_id,mention_mapping,snomed_id,snomed_name,mapping_relation,concept_index,raw_line,spans,span_key,...,semantic_tag,entity_type,mention_original,is_discontinuous_original,mention,split,normalized_mention,phrase_seen_in_train,concept_seen_in_train,novelty_category
202,ARTHROTEC.13,TT1,gastritis,162076009,Excessive upper gastrointestinal gas,single,1,TT1\t162076009 | Excessive upper gastrointesti...,"((6, 15),)",6:15,...,NaN,ADR,gastritis,False,gastritis,test,gastritis,False,True,New phrase + known concept



Diagnostic complete.


#Run the sensitivity analysis

In [63]:
# FINAL CODING — STEP 2: Sensitivity analysis for the flagged mapping

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Check that the ranking arrays align with the test data
# ---------------------------------------------------------

assert len(test_adr_norm) == len(tfidf_ranked_ids_adr), (
    "TF-IDF ranking rows do not match test records."
)
assert len(test_adr_norm) == len(sapbert_ranked_ids_adr), (
    "SapBERT ranking rows do not match test records."
)

# Standardize concept IDs so strings and integer IDs can be compared safely.
def concept_id_key(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    if value.endswith(".0"):
        try:
            value = str(int(float(value)))
        except ValueError:
            pass

    return value


# ---------------------------------------------------------
# 2. Find the flagged record's positional row
# ---------------------------------------------------------

flag_mask = (
    test_adr_norm["doc_id"].eq("ARTHROTEC.13")
    & test_adr_norm["mention"].str.strip().str.lower().eq("gastritis")
)

flag_positions = np.flatnonzero(flag_mask.to_numpy())

if len(flag_positions) != 1:
    raise ValueError(
        f"Expected exactly one flagged test record, found {len(flag_positions)}."
    )

flag_pos = int(flag_positions[0])
flagged_record = test_adr_norm.iloc[flag_pos]

print("Flagged record:")
print("Document:", flagged_record["doc_id"])
print("Mention:", flagged_record["mention"])
print("Supplied concept:", flagged_record["snomed_name"])
print("Concept ID:", flagged_record["snomed_id"])
print("Novelty group:", flagged_record["novelty_category"])
print("Ranking-matrix row:", flag_pos)


# ---------------------------------------------------------
# 3. Convert rankings into the gold concept's rank
# ---------------------------------------------------------

def get_gold_rankings(ranked_ids, dataframe):
    ranks = []

    for pos in range(len(dataframe)):
        gold_id = concept_id_key(dataframe.iloc[pos]["snomed_id"])

        ranked_row = [
            concept_id_key(concept_id)
            for concept_id in ranked_ids[pos]
        ]

        try:
            rank = ranked_row.index(gold_id)  # Zero-based rank
        except ValueError:
            rank = None

        ranks.append(rank)

    return ranks


tfidf_gold_ranks = get_gold_rankings(
    tfidf_ranked_ids_adr, test_adr_norm
)
sapbert_gold_ranks = get_gold_rankings(
    sapbert_ranked_ids_adr, test_adr_norm
)


# ---------------------------------------------------------
# 4. Define evaluation metrics
# ---------------------------------------------------------

def ranking_metrics(ranks):
    ranks = list(ranks)
    n = len(ranks)

    if n == 0:
        return {
            "N": 0,
            "Accuracy@1": np.nan,
            "Recall@5": np.nan,
            "MRR": np.nan
        }

    return {
        "N": n,
        "Accuracy@1": np.mean([
            rank == 0 for rank in ranks
        ]),
        "Recall@5": np.mean([
            rank is not None and rank < 5 for rank in ranks
        ]),
        "MRR": np.mean([
            1 / (rank + 1) if rank is not None else 0
            for rank in ranks
        ])
    }


# ---------------------------------------------------------
# 5. Evaluate the new-phrase, known-concept subgroup
# ---------------------------------------------------------

target_group = "New phrase + known concept"

group_positions = np.flatnonzero(
    test_adr_norm["novelty_category"]
    .eq(target_group)
    .to_numpy()
)

adjusted_positions = [
    pos for pos in group_positions if pos != flag_pos
]

results = []

for model_name, all_ranks in [
    ("TF-IDF", tfidf_gold_ranks),
    ("SapBERT", sapbert_gold_ranks)
]:
    original = ranking_metrics([
        all_ranks[pos] for pos in group_positions
    ])

    adjusted = ranking_metrics([
        all_ranks[pos] for pos in adjusted_positions
    ])

    for version, metrics in [
        ("Original benchmark", original),
        ("Sensitivity: flagged record excluded", adjusted)
    ]:
        results.append({
            "Model": model_name,
            "Evaluation": version,
            **metrics
        })

    print(f"\n{model_name} ranking for the flagged record:")
    rank = all_ranks[flag_pos]
    print(
        "Gold concept rank:",
        f"{rank + 1} (rank 1 is best)" if rank is not None
        else "Not present among candidate concepts"
    )

results_df = pd.DataFrame(results)

print("\n" + "=" * 90)
print("SENSITIVITY ANALYSIS: NEW PHRASE + KNOWN CONCEPT")
print("=" * 90)

display(
    results_df.style.format({
        "Accuracy@1": "{:.4f}",
        "Recall@5": "{:.4f}",
        "MRR": "{:.4f}"
    })
)

print("\nImportant:")
print("- Original benchmark results are preserved.")
print("- The sensitivity analysis excludes exactly one test record.")
print("- No supplied SNOMED CT label has been changed.")

Flagged record:
Document: ARTHROTEC.13
Mention: gastritis
Supplied concept: Excessive upper gastrointestinal gas
Concept ID: 162076009
Novelty group: New phrase + known concept
Ranking-matrix row: 10

TF-IDF ranking for the flagged record:
Gold concept rank: 4 (rank 1 is best)

SapBERT ranking for the flagged record:
Gold concept rank: 35 (rank 1 is best)

SENSITIVITY ANALYSIS: NEW PHRASE + KNOWN CONCEPT


,Model,Evaluation,N,Accuracy@1,Recall@5,MRR
0,TF-IDF,Original benchmark,268,0.3657,0.5896,0.4635
1,TF-IDF,Sensitivity: flagged record excluded,267,0.3670,0.5880,0.4643
2,SapBERT,Original benchmark,268,0.5000,0.7873,0.6342
3,SapBERT,Sensitivity: flagged record excluded,267,0.5019,0.7903,0.6364



Important:
- Original benchmark results are preserved.
- The sensitivity analysis excludes exactly one test record.
- No supplied SNOMED CT label has been changed.


#Verify SapBERT embedding strategy


In [64]:
# FINAL CODING — STEP 3: Verify SapBERT embedding strategy

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd

sapbert_model_adr.eval()

# Use a small sample from the existing test data and concept vocabulary.
n_test_check = min(8, len(test_adr_norm))
n_concept_check = min(8, len(candidate_adr_table))

test_phrases_check = (
    test_adr_norm["mention"].iloc[:n_test_check].astype(str).tolist()
)
concept_names_check = (
    candidate_adr_table["concept_name"]
    .iloc[:n_concept_check].astype(str).tolist()
)

def sapbert_embeddings_for_check(texts, pooling):
    """Create normalized SapBERT embeddings using a selected pooling method."""
    encoded = sapbert_tokenizer_adr(
        texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        output = sapbert_model_adr(**encoded)
        token_vectors = output.last_hidden_state

        if pooling == "mean":
            mask = encoded["attention_mask"].unsqueeze(-1).float()
            vectors = (token_vectors * mask).sum(dim=1)
            vectors = vectors / mask.sum(dim=1).clamp(min=1e-9)
        elif pooling == "cls":
            vectors = token_vectors[:, 0, :]
        else:
            raise ValueError("pooling must be 'mean' or 'cls'")

        vectors = F.normalize(vectors, p=2, dim=1)

    return vectors.cpu().numpy()


def as_numpy_normalized(value):
    if isinstance(value, torch.Tensor):
        value = value.detach().cpu().numpy()

    value = np.asarray(value, dtype=np.float32)
    norms = np.linalg.norm(value, axis=1, keepdims=True)
    return value / np.clip(norms, 1e-12, None)


stored_test = as_numpy_normalized(test_embeddings_adr[:n_test_check])
stored_concepts = as_numpy_normalized(
    concept_embeddings_adr[:n_concept_check]
)

diagnostic_results = []

for pooling in ["mean", "cls"]:
    calculated_test = sapbert_embeddings_for_check(
        test_phrases_check, pooling
    )
    calculated_concepts = sapbert_embeddings_for_check(
        concept_names_check, pooling
    )

    test_cosines = np.sum(calculated_test * stored_test, axis=1)
    concept_cosines = np.sum(
        calculated_concepts * stored_concepts, axis=1
    )

    diagnostic_results.append({
        "Pooling method": pooling,
        "Mean cosine similarity — test phrases": float(
            np.mean(test_cosines)
        ),
        "Minimum cosine similarity — test phrases": float(
            np.min(test_cosines)
        ),
        "Mean cosine similarity — concept names": float(
            np.mean(concept_cosines)
        ),
        "Minimum cosine similarity — concept names": float(
            np.min(concept_cosines)
        )
    })

display(
    pd.DataFrame(diagnostic_results).style.format({
        column: "{:.6f}"
        for column in diagnostic_results[0]
        if column != "Pooling method"
    })
)

print(
    "\nInterpretation: the pooling method with cosine similarities "
    "closest to 1.0 best matches the saved embeddings."
)

,Pooling method,Mean cosine similarity — test phrases,Minimum cosine similarity — test phrases,Mean cosine similarity — concept names,Minimum cosine similarity — concept names
0,mean,0.982000,0.968587,0.970836,0.961506
1,cls,1.000000,1.000000,1.000000,1.000000



Interpretation: the pooling method with cosine similarities closest to 1.0 best matches the saved embeddings.


#Build the integrated pipeline

In [65]:
# FINAL CODING — STEP 4
# Integrated BioBERT ADR extraction + SapBERT concept normalization

import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer

# ---------------------------------------------------------
# 1. Check the existing models and concept embeddings
# ---------------------------------------------------------

assert "trainer_f1" in globals(), "The trained BioBERT model was not found."
assert "sapbert_model_adr" in globals(), "The SapBERT model was not found."
assert "candidate_adr_table" in globals(), "The candidate concept table was not found."
assert "concept_embeddings_adr" in globals(), "The concept embeddings were not found."

bio_model = trainer_f1.model
bio_model.eval()

bio_device = next(bio_model.parameters()).device
sap_device = next(sapbert_model_adr.parameters()).device

assert len(candidate_adr_table) == len(concept_embeddings_adr), (
    "The candidate concepts and their embeddings do not align."
)

# Retain the tokenizer used for training if it provides character offsets.
# Otherwise, load a fast tokenizer with the same BioBERT vocabulary.
if getattr(tokenizer, "is_fast", False):
    biobert_infer_tokenizer = tokenizer
else:
    biobert_infer_tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        use_fast=True
    )

assert getattr(biobert_infer_tokenizer, "is_fast", False), (
    "A fast BioBERT tokenizer is needed to recover character offsets."
)

# Check that the fast tokenizer produces the same ordinary token IDs
# as the tokenizer used during training.
probe_texts = [
    "stomach pain",
    "I felt dizzy and tired"
]

training_ids = tokenizer(
    probe_texts,
    add_special_tokens=False
)["input_ids"]

inference_ids = biobert_infer_tokenizer(
    probe_texts,
    add_special_tokens=False
)["input_ids"]

assert training_ids == inference_ids, (
    "The inference tokenizer produced different token IDs. "
    "Stop here and investigate before evaluating the pipeline."
)

# Normalize the saved concept embeddings for cosine similarity.
concept_matrix = torch.as_tensor(
    np.asarray(concept_embeddings_adr),
    dtype=torch.float32,
    device=sap_device
)
concept_matrix = F.normalize(concept_matrix, p=2, dim=1)

sapbert_model_adr.eval()

print("BioBERT device:", bio_device)
print("SapBERT device:", sap_device)
print("Candidate concepts:", len(candidate_adr_table))
print("Tokenizer IDs verified against the training tokenizer.")
print("Both existing models are ready.")


# ---------------------------------------------------------
# 2. Extract ADR mentions with BioBERT
# ---------------------------------------------------------

def extract_adr_mentions(text, chunk_size=510, stride=384):
    """
    Extract ADR character spans using the trained BioBERT model.

    Long inputs are processed in overlapping token windows. Predictions
    for tokens appearing in multiple windows are averaged before decoding.
    Character offsets refer to the supplied original text.
    """

    if not isinstance(text, str):
        raise TypeError("Input text must be a string.")

    if not text.strip():
        return []

    if not 1 <= stride <= chunk_size <= 510:
        raise ValueError(
            "Use 1 <= stride <= chunk_size <= 510."
        )

    encoded = biobert_infer_tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_offsets_mapping=True
    )

    token_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    if not token_ids:
        return []

    cls_id = biobert_infer_tokenizer.cls_token_id
    sep_id = biobert_infer_tokenizer.sep_token_id

    if cls_id is None or sep_id is None:
        raise ValueError("BioBERT CLS/SEP token IDs are unavailable.")

    all_logits = np.zeros(
        (len(token_ids), bio_model.config.num_labels),
        dtype=np.float64
    )
    prediction_counts = np.zeros(len(token_ids), dtype=np.int32)

    for window_start in range(0, len(token_ids), stride):
        window_end = min(window_start + chunk_size, len(token_ids))
        body_ids = token_ids[window_start:window_end]

        input_ids = [cls_id] + body_ids + [sep_id]

        model_inputs = {
            "input_ids": torch.tensor(
                [input_ids], dtype=torch.long, device=bio_device
            ),
            "attention_mask": torch.ones(
                (1, len(input_ids)), dtype=torch.long, device=bio_device
            ),
            "token_type_ids": torch.zeros(
                (1, len(input_ids)), dtype=torch.long, device=bio_device
            )
        }

        with torch.inference_mode():
            logits = bio_model(**model_inputs).logits[0, 1:1 + len(body_ids)]

        logits = logits.detach().cpu().numpy()

        all_logits[window_start:window_end] += logits
        prediction_counts[window_start:window_end] += 1

    # Average predictions where a token occurred in overlapping windows.
    averaged_logits = all_logits / np.maximum(
        prediction_counts[:, None], 1
    )

    predicted_ids = averaged_logits.argmax(axis=1)

    # Convert token-level BIO labels into original-text character spans.
    predicted_spans = []
    active_start = None
    active_end = None

    def close_active_span():
        nonlocal active_start, active_end

        if active_start is not None and active_end is not None:
            start = active_start
            end = active_end

            while start < end and text[start].isspace():
                start += 1

            while end > start and text[end - 1].isspace():
                end -= 1

            if start < end:
                predicted_spans.append((start, end))

        active_start = None
        active_end = None

    for token_index, label_id in enumerate(predicted_ids):
        start, end = offsets[token_index]

        # Ignore tokens without a usable original-text span.
        if end <= start:
            close_active_span()
            continue

        label = ID2LABEL.get(int(label_id), "O")

        if label == "B-ADR":
            close_active_span()
            active_start = start
            active_end = end

        elif label == "I-ADR":
            if active_start is None:
                # Recover gracefully if an I-tag appears without a B-tag.
                active_start = start

            active_end = end

        else:
            close_active_span()

    close_active_span()

    # De-duplicate any identical spans and return them in text order.
    predicted_spans = sorted(set(predicted_spans))

    return [
        {
            "mention": text[start:end],
            "start": start,
            "end": end
        }
        for start, end in predicted_spans
    ]


# ---------------------------------------------------------
# 3. Normalize an extracted phrase with SapBERT
# ---------------------------------------------------------

def normalize_adr_phrase(phrase, top_k=5):
    """
    Rank the candidate SNOMED CT concepts using SapBERT CLS embeddings.

    Similarity scores are cosine similarities, not calibrated probabilities.
    """

    top_k = min(max(1, int(top_k)), len(candidate_adr_table))

    sap_inputs = sapbert_tokenizer_adr(
        [phrase],
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    sap_inputs = {
        key: value.to(sap_device)
        for key, value in sap_inputs.items()
    }

    with torch.inference_mode():
        output = sapbert_model_adr(**sap_inputs)

        # CLS pooling matches the embeddings from the original experiment.
        phrase_embedding = output.last_hidden_state[:, 0, :]
        phrase_embedding = F.normalize(
            phrase_embedding, p=2, dim=1
        )

        similarities = (
            phrase_embedding @ concept_matrix.T
        )[0]

    top_scores, top_indices = torch.topk(
        similarities, k=top_k
    )

    ranked_concepts = []

    for score, index in zip(
        top_scores.detach().cpu().tolist(),
        top_indices.detach().cpu().tolist()
    ):
        concept = candidate_adr_table.iloc[index]

        ranked_concepts.append({
            "snomed_id": str(concept["snomed_id"]),
            "snomed_name": str(concept["concept_name"]),
            "cosine_similarity": float(score)
        })

    return ranked_concepts


# ---------------------------------------------------------
# 4. Connect extraction and normalization
# ---------------------------------------------------------

def analyze_patient_text(text, top_k=5):
    """
    Full prototype: patient text -> BioBERT ADR spans -> SapBERT concepts.
    """

    mentions = extract_adr_mentions(text)

    results = []

    for item in mentions:
        ranked_concepts = normalize_adr_phrase(
            item["mention"], top_k=top_k
        )

        best = ranked_concepts[0] if ranked_concepts else {}

        results.append({
            "mention": item["mention"],
            "start": item["start"],
            "end": item["end"],
            "predicted_snomed_id": best.get("snomed_id"),
            "predicted_snomed_name": best.get("snomed_name"),
            "top1_cosine_similarity": best.get("cosine_similarity"),
            "ranked_concepts": ranked_concepts
        })

    return results


# ---------------------------------------------------------
# 5. Test the complete prototype on patient-style text
# ---------------------------------------------------------

demo_text = (
    "After taking the medicine, I felt dizzy and tired. "
    "I also had a bad stomach upset and a terrible headache."
)

print("\n" + "=" * 80)
print("INTEGRATED PIPELINE DEMONSTRATION")
print("=" * 80)
print("Input text:")
print(demo_text)

demo_results = analyze_patient_text(demo_text, top_k=5)

if demo_results:
    for i, result in enumerate(demo_results, start=1):
        print("\n" + "-" * 80)
        print(f"Extracted ADR {i}: {result['mention']!r}")
        print(
            f"Character offsets: "
            f"{result['start']}:{result['end']}"
        )

        print("\nRanked SNOMED CT concepts:")

        for rank, concept in enumerate(
            result["ranked_concepts"], start=1
        ):
            print(
                f"{rank}. {concept['snomed_name']} "
                f"| SNOMED CT: {concept['snomed_id']} "
                f"| cosine similarity: "
                f"{concept['cosine_similarity']:.4f}"
            )
else:
    print(
        "\nBioBERT did not extract any ADR mentions from this example. "
        "That is a possible model prediction, not a code failure."
    )


# ---------------------------------------------------------
# 6. Save the demo results as a CSV
# ---------------------------------------------------------

output_rows = []

for result in demo_results:
    for rank, concept in enumerate(
        result["ranked_concepts"], start=1
    ):
        output_rows.append({
            "input_text": demo_text,
            "mention": result["mention"],
            "start": result["start"],
            "end": result["end"],
            "rank": rank,
            "snomed_id": concept["snomed_id"],
            "snomed_name": concept["snomed_name"],
            "cosine_similarity": concept["cosine_similarity"]
        })

output_columns = [
    "input_text",
    "mention",
    "start",
    "end",
    "rank",
    "snomed_id",
    "snomed_name",
    "cosine_similarity"
]

demo_output_df = pd.DataFrame(
    output_rows, columns=output_columns
)

output_path = "/content/integrated_pipeline_demo.csv"
demo_output_df.to_csv(output_path, index=False)

print("\n" + "=" * 80)
print("PIPELINE TEST FINISHED")
print("=" * 80)
print("Extracted mentions:", len(demo_results))
print("Saved demo output:", output_path)
print(
    "Note: similarity scores are ranking signals, not clinical confidence "
    "or evidence of a diagnosis."
)

BioBERT device: cuda:0
SapBERT device: cuda:0
Candidate concepts: 612
Tokenizer IDs verified against the training tokenizer.
Both existing models are ready.

INTEGRATED PIPELINE DEMONSTRATION
Input text:
After taking the medicine, I felt dizzy and tired. I also had a bad stomach upset and a terrible headache.

--------------------------------------------------------------------------------
Extracted ADR 1: 'dizzy'
Character offsets: 34:39

Ranked SNOMED CT concepts:
1. Dizziness | SNOMED CT: 404640003 | cosine similarity: 0.8977
2. Vertigo | SNOMED CT: 399153001 | cosine similarity: 0.7902
3. Severe vertigo | SNOMED CT: 103298005 | cosine similarity: 0.7347
4. Nausea | SNOMED CT: 422587007 | cosine similarity: 0.6936
5. Lightheadedness | SNOMED CT: 386705008 | cosine similarity: 0.6482

--------------------------------------------------------------------------------
Extracted ADR 2: 'tired'
Character offsets: 44:49

Ranked SNOMED CT concepts:
1. Tired | SNOMED CT: 224960004 | cosine si

#Create Streamlit interface

In [66]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
from transformers import AutoTokenizer

EXPORT_ROOT = Path("/content/medphrase_streamlit_export")
BIO_DIR = EXPORT_ROOT / "biobert"
ASSET_DIR = EXPORT_ROOT / "assets"
BIO_DIR.mkdir(parents=True, exist_ok=True)
ASSET_DIR.mkdir(parents=True, exist_ok=True)

required = ["trainer_f1", "MODEL_NAME", "candidate_adr_table", "concept_embeddings_adr"]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(
        "Missing notebook variables: " + ", ".join(missing) +
        ". Run this cell in the same Colab runtime after the project experiments."
    )

# Export the trained BioBERT token-classification checkpoint.
trainer_f1.model.save_pretrained(BIO_DIR, safe_serialization=True)
fast_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
fast_tokenizer.save_pretrained(BIO_DIR)

# Export the candidate vocabulary and its matching SapBERT CLS embeddings.
candidates = candidate_adr_table.copy()
embeddings = np.asarray(concept_embeddings_adr, dtype=np.float32)
if len(candidates) != embeddings.shape[0]:
    raise ValueError(
        f"Candidate/embedding mismatch: {len(candidates)} rows vs {embeddings.shape[0]} vectors."
    )
if embeddings.ndim != 2 or embeddings.shape[1] != 768:
    raise ValueError(f"Expected embeddings shape (N, 768); got {embeddings.shape}.")

candidates.to_csv(ASSET_DIR / "candidate_adr_table.csv", index=False)
np.save(ASSET_DIR / "concept_embeddings_adr.npy", embeddings)
(ASSET_DIR / "README.md").write_text(
    "Exported from the CADEC ADR normalization experiment. Keep candidate row order aligned with the NPY vectors.\n",
    encoding="utf-8",
)

zip_path = Path("/content/medphrase_streamlit_export.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in EXPORT_ROOT.rglob("*"):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(EXPORT_ROOT))

print("Export complete:", zip_path)
print("BioBERT files:", [p.name for p in BIO_DIR.iterdir()])
print("Candidate table:", candidates.shape)
print("Embedding matrix:", embeddings.shape)
print("WARNING: Review CADEC and SNOMED CT licenses before redistributing files or model weights.")

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Download the ZIP from the runtime file browser:", zip_path)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Export complete: /content/medphrase_streamlit_export.zip
BioBERT files: ['tokenizer.json', 'model.safetensors', 'tokenizer_config.json', 'config.json']
Candidate table: (612, 2)
Embedding matrix: (612, 768)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>